# Property Factor Lens — independent verification and backtest

This notebook checks every number the dashboard shows, and tests every forward-looking claim it makes, before the dashboard goes to clients. It works in four layers:

| Layer | Question it answers | How |
|---|---|---|
| **1. Input data audit** | Is the data going in right? | Every price, total-return index, yield, FX rate and fundamental in `docs/data/dashboard_data.js` is checked for gaps, spikes, stale values, unit errors, splices and internal consistency. |
| **2. Engine reconciliation** | Does the dashboard compute what it says it computes? | The dashboard's own JavaScript engine (`docs/assets/analytics.js` + `model.js`) is run on the same data file through Node.js, and every number it produces is recomputed **independently** in Python (numpy, pandas, statsmodels, scipy) from the documented method, then compared. |
| **3. Screen audit** *(optional)* | Is the right number shown under the right label? | The real dashboard is opened in headless Chromium; every tile, table cell and generated sentence is read off the screen for every stock and compared with the independently computed value, formatted the same way. |
| **4. Backtests** | Do the forward-looking numbers hold up? | Volatility forecasts and price ranges, 1-day VaR, the 12-month exposure forecasts and their stated accuracy, the price-signal record and the stress replays are all tested out of sample. |

A methodology review then lists issues that do not show up as numerical mismatches (labels that mean different things on different tabs, date arithmetic that breaks on some dates, survivorship, look-ahead).

**How to read the results.** Each comparison gets a status:

| Status | Meaning |
|---|---|
| **PASS** | Dashboard and independent value agree within numerical tolerance. |
| **WARN** | They differ beyond tolerance, but identically when rounded the way the dashboard displays them (not visible on screen), *or* a check needs a human look. |
| **FAIL** | A difference that changes what is on screen, a number present on one side only, or a failed data/backtest test. |

Every comparison is kept as a data point, so you can filter, sort and plot them yourself: after a run, `CK` holds every comparison (one row each, with both values, the difference and the text shown on screen), `FIND` every finding, and `drilldown("SGRO")` shows everything about one stock. Everything is also exported to `backtester/output/` (Excel workbook, CSV files and a self-contained HTML report).

**To run:** open this notebook from inside the repository (`backtester/`), then *Run All*. Needs Python 3.10+ with the packages in `backtester/requirements.txt` and **Node.js 18+** on the PATH for layer 2 (or `pip install mini-racer` as a fallback). A full run takes five to ten minutes: about three for the dashboard engine across every setting, two for the screen audit, and the rest for the reference implementation and the backtests. The screen audit runs when `playwright` is installed; the online source checks need internet access and `yfinance` and are switched on in the configuration cell.

**Where to look first:** the scorecard and the findings table in section 12 (also in `backtester/output/verification_report_<date>.html`), then the backtest verdicts. Every finding names the stocks affected, how much the numbers on screen move, and a suggested fix.

## Configuration

In [ ]:
# ============================== CONFIGURATION ==============================
DATA_FILE = None            # None = docs/data/dashboard_data.js in this repository
RUN_ENGINE = True           # run the dashboard's JavaScript engine (Node.js 18+, or `pip install mini-racer`)
FULL_SETTINGS_GRID = True   # reconcile regressions for every frequency x window x currency x model (False = default view only)
RUN_BACKTESTS = True        # out-of-sample tests of the forward-looking numbers
RUN_SCREEN_AUDIT = "auto"   # read the rendered dashboard in headless Chromium: "auto" = when playwright is installed
                            # (pip install playwright; playwright install chromium), True = always, False = never
SCREEN_AUDIT_STOCKS = None  # None = every stock, or e.g. ["SGRO", "VNA"]
RUN_ONLINE_CHECKS = False   # re-download source data (yfinance, ECB, Kenneth French) and compare with the data file
EXPORT = True               # write the Excel / CSV / HTML reports to backtester/output/
DEFAULT_VIEW = {"freq": "W", "years": 3, "ccy": "local"}  # the dashboard's default settings (weekly, 3Y, local currency)
GARCH_BACKTEST_DATES = "Q"  # refit the volatility model at every quarter-end ("Q") or month-end ("M", about 3x slower)
GARCH_BACKTEST_IN_JS = False  # True = refit with the dashboard's own JavaScript at each date (exact, but ~10+ minutes)
NODE_BINARY = "node"        # path to node if it is not on the PATH
# ===========================================================================

In [ ]:
import base64
import io
import json
import math
import os
import re
import shutil
import subprocess
import sys
import tempfile
import time
import warnings
from dataclasses import dataclass
from decimal import ROUND_HALF_UP, Decimal
from pathlib import Path
from types import SimpleNamespace

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.optimize as spo
import scipy.stats as sps
import statsmodels.api as sm
from IPython.display import HTML, Markdown, display
from matplotlib.colors import LinearSegmentedColormap
from scipy.signal import lfilter

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)
pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)
T_START = time.time()


def find_repo(start=None):
    """Walk up from the working directory to the repository root."""
    here = Path(start or Path.cwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "docs" / "assets" / "model.js").exists() and (cand / "docs" / "index.html").exists():
            return cand
    raise FileNotFoundError("Open this notebook from inside the VLK_Factor_dashboard repository, or set REPO by hand.")


REPO = find_repo()
DOCS = REPO / "docs"
DATA_PATH = Path(DATA_FILE) if DATA_FILE else DOCS / "data" / "dashboard_data.js"
OUT = REPO / "backtester" / "output"
OUT.mkdir(parents=True, exist_ok=True)
print(f"Repository: {REPO}\nData file:  {DATA_PATH}\nReports:    {OUT}")

### Chart style
Charts follow one system: a light surface, hairline solid gridlines, 2px lines, a fixed categorical order (blue, orange, aqua) and reserved status colours (good / warning / critical) that always come with a text label. Every chart has a table twin in the outputs, so no value is readable only from a colour.

In [ ]:
C = SimpleNamespace(
    surface="#fcfcfb", ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
    s1="#2a78d6", s2="#eb6834", s3="#1baf7a", s4="#eda100", deemph="#c3c2b7",
    good="#0ca30c", warning="#fab219", serious="#ec835a", critical="#d03b3b", neutral="#f0efec",
)
STATUS_COLOR = {"PASS": C.good, "WARN": C.warning, "FAIL": C.critical, "INFO": C.muted}
mpl.rcParams.update({
    "figure.facecolor": C.surface, "axes.facecolor": C.surface, "savefig.facecolor": C.surface,
    "axes.edgecolor": C.axis, "axes.linewidth": 0.8, "axes.grid": True, "grid.color": C.grid,
    "grid.linewidth": 0.6, "grid.linestyle": "-", "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.color": C.muted, "ytick.color": C.muted, "xtick.labelcolor": C.ink2, "ytick.labelcolor": C.ink2,
    "axes.labelcolor": C.ink2, "axes.titlecolor": C.ink, "text.color": C.ink,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.titlelocation": "left", "axes.labelsize": 9.5,
    "font.size": 9.5, "legend.frameon": False, "legend.fontsize": 9,
    "lines.linewidth": 2.0, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
    "axes.prop_cycle": mpl.cycler(color=[C.s1, C.s2, C.s3, C.s4]),
    "figure.dpi": 100, "savefig.dpi": 110, "figure.autolayout": False,
})
SEQ_BLUE = LinearSegmentedColormap.from_list("seq_blue", ["#cde2fb", "#86b6ef", "#3987e5", "#1c5cab", "#0d366b"])
DIVERGING = LinearSegmentedColormap.from_list("div_blue_red", ["#1c5cab", "#86b6ef", C.neutral, "#f0a3a3", "#c42f2f"])
FIGURES = []  # (title, png bytes) for the HTML report


def show(fig, title):
    """Display a figure and keep a copy for the HTML report."""
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight")
    FIGURES.append((title, buf.getvalue()))
    plt.show()
    plt.close(fig)


def status_bar(ax, counts, title):
    """Horizontal bars of PASS/WARN/FAIL counts, each labelled with its number and status word."""
    order = [s for s in ("PASS", "WARN", "FAIL") if s in counts.index]
    vals = [int(counts[s]) for s in order]
    ax.barh(order[::-1], vals[::-1], color=[STATUS_COLOR[s] for s in order[::-1]], height=0.55)
    for y, v in enumerate(vals[::-1]):
        ax.text(v, y, f"  {v:,}", va="center", color=C.ink2, fontsize=9)
    ax.set_title(title)
    ax.grid(axis="y", visible=False)
    ax.set_xlim(0, max(vals + [1]) * 1.18)

### The dashboard's number formats, ported to Python
`docs/assets/ui.js` turns raw numbers into on-screen text (`num`, `pct`, `pp`, `bp`, `mult`, `money`, `compact`). The same rules are reproduced here, with the same rounding (half away from zero on the shortest decimal form of the number, which is what V8's `Intl.NumberFormat` does), so that a difference can be judged by whether it changes the text on screen. The port is itself checked against the JavaScript originals, string for string, in section 5.8.

In [ ]:
MINUS, DASH = "−", "–"
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
SYMBOL = {"EUR": "€", "GBP": "£", "USD": "$"}


def is_num(x):
    return isinstance(x, (int, float, np.integer, np.floating)) and not isinstance(x, bool) and math.isfinite(float(x))


def js_num(x, dp=2, sign=False):
    if not is_num(x):
        return DASH
    x = float(x)
    # V8's Intl.NumberFormat rounds the shortest decimal form of the double half away from zero
    # (0.045 -> "0.05", although the stored double is 0.04499...), so round repr(), not the exact binary value.
    q = Decimal(repr(abs(x))).quantize(Decimal(1).scaleb(-dp), rounding=ROUND_HALF_UP)
    s = f"{q:,.{dp}f}"
    if x < 0 and q != 0:
        return MINUS + s
    return ("+" if sign and x > 0 else "") + s


def js_pct(x, dp=1, sign=False):
    return js_num(float(x) * 100, dp, sign) + "%" if is_num(x) else DASH


def js_pp(x, dp=1):
    return js_num(float(x) * 100, dp, True) + " pp" if is_num(x) else DASH


def js_bp(x, dp=0, sign=True):
    return js_num(x, dp, sign) + " bp" if is_num(x) else DASH


def js_mult(x, dp=2):
    return js_num(x, dp) + "x" if is_num(x) else DASH


def js_compact(x, dp=1):
    if not is_num(x):
        return DASH
    a = abs(float(x))
    if a >= 1e9:
        return js_num(float(x) / 1e9, dp) + "bn"
    if a >= 1e6:
        return js_num(float(x) / 1e6, dp) + "m"
    if a >= 1e3:
        return js_num(float(x) / 1e3, dp) + "k"
    return js_num(x, dp)


def js_money(x, ccy="EUR", dp=2):
    if not is_num(x):
        return DASH
    body = js_num(abs(float(x)), dp)
    sign = MINUS if x < 0 else ""
    sym = SYMBOL.get(ccy)
    return f"{sign}{sym}{body}" if sym else f"{sign}{ccy} {body}"


def js_money_compact(x, ccy="EUR"):
    if not is_num(x):
        return DASH
    sym = SYMBOL.get(ccy)
    return f"{sym}{js_compact(x)}" if sym else f"{ccy} {js_compact(x)}"


def js_date(iso, with_year=True):
    if not iso:
        return DASH
    y, m, d = iso.split("-")
    return f"{int(d)} {MONTHS[int(m) - 1]} {y}" if with_year else f"{int(d)} {MONTHS[int(m) - 1]}"


def js_month_year(iso):
    return f"{MONTHS[int(iso[5:7]) - 1]} {iso[:4]}" if iso else DASH


def js_ordinal(n):
    v = n % 100
    if 11 <= v <= 13:
        return f"{n}th"
    return f"{n}" + {1: "st", 2: "nd", 3: "rd"}.get(n % 10, "th")


def js_sig(t):
    if not is_num(t):
        return ""
    a = abs(t)
    return "***" if a >= 2.58 else "**" if a >= 1.96 else "*" if a >= 1.65 else ""


def js_round(x):
    """JavaScript Math.round (halves go up)."""
    return math.floor(x + 0.5)

### Check registry
Every comparison lands in one table. `add_checks` takes a frame of dashboard values and independent values and assigns PASS / WARN / FAIL; `finding` records qualitative issues (data, methodology, presentation) with a severity.

In [ ]:
CHECK_FRAMES = []
FINDINGS = []
SEVERITY_ORDER = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3, "INFO": 4}


def _as_float(v):
    if v is None:
        return np.nan
    if isinstance(v, (bool, np.bool_)):
        return float(v)
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan


def add_checks(rows, section, *, tol_abs=0.0, tol_rel=1e-9, display=None, layer="2. Engine"):
    """rows: iterable of dicts with keys item, stock, setting, dashboard, reference (+ optional note, fmt).

    display (or a per-row `fmt`): function value -> on-screen text, used to judge whether a
    difference beyond tolerance would be visible on screen. Returns the checks frame for this batch.
    """
    df = pd.DataFrame(list(rows))
    if df.empty:
        return df
    for col in ("stock", "setting", "note", "fmt"):
        if col not in df:
            df[col] = None
    d = df["dashboard"].map(_as_float).to_numpy()
    r = df["reference"].map(_as_float).to_numpy()
    both_nan = np.isnan(d) & np.isnan(r)
    one_nan = np.isnan(d) ^ np.isnan(r)
    diff = np.abs(d - r)
    scale = np.fmax(np.abs(d), np.abs(r))
    ok = both_nan | (diff <= tol_abs + tol_rel * np.nan_to_num(scale))
    status = np.where(ok, "PASS", "FAIL").astype(object)
    status[one_nan] = "FAIL"
    shown_d = np.array([None] * len(df), dtype=object)
    shown_r = np.array([None] * len(df), dtype=object)
    fmts = df["fmt"].to_numpy()
    for i in np.flatnonzero(~ok):
        fn = fmts[i] if callable(fmts[i]) else display
        if fn is None:
            continue
        shown_d[i], shown_r[i] = fn(d[i]), fn(r[i])
        if not one_nan[i] and shown_d[i] == shown_r[i]:
            status[i] = "WARN"
    out = pd.DataFrame({
        "layer": layer,
        "section": section,
        "item": df["item"].astype(str),
        "stock": df["stock"],
        "setting": df["setting"],
        "dashboard": d,
        "reference": r,
        "abs_diff": np.where(both_nan, 0.0, diff),
        "rel_diff": np.where(both_nan, 0.0, diff / np.where(scale > 0, scale, 1.0)),
        "tolerance": tol_abs + tol_rel * np.nan_to_num(scale),
        "shown_dashboard": shown_d,
        "shown_reference": shown_r,
        "status": status,
        "note": df["note"],
    })
    CHECK_FRAMES.append(out)
    return out


def add_bool_checks(rows, section, layer="2. Engine"):
    """rows: dicts with item, stock, setting, passed (bool), dashboard/reference (any, shown as text), note."""
    df = pd.DataFrame(list(rows))
    if df.empty:
        return df
    for col in ("stock", "setting", "note", "dashboard", "reference"):
        if col not in df:
            df[col] = None
    out = pd.DataFrame({
        "layer": layer, "section": section, "item": df["item"].astype(str), "stock": df["stock"], "setting": df["setting"],
        "dashboard": np.nan, "reference": np.nan, "abs_diff": np.nan, "rel_diff": np.nan, "tolerance": np.nan,
        "shown_dashboard": df["dashboard"].map(lambda v: None if v is None else str(v)),
        "shown_reference": df["reference"].map(lambda v: None if v is None else str(v)),
        "status": np.where(df["passed"].astype(bool), "PASS", "FAIL"), "note": df["note"],
    })
    CHECK_FRAMES.append(out)
    return out


def finding(severity, area, title, detail, *, stocks=None, evidence=None):
    FINDINGS.append({
        "severity": severity, "area": area, "title": title, "detail": detail,
        "stocks": ", ".join(stocks) if isinstance(stocks, (list, tuple, set)) else stocks, "evidence": evidence,
    })


def checks_frame():
    return pd.concat(CHECK_FRAMES, ignore_index=True) if CHECK_FRAMES else pd.DataFrame()


def summarise(frame, by=("section",)):
    if frame.empty:
        return frame
    t = frame.groupby(list(by))["status"].value_counts().unstack(fill_value=0)
    for s in ("PASS", "WARN", "FAIL"):
        if s not in t:
            t[s] = 0
    t = t[["PASS", "WARN", "FAIL"]]
    t["total"] = t.sum(axis=1)
    t["max_rel_diff"] = frame.groupby(list(by))["rel_diff"].max()
    return t


def style_status(df, col="status"):
    """Tint the status column (the text itself still says PASS / WARN / FAIL)."""
    colors = {"PASS": "#e3f4e3", "WARN": "#fdf0cf", "FAIL": "#f8dcdc", "CRITICAL": "#f8dcdc", "HIGH": "#fbe3d6", "MEDIUM": "#fdf0cf"}
    styler = df.style
    apply = getattr(styler, "map", None) or styler.applymap
    return apply(lambda v: f"background-color: {colors.get(v, '')}", subset=[c for c in df.columns if c == col])


def worst_point(js_vals, py_vals, tol_rel=1e-9, tol_abs=0.0):
    """Index of the largest gap (relative to tolerance) between two equally long series."""
    a, b = np.asarray(js_vals, float), np.asarray(py_vals, float)
    gap = np.abs(a - b)
    gap[np.isnan(a) & np.isnan(b)] = 0.0
    gap[np.isnan(a) ^ np.isnan(b)] = np.inf
    excess = gap - (tol_abs + tol_rel * np.nan_to_num(np.fmax(np.abs(a), np.abs(b))))
    i = int(np.argmax(excess)) if len(a) else 0
    return i, (a[i] if len(a) else np.nan), (b[i] if len(b) else np.nan)

## 1. Load the data file the dashboard reads

In [ ]:
def read_data_js(path):
    text = Path(path).read_text(encoding="utf-8").strip()
    prefix = "window.DASHBOARD_DATA = "
    if text.startswith(prefix):
        text = text[len(prefix):]
    return json.loads(text.rstrip(";").strip())


def arr(values):
    return np.array([np.nan if v is None else float(v) for v in values], dtype=float)


DATA = read_data_js(DATA_PATH)
META = DATA["meta"]
ISO = list(DATA["dates"])
DATES = pd.DatetimeIndex(pd.to_datetime(ISO))
N = len(DATES)
STOCK_IDS = [s["id"] for s in DATA["stocks"]]
STOCK_META = pd.DataFrame(
    [{k: s.get(k) for k in ("id", "name", "ticker", "exchange", "country", "currency", "subsector", "segment", "history_ticker", "splice_date", "note")} for s in DATA["stocks"]]
).set_index("id")
NAME = STOCK_META["name"].to_dict()
TRI_RAW = pd.DataFrame({s["id"]: arr(s["tri"]) for s in DATA["stocks"]}, index=DATES)
PX_RAW = pd.DataFrame({s["id"]: arr(s["px"]) for s in DATA["stocks"]}, index=DATES)
SERIES_RAW = pd.DataFrame({k: arr(v["values"]) for k, v in DATA["series"].items()}, index=DATES)
FUND = {s["id"]: (s.get("fund") or {}) for s in DATA["stocks"]}
AS_OF = META["as_of"]

display(Markdown(
    f"**As of {AS_OF}** (built {META.get('generated_at')}) · {N:,} trading days from {ISO[0]} · "
    f"{len(STOCK_IDS)} stocks · {SERIES_RAW.shape[1]} market series · synthetic: {META.get('synthetic')}"
))
status_rows = []
for sid, st in META.get("series_status", {}).items():
    status_rows.append({"series": sid, "label": st.get("label"), "status": st.get("status"), "source": st.get("source"), "code": st.get("code"),
                        "first": st.get("first"), "last": st.get("last"), "in data file": sid in DATA["series"],
                        "errors": "; ".join(st.get("errors", []))[:160]})
SERIES_STATUS = pd.DataFrame(status_rows).set_index("series")
display(SERIES_STATUS)
if META.get("warnings"):
    display(Markdown("**Build warnings:**\n" + "\n".join(f"- {w}" for w in META["warnings"])))

## 2. Layer 1 — input data audit
The dashboard can only be as right as its inputs. These checks look at the raw arrays in the data file before any analytics touch them.

### 2.1 Calendar and build metadata

In [ ]:
bdays = pd.bdate_range(DATES[0], DATES[-1])
missing_weekdays = bdays.difference(DATES)
gaps = np.diff(DATES.values).astype("timedelta64[D]").astype(int)
add_bool_checks([
    {"item": "dates strictly increasing and unique", "passed": DATES.is_monotonic_increasing and DATES.is_unique},
    {"item": "no weekend dates", "passed": bool((DATES.dayofweek < 5).all())},
    {"item": "meta.as_of equals the last date", "passed": AS_OF == ISO[-1], "dashboard": AS_OF, "reference": ISO[-1]},
    {"item": "meta.start equals the first date", "passed": META.get("start") == ISO[0], "dashboard": META.get("start"), "reference": ISO[0]},
    {"item": "every stock array has one value per date", "passed": all(len(s["tri"]) == N and len(s["px"]) == N for s in DATA["stocks"])},
    {"item": "every series array has one value per date", "passed": all(len(v["values"]) == N for v in DATA["series"].values())},
    {"item": "largest gap between dates <= 5 calendar days", "passed": int(gaps.max()) <= 5, "dashboard": int(gaps.max())},
    {"item": "data file is not synthetic demo data", "passed": not META.get("synthetic")},
], "Calendar & metadata", layer="1. Data")
print(f"Weekdays missing from the calendar (fewer than 10% of stocks traded): {len(missing_weekdays)}")
print(", ".join(d.strftime("%Y-%m-%d") for d in missing_weekdays))
for sid, st in META.get("series_status", {}).items():
    if st.get("status") == "failed":
        finding("MEDIUM", "Data", f"Source unavailable in this build: {st.get('label', sid)}",
                "; ".join(st.get("errors", []))[:400], evidence=sid)
    elif str(st.get("status", "")).startswith("stale"):
        finding("MEDIUM", "Data", f"Series reused from the previous build: {st.get('label', sid)}", st.get("status"), evidence=sid)
for w in META.get("warnings", []):
    if "history ticker" in w:
        finding("LOW", "Data", "Shorter price history after a corporate event", w, stocks=[w.split(":")[0]])

### 2.2 Market series: freshness, gaps, jumps and flat runs
Thresholds are for *review*, not errors: a 40bp daily gilt move did happen (September 2022). Hard failures are values that cannot be right (non-positive prices or FX, yields outside −2% to 20%, 100× jumps) or a series older than the three weeks the build allows.

In [ ]:
SERIES_KIND = {k: v.get("kind") for k, v in DATA["series"].items()}
JUMP_RULE = {  # (type of move, review threshold)
    "price": ("log return", 0.08), "fx": ("log return", 0.03), "rate": ("change (pp)", 0.30),
    "level": ("change (pts)", 15.0), "index": ("log return", 0.05),
}
series_rows, jump_rows = [], []
as_of_ts = pd.Timestamp(AS_OF)
for sid in SERIES_RAW.columns:
    v = SERIES_RAW[sid]
    kind = SERIES_KIND.get(sid, "index")
    valid = v.dropna()
    first, last = valid.index[0], valid.index[-1]
    inside = v.loc[first:last]
    run = inside.isna().astype(int)
    longest_gap = int(run.groupby((run != run.shift()).cumsum()).sum().max()) if run.any() else 0
    how, thr = JUMP_RULE.get(kind, ("log return", 0.08))
    if how == "log return":
        move = np.log(valid / valid.shift(1))
    else:
        move = valid.diff()
    big = move[move.abs() > thr]
    for dt, mv in big.items():
        jump_rows.append({"series": sid, "date": dt.date(), "move": mv, "type": how, "value": valid.loc[dt], "previous": valid.shift(1).loc[dt]})
    flat = (valid.diff() == 0).astype(int)
    flat_run = int(flat.groupby((flat != flat.shift()).cumsum()).sum().max()) if flat.any() else 0
    nonpos = int((valid <= 0).sum()) if kind in ("price", "fx", "index") else 0
    ratio = valid / valid.shift(1)
    hundred = int(((ratio > 50) | (ratio < 0.02)).sum()) if kind in ("price", "fx", "index") else 0
    out_of_range = int(((valid < -2) | (valid > 20)).sum()) if kind == "rate" else 0
    stale_days = (as_of_ts - last).days
    series_rows.append({
        "series": sid, "label": DATA["series"][sid].get("label"), "kind": kind, "source": DATA["series"][sid].get("source"),
        "code": DATA["series"][sid].get("code"), "first": first.date(), "last": last.date(), "days_stale": stale_days,
        "longest_gap_rows": longest_gap, "reviews": len(big), "worst_move": float(move.abs().max()),
        "longest_flat_run": flat_run, "non_positive": nonpos, "hundred_x_jumps": hundred, "outside_rate_range": out_of_range,
        "stale_flag": bool(DATA["series"][sid].get("stale")),
    })
SERIES_AUDIT = pd.DataFrame(series_rows).set_index("series")
SERIES_JUMPS = pd.DataFrame(jump_rows)
rows = []
for sid, r in SERIES_AUDIT.iterrows():
    rows += [
        {"item": "last observation within 21 days of as-of", "stock": sid, "passed": r.days_stale <= 21, "dashboard": f"{r.days_stale} days"},
        {"item": "no non-positive values", "stock": sid, "passed": r.non_positive == 0},
        {"item": "no 100x jumps", "stock": sid, "passed": r.hundred_x_jumps == 0},
        {"item": "yields between -2% and 20%", "stock": sid, "passed": r.outside_rate_range == 0},
        {"item": "no interior gap longer than the 7-day forward-fill", "stock": sid, "passed": r.longest_gap_rows <= 7, "dashboard": f"{r.longest_gap_rows} rows"},
    ]
add_bool_checks(rows, "Market series", layer="1. Data")
display(SERIES_AUDIT)
if len(SERIES_JUMPS):
    display(Markdown(f"**{len(SERIES_JUMPS)} market-series moves above the review thresholds** (look at these by eye; most will be genuine):"))
    display(SERIES_JUMPS.sort_values(["series", "date"]).reset_index(drop=True))
lagging = SERIES_AUDIT[SERIES_AUDIT.days_stale > 0]
for sid, r in lagging.iterrows():
    if r.days_stale > 7:
        finding("LOW", "Data", f"{sid} ends {r.days_stale} days before the as-of date",
                f"{r.label} last observation {r['last']}. Regressions that include it end at that date (rows after it are dropped).", evidence=sid)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 0.26 * len(SERIES_AUDIT) + 1.2))
s = SERIES_AUDIT.sort_values("days_stale")
colors = [C.critical if d > 21 else C.warning if d > 3 else C.s1 for d in s.days_stale]
ax.barh(s.index, s.days_stale, color=colors, height=0.6)
for y, (d, lab) in enumerate(zip(s.days_stale, s.index)):
    tag = "  stale (>21d)" if d > 21 else "  lagging" if d > 3 else ""
    ax.text(d, y, f"  {d}d{tag}", va="center", fontsize=8.5, color=C.ink2)
ax.axvline(21, color=C.axis, lw=1)
ax.text(21, len(s) - 0.4, " build limit 21d", color=C.muted, fontsize=8.5, va="bottom")
ax.set_title("Days between each series' last observation and the as-of date")
ax.set_xlabel("calendar days")
ax.grid(axis="y", visible=False)
show(fig, "Series freshness")

### 2.3 Share prices and total-return indices
For each stock: history length, trailing staleness, gaps longer than the 7-day forward-fill (which silently drop returns), flat runs (stale quotes), extreme daily moves, 100× unit jumps, the pence/pound conversion for London lines, and whether the header price and 52-week range match the price history.

In [ ]:
def runs_of_true(mask):
    """Lengths of consecutive True runs."""
    m = np.asarray(mask, dtype=int)
    if not m.any():
        return np.array([], dtype=int)
    edges = np.diff(np.r_[0, m, 0])
    return np.flatnonzero(edges == -1) - np.flatnonzero(edges == 1)


mkt_ret = np.log(SERIES_RAW["MKT"] / SERIES_RAW["MKT"].ffill().shift(1)) if "MKT" in SERIES_RAW else None
stock_rows, move_rows = [], []
for sid in STOCK_IDS:
    tri, px = TRI_RAW[sid], PX_RAW[sid]
    obs = tri.notna()
    f = FUND[sid]
    if not obs.any():
        stock_rows.append({"stock": sid, "obs": 0})
        continue
    first, last = obs.idxmax(), obs[::-1].idxmax()
    span = obs.loc[first:last]
    gaps_rows = runs_of_true(~span.to_numpy())
    t_obs = tri.dropna()
    lr = np.log(t_obs / t_obs.shift(1)).dropna()
    big = lr[lr.abs() > 0.20]
    for dt, mv in big.items():
        move_rows.append({"stock": sid, "date": dt.date(), "log_return": mv, "market_same_day": float(mkt_ret.get(dt, np.nan)) if mkt_ret is not None else np.nan})
    ratio = t_obs / t_obs.shift(1)
    hundred = int(((ratio > 50) | (ratio < 0.02)).sum())
    flat = runs_of_true((t_obs.diff() == 0).to_numpy())
    p_obs = px.dropna()
    year = p_obs[p_obs.index > p_obs.index[-1] - pd.Timedelta(days=365)]
    ccy = STOCK_META.loc[sid, "currency"]
    stock_rows.append({
        "stock": sid, "currency": ccy, "first": first.date(), "last": last.date(), "obs": int(obs.sum()),
        "days_stale": (as_of_ts - last).days, "gaps_over_7_rows": int((gaps_rows > 7).sum()), "longest_gap_rows": int(gaps_rows.max()) if gaps_rows.size else 0,
        "longest_flat_run": int(flat.max()) if flat.size else 0, "moves_over_20pct": len(big), "hundred_x_jumps": hundred,
        "non_positive": int((t_obs <= 0).sum() + (p_obs <= 0).sum()), "last_px": float(p_obs.iloc[-1]),
        "fund_price": f.get("price"), "fund_price_date": f.get("price_date"), "high_52w": f.get("high_52w"), "low_52w": f.get("low_52w"),
        "px_high_365d": float(year.max()), "px_low_365d": float(year.min()),
    })
STOCK_AUDIT = pd.DataFrame(stock_rows).set_index("stock")
STOCK_MOVES = pd.DataFrame(move_rows)

rows = []
for sid, r in STOCK_AUDIT.iterrows():
    rows += [
        {"item": "prices up to the as-of date", "stock": sid, "passed": r.days_stale == 0, "dashboard": str(r["last"]), "reference": AS_OF},
        {"item": "no gap longer than the 7-day forward-fill", "stock": sid, "passed": r.gaps_over_7_rows == 0, "dashboard": f"{r.gaps_over_7_rows} gaps, longest {r.longest_gap_rows} rows"},
        {"item": "no 100x jumps (pence/pound glitches)", "stock": sid, "passed": r.hundred_x_jumps == 0},
        {"item": "no non-positive prices", "stock": sid, "passed": r.non_positive == 0},
        {"item": "no run of 5+ identical closes", "stock": sid, "passed": r.longest_flat_run < 5, "dashboard": f"{r.longest_flat_run} days"},
    ]
    if r.currency == "GBP":
        rows.append({"item": "London price in pounds, not pence", "stock": sid, "passed": r.last_px < 100, "dashboard": f"{r.last_px:.4g}"})
add_bool_checks(rows, "Share prices", layer="1. Data")
add_checks([{"item": "header price = last close in history", "stock": sid, "dashboard": r.fund_price, "reference": r.last_px} for sid, r in STOCK_AUDIT.iterrows()],
           "Share prices", tol_rel=5e-6, display=lambda v: js_num(v, 2), layer="1. Data")
add_checks([{"item": "52-week high = max close over 365 days", "stock": sid, "dashboard": r.high_52w, "reference": r.px_high_365d} for sid, r in STOCK_AUDIT.iterrows()]
           + [{"item": "52-week low = min close over 365 days", "stock": sid, "dashboard": r.low_52w, "reference": r.px_low_365d} for sid, r in STOCK_AUDIT.iterrows()],
           "Share prices", tol_rel=5e-6, display=lambda v: js_num(v, 2), layer="1. Data")
display(STOCK_AUDIT)
if len(STOCK_MOVES):
    display(Markdown(f"**{len(STOCK_MOVES)} daily total-return moves above 20%** — confirm each against company news:"))
    display(STOCK_MOVES.sort_values("log_return").reset_index(drop=True))
short = STOCK_AUDIT[pd.to_datetime(STOCK_AUDIT["first"]) > pd.Timestamp(AS_OF) - pd.DateOffset(years=5)]
if len(short):
    finding("LOW", "Data", "Stocks with less than five years of prices",
            "5Y windows, 3Y/5Y returns and long attribution periods use shorter samples or show a dash for: "
            + ", ".join(f"{s} (from {r['first']})" for s, r in short.iterrows()), stocks=list(short.index))

### 2.3b One-day spikes that reverse the next day
A move of more than 10% that is almost fully reversed on the next trading day while the market is calm is usually a bad print, not news. Reversals above 20% (and at least 90% undone) are treated as probable data errors and their effect on the dashboard is measured in section 6b; smaller ones are listed for review.

In [ ]:
spike_rows = []
mkt_log = np.log(SERIES_RAW["MKT"] / SERIES_RAW["MKT"].ffill().shift(1)) if "MKT" in SERIES_RAW else pd.Series(0.0, index=DATES)
for sid in STOCK_IDS:
    t_obs = TRI_RAW[sid].dropna()
    r = np.log(t_obs / t_obs.shift(1)).dropna()
    for i in range(len(r) - 1):
        a, b = r.iloc[i], r.iloc[i + 1]
        m_ = mkt_log.get(r.index[i], np.nan)
        if abs(a) > 0.10 and abs(a + b) < 0.3 * abs(a) and (not np.isfinite(m_) or abs(m_) < 0.03):
            spike_rows.append({"stock": sid, "date": r.index[i].date().isoformat(), "move": a, "next day": b, "market that day": m_,
                               "price": float(PX_RAW[sid].get(r.index[i], np.nan)), "price before": float(PX_RAW[sid].dropna().loc[:r.index[i]].iloc[-2]),
                               "probable bad print": bool(abs(a) > 0.20 and abs(a + b) < 0.1 * abs(a))})
SPIKES = pd.DataFrame(spike_rows, columns=["stock", "date", "move", "next day", "market that day", "price", "price before", "probable bad print"])
BAD_PRINTS = [(r.stock, r.date) for r in SPIKES.itertuples() if r[-1]]
add_bool_checks([{"item": "no one-day spike above 20% that reverses the next day", "stock": sid, "passed": sid not in {s for s, _ in BAD_PRINTS},
                  "dashboard": ", ".join(d for s, d in BAD_PRINTS if s == sid) or None} for sid in STOCK_IDS], "Share prices", layer="1. Data")
display(SPIKES.style.format({"move": "{:+.1%}", "next day": "{:+.1%}", "market that day": "{:+.1%}"}) if len(SPIKES) else Markdown("No one-day spike-reversals."))
for sid, dt in BAD_PRINTS:
    r_ = SPIKES[(SPIKES.stock == sid) & (SPIKES.date == dt)].iloc[0]
    finding("HIGH", "Data", f"Probable bad price print: {NAME[sid]} on {dt}",
            f"The close of {r_.price:.4g} sits between {r_['price before']:.4g} the day before and a full recovery the day after "
            f"({r_.move:+.1%} then {r_['next day']:+.1%}, market {r_['market that day']:+.1%}). Daily statistics that include this date (GARCH volatility forecast and price ranges, "
            "daily-frequency betas, rolling volatility) are distorted; section 6b measures by how much. Fix: drop or replace the print in the pipeline (a one-day reversal filter "
            "next to fix_unit_glitches in prices.py).", stocks=[sid])
if len(SPIKES):
    show_s = SPIKES.head(8)
    fig, axes = plt.subplots(1, len(show_s), figsize=(3.1 * len(show_s), 2.7), squeeze=False)
    for ax, r_ in zip(axes[0], show_s.itertuples()):
        dt = pd.Timestamp(r_.date)
        win = PX_RAW[r_.stock].dropna().loc[dt - pd.Timedelta(days=14): dt + pd.Timedelta(days=14)]
        ax.plot(win.index, win.to_numpy(), color=C.s1, lw=2, marker="o", ms=3)
        ax.scatter([dt], [PX_RAW[r_.stock][dt]], s=46, color=C.critical if r_[-1] else C.s2, zorder=4)
        ax.set_title(f"{r_.stock} {r_.date}\n{'probable bad print' if r_[-1] else 'review'}", fontsize=9)
        ax.tick_params(axis="x", labelsize=7, rotation=45)
    fig.suptitle("One-day spikes that reverse (price, ±2 weeks)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    show(fig, "Spike reversals")

### 2.4 Total return versus price: dividend steps
Yahoo's adjusted close is the close scaled back by a factor for every dividend, so the ratio *total-return index / price* should only ever step **up**, on ex-dividend dates, by roughly dividend ÷ price. That gives an independent reading of each stock's dividends to set against the trailing dividend yield the dashboard shows, and exposes adjustment errors (a downward step means the two series disagree).

In [ ]:
div_rows = []
for sid in STOCK_IDS:
    both = pd.concat([TRI_RAW[sid], PX_RAW[sid]], axis=1, keys=["tri", "px"]).dropna()
    if len(both) < 30:
        continue
    ratio = both.tri / both.px
    step = ratio / ratio.shift(1) - 1
    ups = step[step > 0.001]
    downs = step[step < -0.005]
    last = both.index[-1]
    yr = ups[ups.index > last - pd.Timedelta(days=365)]
    implied_yield = float((1 - 1 / (1 + yr)).sum())  # dividend / previous close for each ex-date
    f = FUND[sid]
    div_rows.append({
        "stock": sid, "ex_dates_last_365d": len(yr), "implied_ttm_yield": implied_yield,
        "reported_dividend_yield": f.get("dividend_yield"), "dps_ttm": f.get("dps_ttm"),
        "down_steps": len(downs), "worst_down_step": float(downs.min()) if len(downs) else 0.0,
        "largest_up_step": float(ups.max()) if len(ups) else 0.0, "ratio_last": float(ratio.iloc[-1]),
    })
DIVIDENDS = pd.DataFrame(div_rows).set_index("stock")
DIVIDENDS["gap_pp"] = (DIVIDENDS.reported_dividend_yield - DIVIDENDS.implied_ttm_yield) * 100
rows = []
for sid, r in DIVIDENDS.iterrows():
    rows.append({"item": "TRI/price ratio never steps down by more than 0.5%", "stock": sid, "passed": r.down_steps == 0,
                 "dashboard": f"{r.down_steps} down-steps, worst {r.worst_down_step:.2%}"})
    rep = r.reported_dividend_yield
    ok = is_num(rep) and abs(rep - r.implied_ttm_yield) <= max(0.01, 0.3 * max(rep, r.implied_ttm_yield))
    rows.append({"item": "reported dividend yield close to the yield implied by TRI/price steps", "stock": sid, "passed": bool(ok) or not is_num(rep) and r.implied_ttm_yield < 0.005,
                 "dashboard": js_pct(rep, 1), "reference": js_pct(r.implied_ttm_yield, 1),
                 "note": "Implied yield uses each ex-date's price, so a gap of a few tenths of a point is normal; large gaps point to unit errors, specials or missing dividends."})
add_bool_checks(rows, "Dividends (TRI vs price)", layer="1. Data")
display(DIVIDENDS.sort_values("gap_pp"))

fig, ax = plt.subplots(figsize=(6.4, 5.2))
d = DIVIDENDS.dropna(subset=["reported_dividend_yield"])
lim = max(d.reported_dividend_yield.max(), d.implied_ttm_yield.max()) * 100 * 1.08
ax.plot([0, lim], [0, lim], color=C.axis, lw=1)
outlier = (d.gap_pp.abs() > 1) & (d.gap_pp.abs() > 30 * np.fmax(d.reported_dividend_yield, d.implied_ttm_yield))
ax.scatter(d.implied_ttm_yield[~outlier] * 100, d.reported_dividend_yield[~outlier] * 100, s=42, color=C.s1, edgecolor=C.surface, linewidth=1.5, label="consistent", zorder=3)
ax.scatter(d.implied_ttm_yield[outlier] * 100, d.reported_dividend_yield[outlier] * 100, s=48, color=C.s2, edgecolor=C.surface, linewidth=1.5, label="gap > 1pp and > 30%", zorder=4)
for sid, r in d[outlier].iterrows():
    ax.annotate(sid, (r.implied_ttm_yield * 100, r.reported_dividend_yield * 100), xytext=(5, 3), textcoords="offset points", fontsize=8.5, color=C.ink2)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel("yield implied by TRI/price steps, last 365 days (%)")
ax.set_ylabel("dividend yield shown on the dashboard (%)")
ax.set_title("Dividend yield: dashboard vs total-return data")
ax.legend(loc="upper left")
show(fig, "Dividend yield: dashboard vs total-return data")

### 2.4b Dividends applied 100× too small (pence versus pounds)
For some London lines Yahoo applies a dividend declared in **pounds** to a price quoted in **pence**, so every adjustment is a hundredth of its true size and the "total return" index is really a price index. The test: take the tiny steps in the TRI/price ratio (between 0.001% and 0.1%), scale them by 100, and compare with the dividend yield reported from the dividend history.

In [ ]:
div100 = []
for sid in STOCK_IDS:
    both = pd.concat([TRI_RAW[sid], PX_RAW[sid]], axis=1, keys=["tri", "px"]).dropna()
    if len(both) < 30:
        continue
    ratio = both.tri / both.px
    step = (ratio / ratio.shift(1) - 1).dropna()
    micro = step[(step > 1e-5) & (step < 1e-3)]
    normal = step[step >= 1e-3]
    last = both.index[-1]
    recent = micro[micro.index > last - pd.Timedelta(days=365)]
    implied_x100 = float((100 * recent / (1 + recent)).sum())
    rep = FUND[sid].get("dividend_yield")
    diagnosis = "ok"
    if len(normal) == 0 and len(micro) >= 2 and is_num(rep) and rep > 0.005 and abs(implied_x100 - rep) <= max(0.01, 0.3 * rep):
        diagnosis = "dividends applied 100x too small"
    elif len(normal) == 0 and is_num(rep) and rep > 0.005:
        diagnosis = "no dividends in the total-return series"
    div100.append({"stock": sid, "diagnosis": diagnosis, "tiny steps since 2019": len(micro), "normal steps since 2019": len(normal),
                   "tiny steps x100, last 365d": implied_x100, "reported dividend yield": rep,
                   "TRI/price drift since 2019": float(ratio.iloc[-1] / ratio.iloc[0] - 1)})
DIV_CHECK = pd.DataFrame(div100).set_index("stock")
DIV_100X = list(DIV_CHECK.index[DIV_CHECK.diagnosis == "dividends applied 100x too small"])
NO_DIVS = list(DIV_CHECK.index[DIV_CHECK.diagnosis == "no dividends in the total-return series"])
add_bool_checks([{"item": "total-return series carries the dividends", "stock": sid, "passed": r.diagnosis == "ok", "dashboard": r.diagnosis,
                  "reference": f"reported yield {js_pct(r['reported dividend yield'], 1)}"} for sid, r in DIV_CHECK.iterrows()], "Dividends (TRI vs price)", layer="1. Data")
display(DIV_CHECK[DIV_CHECK.diagnosis != "ok"].style.format({"tiny steps x100, last 365d": "{:.2%}", "reported dividend yield": "{:.2%}", "TRI/price drift since 2019": "{:+.2%}"})
        if (DIV_CHECK.diagnosis != "ok").any() else Markdown("Every total-return series carries its dividends."))
if DIV_100X:
    lse = all(str(STOCK_META.loc[s, "exchange"]).startswith("London") for s in DIV_100X)
    yld = DIV_CHECK.loc[DIV_100X, "reported dividend yield"]
    finding("CRITICAL", "Data", f"Total returns exclude dividends for {len(DIV_100X)} {'London-listed ' if lse else ''}stocks",
            "Yahoo's adjusted close for " + ", ".join(f"{NAME[s]} ({s})" for s in DIV_100X) + " applies each dividend 100x too small"
            + (" (pounds against a pence price)" if lse else "") + ": scaling the tiny adjustments by 100 reproduces the reported dividend yields. "
            "Every 'total return' for these stocks is in effect a price return, so returns, relative returns, momentum, attribution and the sector index "
            f"miss {yld.min():.1%}-{yld.max():.1%} a year of dividends (their reported yields). Section 6b lists every number on screen that changes. "
            "Fix in the pipeline: for GBp-quoted lines, rebuild the total-return index from the close and Yahoo's dividend series (after the pence/pound check already used "
            "for the dividend yield in fundamentals.py) instead of trusting Adj Close.", stocks=DIV_100X)
if NO_DIVS:
    finding("HIGH", "Data", "Total-return series without any dividend adjustment", "No dividend step at all although a dividend yield is reported: " + ", ".join(NO_DIVS), stocks=NO_DIVS)

### 2.5 FX used for market caps versus FX in the return series
Market caps in EUR use `meta.fx_latest` (the last Yahoo quote, which can be a weekend print), while returns use the FX series on the trading calendar.

In [ ]:
fx_rows = []
for ccy, sid in {"USD": "FX_USD", "GBP": "FX_GBP", "SEK": "FX_SEK", "CHF": "FX_CHF", "NOK": "FX_NOK"}.items():
    if sid in SERIES_RAW and ccy in META.get("fx_latest", {}):
        s = SERIES_RAW[sid].dropna()
        fx_rows.append({"item": f"fx_latest[{ccy}] vs last calendar value of {sid}", "stock": ccy, "dashboard": META["fx_latest"][ccy], "reference": float(s.iloc[-1]),
                        "note": f"series last date {s.index[-1].date()}; source last {META['series_status'].get(sid, {}).get('last')}"})
fx_checks = add_checks(fx_rows, "FX snapshot", tol_rel=5e-3, display=lambda v: js_num(v, 4), layer="1. Data")
display(fx_checks[["item", "dashboard", "reference", "rel_diff", "status", "note"]])
if (fx_checks.rel_diff > 1e-3).any():
    finding("INFO", "Data", "Market caps in EUR use a later FX quote than the returns",
            f"meta.fx_latest is Yahoo's last print (up to {fx_checks.rel_diff.max():.2%} away from the as-of close for "
            f"{', '.join(fx_checks.loc[fx_checks.rel_diff > 1e-3, 'stock'])}), so EUR market caps and cap weights use a slightly different rate than EUR returns. "
            "Immaterial at the precision shown; worth aligning to the as-of close.")

### 2.6 Where the data is: price availability by stock and month

In [ ]:
avail = TRI_RAW.notna().groupby(DATES.to_period("M")).mean()
order = [s for ss in META["subsectors"] for s in STOCK_IDS if STOCK_META.loc[s, "subsector"] == ss]
mat = avail[order].T
fig, ax = plt.subplots(figsize=(11, 0.17 * len(order) + 1.6))
im = ax.imshow(mat.to_numpy(), aspect="auto", cmap=SEQ_BLUE, vmin=0, vmax=1, interpolation="nearest")
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order, fontsize=7.5)
ticks = [i for i, p in enumerate(mat.columns) if p.month == 1]
ax.set_xticks(ticks)
ax.set_xticklabels([str(mat.columns[i].year) for i in ticks])
ax.grid(False)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
cb.set_label("share of trading days with a price", color=C.ink2)
cb.outline.set_visible(False)
ax.set_title("Price availability by stock (grouped by sub-sector) and month")
show(fig, "Price availability")

## 3. Layer 2 — run the dashboard's own engine
The dashboard computes everything in the browser with `docs/assets/analytics.js` and `docs/assets/model.js`. Rather than trusting a copy of that logic, this cell loads **those exact files** (and the formatting helpers in `ui.js`) into an isolated V8 context with Node.js, points them at the same data file, and calls every function the way `app.js` calls it: every model preset and helper model, every frequency × window × currency setting, all 55 stocks. The result (`ENGINE`) is the set of raw numbers the dashboard puts on screen.

The harness below is the only JavaScript written for this notebook; it contains no analytics of its own, only the calls (and the few small helpers `app.js` defines inline, such as the GARCH input window, rolling volatility and scenario presets, copied verbatim).

In [ ]:
HARNESS_JS = r"""/*
 * Dashboard engine extraction harness (used by the verification notebook).
 *
 * Evaluated in a plain JavaScript context after analytics.js, model.js, ui.js
 * and the data file, so FA, PFModel, PFUI and DASHBOARD_DATA are globals.
 * Every call below mirrors the call app.js makes for the same number, so the
 * output is what the dashboard computes before it is formatted for screen.
 */
globalThis.__serialize = function (obj) {
  return JSON.stringify(obj, function (key, value) {
    if (value && typeof value === "object" && ArrayBuffer.isView(value)) return Array.from(value);
    if (typeof value === "number" && !Number.isFinite(value)) return null;
    return value;
  });
};

globalThis.__runHarness = function (cfg) {
  "use strict";
  const FA = globalThis.FA;
  const M = globalThis.PFModel;
  const U = globalThis.PFUI;
  const DATA = globalThis.DASHBOARD_DATA;
  const isNum = FA.isNum;
  const timing = {};
  const timed = (name, fn) => {
    const t0 = Date.now();
    const r = fn();
    timing[name] = (Date.now() - t0) / 1000;
    return r;
  };
  const want = (block) => !cfg.blocks || cfg.blocks.includes(block);

  const ctx = timed("prepare", () => M.prepare(DATA));
  const STOCKS = ctx.stocks;
  const resolve = (ids) => M.resolveFactors(ctx, ids);
  const MODELS = {
    full: resolve(M.PRESETS.full.factors),
    relative: resolve(M.PRESETS.relative.factors),
    macro: resolve(M.MACRO_SPEC),
    ffc: resolve(M.PRESETS.ffc.factors),
    etf: resolve(M.PRESETS.etf.factors),
    capm: resolve(M.PRESETS.capm.factors),
    style: resolve(M.STYLE_SPEC),
    attrib: resolve(M.ATTRIB_SPEC),
    mktrates: resolve(["MKT", "RATES"]),
  };
  const DEF = cfg.default || { freq: "W", years: 3, ccy: "local" };
  const opts = (s) => ({ freq: s.freq, years: s.years, ccy: s.ccy });
  const keyOf = (s, model) => `${s.freq}|${s.years}|${s.ccy}|${model}`;
  // app.js median(): median of the finite values, NaN when there are none.
  const median = (values) => {
    const v = values.filter(isNum);
    return v.length ? FA.median(v) : NaN;
  };
  const peersOf = (st) => STOCKS.filter((s) => s.meta.subsector === st.meta.subsector);

  const out = {
    engine: "dashboard JS (analytics.js + model.js)",
    timing,
    as_of: DATA.meta.as_of,
    n: ctx.n,
    stock_ids: STOCKS.map((s) => s.id),
    models: MODELS,
    default: DEF,
    catalogue: M.FACTORS.map((f) => ({
      id: f.id,
      label: f.label,
      short: f.short,
      group: f.group,
      kind: f.kind,
      series: f.series === undefined ? null : f.series,
      convert: !!f.convert,
      orth: !!f.orth,
      unit: f.unit,
      shock: f.shock,
      shockLabel: f.shockLabel,
    })),
    constants: {
      PERIODS_PER_YEAR: M.PERIODS_PER_YEAR,
      ROLLING_WINDOW: M.ROLLING_WINDOW,
      FORECAST: M.FORECAST,
      EPISODES: M.EPISODES,
      PRESETS: M.PRESETS,
      MACRO_SPEC: M.MACRO_SPEC,
      STYLE_SPEC: M.STYLE_SPEC,
      ATTRIB_SPEC: M.ATTRIB_SPEC,
      MACRO_DRIVERS: M.MACRO_DRIVERS,
      SIGNALS: M.SIGNALS,
    },
  };

  // ------------------------------------------------------------ preparation
  if (want("prep")) {
    out.prep = {
      weekEnds: Array.from(ctx.weekEnd).flatMap((v, t) => (v ? [t] : [])),
      monthEnds: Array.from(ctx.monthEnd).flatMap((v, t) => (v ? [t] : [])),
      stocks: STOCKS.map((st) => ({
        id: st.id,
        rateSeries: st.rateSeries,
        first: st.first,
        last: st.last,
        obsCount: Array.from(st.obs).reduce((a, b) => a + b, 0),
        shares: st.shares,
        hasFx: !!st.fx,
      })),
      available: Object.fromEntries(M.FACTORS.map((f) => [f.id, M.available(ctx, f.id)])),
    };
  }
  if (want("peerIndex")) {
    out.peerIndex = timed("peerIndex", () => STOCKS.map((st) => ({ all: M.peerIndex(ctx, st, "all"), sub: M.peerIndex(ctx, st, "sub") })));
  }

  // ------------------------------------------------------------ regressions
  const fitSummary = (f, withSeries) => {
    const o = { ok: f.ok, error: f.error || null, n: f.n, factors: f.factors, dropped: f.dropped };
    if (!f.ok) return o;
    for (const k of [
      "start", "end", "alpha", "alphaT", "alphaAnn", "betas", "se", "t", "r2", "adjR2", "lags",
      "factorStd", "factorMean", "gammas", "risk", "volAnn", "specificVolAnn", "systematicVolAnn", "ppy", "factorCov",
    ]) o[k] = f[k];
    o.rowsFirst = f.rows[0];
    o.rowsLast = f.rows[f.rows.length - 1];
    o.rowsSum = f.rows.reduce((a, b) => a + b, 0);
    if (withSeries) {
      o.rows = f.rows;
      o.resid = f.resid;
      o.y = f.y;
    }
    return o;
  };
  const isDefault = (s) => s.freq === DEF.freq && s.years === DEF.years && s.ccy === DEF.ccy;
  if (want("fits")) {
    out.fits = {};
    timed("fits", () => {
      for (const s of cfg.grid) {
        for (const model of cfg.models) {
          out.fits[keyOf(s, model)] = STOCKS.map((st) =>
            fitSummary(M.fitModel(ctx, st, MODELS[model], { ...opts(s), hac: true }), isDefault(s))
          );
        }
      }
    });
  }
  // Fits at the default settings, exactly as app.js memoises them: fits(ids, hac).
  const defFits = new Map();
  const fits = (model, hac = true) => {
    const key = `${model}|${hac}`;
    if (!defFits.has(key)) defFits.set(key, M.fitUniverse(ctx, MODELS[model], { ...opts(DEF), hac }));
    return defFits.get(key);
  };
  const statsCache = {};
  const stats = (ccy) => statsCache[ccy] || (statsCache[ccy] = STOCKS.map((st) => M.stockStats(ctx, st, ccy)));

  // ------------------------------------------------------------ returns, risk, technicals
  if (want("stats")) out.stats = timed("stats", () => ({ local: stats("local"), EUR: stats("EUR") }));

  if (want("macro")) {
    out.macroSens = {};
    timed("macroSens", () => {
      for (const s of cfg.macroSettings || [DEF]) out.macroSens[`${s.freq}|${s.years}|${s.ccy}`] = STOCKS.map((st) => M.macroSensitivities(ctx, st, opts(s)));
    });
  }

  if (want("rolling")) {
    out.rolling = {};
    timed("rolling", () => {
      const specs = [
        ["model", "full", DEF.freq, 1],
        ["rates", "mktrates", DEF.freq, DEF.freq === "D" ? 5 : 1],
        ["macroW", "macro", "W", 1],
      ];
      for (const [name, model, freq, step] of specs) {
        out.rolling[name] = STOCKS.map((st) => M.rollingFit(ctx, st, MODELS[model], { freq, ccy: DEF.ccy, step }));
      }
    });
  }

  // ------------------------------------------------------------ attribution
  const ATTR_PERIODS = [
    ["1M", 1 / 12],
    ["3M", 0.25],
    ["6M", 0.5],
    ["YTD", "ytd"],
    ["1Y", 1],
    ["3Y", 3],
  ];
  const attrPeriod = (key) => {
    const end = ctx.asOfIndex;
    const p = ATTR_PERIODS.find((x) => x[0] === key) || ATTR_PERIODS[1];
    if (p[1] === "ytd") return { start: M.indexOnOrBefore(ctx.dates, `${Number(ctx.dates[end].slice(0, 4)) - 1}-12-31`), end };
    return { start: M.windowStart(ctx, end, p[1]), end };
  };
  if (want("attribution")) {
    out.attribution = {};
    timed("attribution", () => {
      for (const [mode, model] of [["sector", "attrib"], ["model", "full"]]) {
        for (const [key] of ATTR_PERIODS) {
          const { start, end } = attrPeriod(key);
          out.attribution[`${mode}|${key}`] = STOCKS.map((st) => {
            const a = M.attribution(ctx, st, MODELS[model], opts(DEF), start, end);
            if (!a.ok) return { ok: false, error: a.error, startIdx: start, endIdx: end };
            const lf = a.lastFit;
            return {
              ok: true,
              factors: a.factors,
              start: a.start,
              end: a.end,
              days: a.days,
              total: a.total,
              contributions: a.contributions,
              specific: a.specific,
              avgBeta: a.avgBeta,
              moves: a.moves,
              movesTo: a.movesTo,
              missing: a.missing,
              lastFitOk: !!(lf && lf.ok),
              lastFitSpecificVolAnn: lf && lf.ok ? lf.specificVolAnn : null,
              lastFitEnd: lf && lf.ok ? lf.end : null,
              path: (cfg.attrPathPeriods || ["3M"]).includes(key) ? a.path : undefined,
              startIdx: start,
              endIdx: end,
            };
          });
        }
      }
    });
  }

  // ------------------------------------------------------------ scenarios & episodes
  if (want("scenario")) {
    timed("scenario", () => {
      const all = fits("macro");
      const ids = MODELS.macro;
      const creditId = ids.includes("CREDIT") ? "CREDIT" : ids.includes("CREDIT_ETF") ? "CREDIT_ETF" : null;
      const presets = [
        ["Rates +50bp", { MKT: 0, RATES: 50, CREDIT: 0, FXL: 0 }],
        ["Rates −50bp", { MKT: 0, RATES: -50, CREDIT: 0, FXL: 0 }],
        ["Equity sell-off", { MKT: -10, RATES: 0, CREDIT: 0, FXL: 0 }],
        ["Risk-off", { MKT: -10, RATES: -25, CREDIT: creditId === "CREDIT_ETF" ? -3 : 150, FXL: 0 }],
        ["Stagflation", { MKT: -5, RATES: 75, CREDIT: creditId === "CREDIT_ETF" ? -2 : 100, FXL: 0 }],
      ];
      const shocksFor = (fit, scen) => {
        const s = {};
        if (scen.MKT) s.MKT = scen.MKT / 100;
        if (scen.RATES) s.RATES = scen.RATES;
        if (scen.CREDIT) {
          if (fit.factors.includes("CREDIT")) s.CREDIT = scen.CREDIT;
          if (fit.factors.includes("CREDIT_ETF")) s.CREDIT_ETF = scen.CREDIT / 100;
        }
        if (scen.FXL && fit.factors.includes("FXL")) s.FXL = scen.FXL / 100;
        return s;
      };
      out.scenario = { presets, creditId, results: {} };
      for (const [label, scen] of presets) {
        for (const cond of [true, false]) {
          out.scenario.results[`${label}|${cond}`] = STOCKS.map((st) => {
            const fit = all[st.i];
            if (!fit.ok) return null;
            const shocks = shocksFor(fit, scen);
            return { ...M.scenario(fit, shocks, cond), shocks };
          });
        }
      }
      out.scenario.residSd = all.map((f) => (f.ok ? f.specificVolAnn / Math.sqrt(f.ppy) : null));
    });
  }
  if (want("episodes")) {
    out.episodes = timed("episodes", () => {
      const all = fits("macro");
      return STOCKS.map((st) =>
        M.EPISODES.map((ep) => {
          const imp = M.episodeImpact(ctx, st, all[st.i], ep, DEF.ccy);
          if (!imp) return null;
          const a = M.indexOnOrBefore(ctx.dates, ep.start);
          const b = M.indexOnOrBefore(ctx.dates, ep.end);
          const peerActual = median(
            peersOf(st)
              .filter((p) => p !== st)
              .map((p) => {
                const L = DEF.ccy === "EUR" ? p.triEur : p.tri;
                return a >= p.first ? L[b] / L[a] - 1 : NaN;
              })
          );
          return { id: ep.id, ...imp, peerActual };
        })
      );
    });
  }

  // ------------------------------------------------------------ outlook
  if (want("forecast")) {
    out.forecast = timed("forecast", () => ({
      local: M.exposureForecast(ctx, MODELS.macro, { ccy: "local" }),
      EUR: M.exposureForecast(ctx, MODELS.macro, { ccy: "EUR" }),
    }));
  }
  // app.js garchOf(): GARCH(1,1) on up to three years of daily log total returns, quote currency.
  const garchAt = (st, T) => {
    const r = [];
    const dates = [];
    for (let t = Math.max(T - 756, st.first + 1); t <= T; t++) {
      if (!st.obs[t]) continue;
      const v = Math.log(st.tri[t] / st.tri[t - 1]);
      if (!isNum(v)) continue;
      r.push(v);
      dates.push(ctx.dates[t]);
    }
    const g = FA.garch11(r);
    return g ? { g, dates } : null;
  };
  const lastPrice = (st) => (isNum(st.px[st.last]) ? st.px[st.last] : st.fund.price);
  if (want("garch")) {
    out.garch = timed("garch", () =>
      STOCKS.map((st) => {
        const gv = st.last >= 0 ? garchAt(st, st.last) : null;
        if (!gv) return null;
        const g = gv.g;
        const price = lastPrice(st);
        const vol = (h) => Math.sqrt((FA.garchCumVar(g, h) / h) * 252);
        const cns = st.fund.consensus || {};
        const year = isNum(price) ? M.priceRange(price, g, 252, 1) : null;
        const zTarget = year && isNum(cns.target_mean) ? Math.log(cns.target_mean / price) / year.sd : NaN;
        const sd252 = Math.sqrt(FA.garchCumVar(g, 252));
        const zWatch = isNum(cns.target_mean) && isNum(price) ? Math.log(cns.target_mean / price) / sd252 : NaN;
        return {
          alpha: g.alpha,
          beta: g.beta,
          omega: g.omega,
          persistence: g.persistence,
          lrVar: g.lrVar,
          nextVar: g.nextVar,
          halfLife: g.halfLife,
          logLik: g.logLik,
          n: g.n,
          firstDate: gv.dates[0],
          lastDate: gv.dates[gv.dates.length - 1],
          vol21: vol(21),
          vol63: vol(63),
          vol252: vol(252),
          lrVol: Math.sqrt(g.lrVar * 252),
          price,
          range: Object.fromEntries([21, 63, 252].map((h) => [h, M.priceRange(price, g, h, 1)])),
          range196: M.priceRange(price, g, 252, 1.96),
          zTarget,
          probAbove: isNum(zWatch) ? 1 - FA.normCdf(zWatch) : NaN,
          condVarLast: g.condVar[g.condVar.length - 1],
          pathVar: Array.from({ length: 52 }, (_, k) => FA.garchVarAt(g, 5 * (k + 1))),
        };
      })
    );
  }
  if (want("garchBacktest")) {
    const H = cfg.garchHorizons || [1, 5, 21, 63, 126, 252];
    out.garchBacktest = timed("garchBacktest", () =>
      STOCKS.map((st) => {
        const res = [];
        const quarterly = cfg.garchDates !== "M";
        for (let T = 1; T < ctx.n - 1; T++) {
          if (!ctx.monthEnd[T] || T > st.last || !st.obs[T]) continue;
          if (quarterly && !["03", "06", "09", "12"].includes(ctx.dates[T].slice(5, 7))) continue;
          if (cfg.garchFullWindow && T - 756 < st.first + 1) continue;
          const gv = garchAt(st, T);
          if (!gv) continue;
          const g = gv.g;
          res.push({ T, date: ctx.dates[T], n: g.n, alpha: g.alpha, beta: g.beta, lrVar: g.lrVar, nextVar: g.nextVar, cum: H.map((h) => FA.garchCumVar(g, h)) });
        }
        return res;
      })
    );
    out.garchHorizons = H;
  }
  if (want("balance")) out.balance = STOCKS.map((st) => M.balanceSheetOutlook(st.fund));
  if (want("signals")) out.signals = timed("signals", () => M.signalBacktest(ctx));

  // ------------------------------------------------------------ peers & coverage
  if (want("chars")) {
    out.chars = timed("chars", () => M.characteristics(ctx, stats(DEF.ccy), fits("capm", false).map((f) => (f.ok ? f.betas.MKT : NaN))));
  }
  if (want("corr")) {
    timed("corr", () => {
      const allFits = fits("full");
      out.corr = STOCKS.map((st) => M.correlations(ctx, st, allFits, opts(DEF)));
      out.pairs = STOCKS.map((a) => STOCKS.map((b) => (a === b ? null : M.pairSpread(ctx, a, b))));
      // app.js weeklyReturns()/pairCorr() for the sub-sector correlation matrix.
      const weekly = (st) => {
        const end = ctx.asOfIndex;
        const start = M.windowStart(ctx, end, DEF.years);
        const pts = M.samplePoints(ctx, st, "W", start, end);
        const m = new Map();
        for (let k = 1; k < pts.length; k++) {
          const r = st.triEur[pts[k]] / st.triEur[pts[k - 1]] - 1;
          if (isNum(r)) m.set(pts[k], r);
        }
        return m;
      };
      const pairCorr = (a, b) => {
        const x = [];
        const y = [];
        for (const [k, v] of a) if (b.has(k)) {
          x.push(v);
          y.push(b.get(k));
        }
        return x.length > 20 ? FA.correlation(x, y) : NaN;
      };
      const W = STOCKS.map(weekly);
      out.corrMatrix = STOCKS.map((a) => STOCKS.map((b) => (a === b ? 1 : pairCorr(W[a.i], W[b.i]))));
    });
  }
  if (want("relval")) {
    timed("relval", () => {
      const capm = fits("capm", false);
      const macro = fits("macro");
      const s = stats(DEF.ccy);
      const METRICS = {
        pb: (st) => st.fund.pb,
        dy: (st) => st.fund.dividend_yield,
        ltv: (st) => st.fund.ltv,
        pe: (st) => st.fund.pe_forward,
        upside: (st) => st.fund.consensus && st.fund.consensus.upside,
        mcap: (st) => (isNum(st.fund.market_cap_eur) ? st.fund.market_cap_eur / 1e9 : NaN),
        beta: (st) => (capm[st.i].ok ? capm[st.i].betas.MKT : NaN),
        rate10: (st) => (macro[st.i].ok && isNum(macro[st.i].betas.RATES) ? macro[st.i].betas.RATES * 1000 : NaN),
        specvol: (st) => (macro[st.i].ok ? macro[st.i].specificVolAnn : NaN),
        vol: (st) => s[st.i].vol1y,
        r1y: (st) => s[st.i].ret.y1,
        r3m: (st) => s[st.i].ret.m3,
        mom: (st) => s[st.i].mom12_1,
      };
      const values = Object.fromEntries(Object.entries(METRICS).map(([k, fn]) => [k, STOCKS.map((st) => {
        const v = fn(st);
        return v === undefined ? null : v;
      })]));
      const lines = {};
      for (const kx of Object.keys(METRICS)) for (const ky of Object.keys(METRICS)) if (kx !== ky) lines[`${kx}|${ky}`] = FA.fitLine(values[kx], values[ky]);
      out.relval = { values, lines };
    });
  }

  // ------------------------------------------------------------ chart series (app.js helpers)
  if (want("charts")) {
    timed("charts", () => {
      const seriesPoints = (levels, start, end, rebase) => {
        const pts = [];
        let base = NaN;
        for (let t = start; t <= end; t++) {
          const v = levels[t];
          if (!isNum(v)) continue;
          if (!isNum(base)) base = v;
          pts.push([ctx.dates[t], rebase ? (v / base) * 100 : v]);
        }
        return pts;
      };
      const rollingVol = (levels, start, end, obs) => {
        const res = [];
        const win = 63;
        const rets = [];
        for (let t = Math.max(start - win, 1); t <= end; t++) {
          const r = levels[t] / levels[t - 1] - 1;
          if (obs && !obs[t]) continue;
          if (!isNum(r)) continue;
          rets.push(r);
          if (rets.length > win) rets.shift();
          if (t >= start && rets.length >= 40) res.push([ctx.dates[t], FA.std(rets) * Math.sqrt(252) * 100]);
        }
        return res;
      };
      const end = ctx.asOfIndex;
      const start = M.windowStart(ctx, end, DEF.years);
      out.charts = STOCKS.map((st) => {
        const own = DEF.ccy === "EUR" ? st.triEur : st.tri;
        const ddStart = Math.max(start, st.first);
        const pts = seriesPoints(own, ddStart, end, false);
        const dd = FA.drawdowns(pts.map((p) => p[1])).series;
        return {
          rollingVol: rollingVol(own, start, end, st.obs),
          drawdown: pts.map((p, k) => [p[0], dd[k] * 100]),
          perf: seriesPoints(own, Math.max(start, st.first), end, true),
        };
      });
    });
  }

  // ------------------------------------------------------------ number formatting (ui.js)
  if (want("formatTests") && U) {
    const vals = cfg.formatValues || [];
    out.formatTests = vals.map((x) => ({
      x,
      num0: U.num(x, 0),
      num1: U.num(x, 1),
      num2: U.num(x, 2),
      num3: U.num(x, 3),
      num1s: U.num(x, 1, true),
      num2s: U.num(x, 2, true),
      pct0: U.pct(x, 0),
      pct0s: U.pct(x, 0, true),
      pct1: U.pct(x, 1),
      pct1s: U.pct(x, 1, true),
      pct2s: U.pct(x, 2, true),
      pp: U.pp(x),
      bp: U.bp(x, 0),
      mult: U.mult(x),
      mult1: U.mult(x, 1),
      compact: U.compact(x),
      moneyGBP: U.money(x, "GBP"),
      moneySEK: U.money(x, "SEK"),
      mcEUR: U.moneyCompact(x, "EUR"),
      mcSEK: U.moneyCompact(x, "SEK"),
      sig: U.sig(x),
    }));
    out.ordinalTests = Array.from({ length: 125 }, (_, n) => U.ordinal(n));
    out.dateTests = ["2026-09-25", "2026-01-01", "2025-12-31"].map((d) => [U.date(d), U.date(d, false), U.monthYear(d)]);
  }
  return out;
};"""

RUNNER_JS = r"""// Loads the dashboard's own scripts into an isolated context and runs the harness.
// Usage: node runner.js <job.json>   (job: {files: [...], harness: {...}, out: "..."})
const fs = require("fs");
const vm = require("vm");

const job = JSON.parse(fs.readFileSync(process.argv[2], "utf8"));
const sandbox = { console };
vm.createContext(sandbox);
// The dashboard scripts attach to `self`/`window` when no CommonJS module is present.
vm.runInContext("var window = this; var self = this;", sandbox);
for (const file of job.files) vm.runInContext(fs.readFileSync(file, "utf8"), sandbox, { filename: file });
sandbox.__cfg = job.harness;
const t0 = Date.now();
const json = vm.runInContext("__serialize(__runHarness(__cfg))", sandbox);
fs.writeFileSync(job.out, json);
process.stdout.write(JSON.stringify({ ok: true, seconds: (Date.now() - t0) / 1000, bytes: json.length }) + "\n");"""

GRID = [{"freq": f, "years": y, "ccy": c} for f in ("D", "W", "M") for y in (1, 2, 3, 5) for c in ("local", "EUR") if not (f == "M" and y < 3)]
if not FULL_SETTINGS_GRID:
    GRID = [dict(DEFAULT_VIEW), {**DEFAULT_VIEW, "ccy": "EUR"}]
MODEL_KEYS = ["full", "relative", "macro", "ffc", "etf", "capm", "style", "attrib"]
_rng = np.random.default_rng(7)
FORMAT_VALUES = sorted(set(
    [0.0, 0.005, -0.005, 0.0049999, 0.125, -0.125, 0.0125, 1.005, 2.675, 999.95, 1234.5, -1234.5, 999999.5, 1e9, 2.5e9, -3.25e6, 12.5, 0.045, -0.0004, 4.5e-6]
    + list(np.round(_rng.normal(0, 0.05, 150), 7)) + list(np.round(_rng.lognormal(3, 3, 80), 4)) + list(-np.round(_rng.lognormal(0, 2, 40), 6))
))
ENGINE_CFG = {
    "default": dict(DEFAULT_VIEW),
    "grid": GRID,
    "models": MODEL_KEYS,
    "macroSettings": [dict(DEFAULT_VIEW), {**DEFAULT_VIEW, "ccy": "EUR"}],
    "attrPathPeriods": ["3M"],
    "garchFullWindow": False,
    "garchHorizons": [1, 5, 21, 63, 126, 252],
    "garchDates": GARCH_BACKTEST_DATES,
    "formatValues": FORMAT_VALUES,
    "blocks": ["prep", "peerIndex", "fits", "stats", "macro", "rolling", "attribution", "scenario", "episodes", "forecast", "garch",
               "balance", "signals", "chars", "corr", "relval", "charts", "formatTests"] + (["garchBacktest"] if RUN_BACKTESTS and GARCH_BACKTEST_IN_JS else []),
}


def run_engine(cfg):
    """Run the dashboard's JavaScript on the data file; returns (results, engine description)."""
    work = Path(tempfile.mkdtemp(prefix="pfl_engine_"))
    (work / "harness.js").write_text(HARNESS_JS, encoding="utf-8")
    (work / "runner.js").write_text(RUNNER_JS, encoding="utf-8")
    files = [str(DATA_PATH), *(str(DOCS / "assets" / f) for f in ("analytics.js", "model.js", "ui.js")), str(work / "harness.js")]
    out = work / "engine.json"
    (work / "job.json").write_text(json.dumps({"files": files, "harness": cfg, "out": str(out)}), encoding="utf-8")
    node = shutil.which(NODE_BINARY) or (NODE_BINARY if Path(NODE_BINARY).exists() else None)
    if node:
        version = subprocess.run([node, "--version"], capture_output=True, text=True).stdout.strip()
        proc = subprocess.run([node, "--max-old-space-size=6144", str(work / "runner.js"), str(work / "job.json")],
                              capture_output=True, text=True, timeout=7200)
        if proc.returncode != 0:
            raise RuntimeError(f"The dashboard engine failed under Node.js {version}:\n{proc.stderr[-4000:]}")
        info = json.loads(proc.stdout.strip().splitlines()[-1])
        result = json.loads(out.read_text(encoding="utf-8"))
        shutil.rmtree(work, ignore_errors=True)
        return result, f"Node.js {version}", info
    try:
        from py_mini_racer import MiniRacer
    except ImportError as exc:
        raise RuntimeError("Node.js is not on the PATH and mini-racer is not installed. Install Node.js 18+ "
                           "(https://nodejs.org) or run `pip install mini-racer`, or set RUN_ENGINE = False.") from exc
    js = MiniRacer()
    js.eval("var window = this; var self = this;")
    for f in files:
        js.eval(Path(f).read_text(encoding="utf-8"))
    js.eval("var __cfg = " + json.dumps(cfg) + ";")
    t0 = time.time()
    text = js.eval("__serialize(__runHarness(__cfg))")
    shutil.rmtree(work, ignore_errors=True)
    return json.loads(text), "mini-racer (embedded V8)", {"seconds": time.time() - t0, "bytes": len(text)}


ENGINE = None
_cache = os.environ.get("PFL_ENGINE_CACHE")  # optional: path of a saved engine run, to re-run the analysis without Node
if RUN_ENGINE:
    t0 = time.time()
    if _cache and Path(_cache).exists():
        ENGINE, ENGINE_NAME, ENGINE_INFO = json.loads(Path(_cache).read_text()), "cached engine run", {"bytes": Path(_cache).stat().st_size}
    else:
        ENGINE, ENGINE_NAME, ENGINE_INFO = run_engine(ENGINE_CFG)
        if _cache:
            Path(_cache).write_text(json.dumps(ENGINE))
    print(f"Dashboard engine ran under {ENGINE_NAME} in {time.time() - t0:.1f}s; {ENGINE_INFO['bytes'] / 1e6:.1f} MB of results.")
    print("Time per block (s):", {k: round(v, 2) for k, v in ENGINE["timing"].items()})
    assert ENGINE["as_of"] == AS_OF and ENGINE["stock_ids"] == STOCK_IDS, "engine and data file disagree on the stock list or date"
    print("Models as resolved by the engine (fallbacks applied):")
    for k, v in ENGINE["models"].items():
        print(f"  {k:<9} {', '.join(v)}")
else:
    print("RUN_ENGINE is False: sections that compare against the dashboard engine are skipped.")

## 4. The independent reference implementation
Written from the method as documented (README *How the numbers are calculated* and the dashboard's *Data & method* tab) with standard libraries: **statsmodels** for OLS and Newey–West errors, **numpy** QR for the many small regressions, **scipy** for rank correlation and maximum likelihood, **pandas** for calendars and forward-fills. Where the documentation leaves a convention open (for example which day a window starts on), the dashboard's convention is followed and named in a comment.

Two dashboard conventions look like defects rather than choices. They are switches, so section 8 can measure what each one moves:
- `month_arith="js"` reproduces JavaScript's `setUTCMonth`, which turns *31 March minus one month* into **3 March**; `"calendar"` clamps to the month end (28/29 February), as pandas, Excel and Bloomberg do.
- `fx_holiday_fix=False` reproduces how daily EUR returns are formed on the stock's own trading days, which drops the FX move on a local holiday; `True` measures from the previous trading day, so no FX move is lost.

In [ ]:
FFILL_LIMIT = 7
PPY = {"D": 252, "W": 52, "M": 12}
ROLLING_WINDOW = {"D": 126, "W": 52, "M": 36}
MIN_OBS = {"D": 120, "W": 40, "M": 24}
FREQ_LABEL = {"D": "daily", "W": "weekly", "M": "monthly"}


@dataclass(frozen=True)
class Factor:
    id: str
    kind: str          # asset | sector | rate | spread | active | active2 | fxlocal | level | ls
    series: object = None
    convert: bool = False  # converted to the stock's currency in the local view
    orth: bool = False     # orthogonalised to the market inside the window
    unit: str = "%"
    shock: float = 0.01    # the standard shock used to display the exposure


FACTOR_SPEC = [
    Factor("MKT", "asset", "MKT", convert=True),
    Factor("SECTOR", "sector", None, convert=True, orth=True),
    Factor("RATES", "rate", "local", unit="bp", shock=10),
    Factor("EUR10Y", "rate", "EUR10Y", unit="bp", shock=10),
    Factor("EUR2Y", "rate", "EUR2Y", unit="bp", shock=10),
    Factor("CURVE", "spread", ("EUR10Y", "EUR2Y"), unit="bp", shock=10),
    Factor("CREDIT", "rate", "EUR_HY_OAS", unit="bp", shock=25),
    Factor("CREDIT_ETF", "active2", ("HY_BOND", "GOVT_BOND"), shock=-0.01),
    Factor("FXL", "fxlocal"),
    Factor("EURUSD", "asset", "FX_USD"),
    Factor("OIL", "asset", "BRENT", shock=0.1),
    Factor("VOL", "level", "VOL", unit="pts", shock=5),
    Factor("SMB", "ls", "FF_SMB"), Factor("HML", "ls", "FF_HML"), Factor("RMW", "ls", "FF_RMW"),
    Factor("CMA", "ls", "FF_CMA"), Factor("WML", "ls", "FF_WML"),
    Factor("E_VAL", "active", "ETF_VALUE", orth=True), Factor("E_MOM", "active", "ETF_MOM", orth=True),
    Factor("E_QUAL", "active", "ETF_QUAL", orth=True), Factor("E_MINV", "active", "ETF_MINVOL", orth=True),
    Factor("E_SIZE", "active", "ETF_SIZE", orth=True),
]
FACTORS = {f.id: f for f in FACTOR_SPEC}
PRESETS = {
    "full": ["MKT", "RATES", "CREDIT", "FXL", "SMB", "HML", "WML"],
    "relative": ["MKT", "SECTOR", "RATES", "CREDIT", "SMB", "HML", "WML"],
    "macro": ["MKT", "RATES", "CREDIT", "FXL"],
    "ffc": ["MKT", "SMB", "HML", "RMW", "CMA", "WML"],
    "etf": ["MKT", "E_VAL", "E_MOM", "E_QUAL", "E_MINV", "E_SIZE"],
    "capm": ["MKT"],
}
MACRO_SPEC = ["MKT", "RATES", "CREDIT", "FXL"]
STYLE_SPEC = ["MKT", "SECTOR", "SMB", "HML", "RMW", "CMA", "WML"]
ATTRIB_SPEC = ["MKT", "SECTOR", "RATES", "CREDIT", "FXL"]
MODEL_SPECS = {**PRESETS, "style": STYLE_SPEC, "attrib": ATTRIB_SPEC, "mktrates": ["MKT", "RATES"]}
MACRO_DRIVERS = ["RATES", "EUR10Y", "EUR2Y", "CURVE", "CREDIT", "CREDIT_ETF", "FXL", "EURUSD", "OIL", "VOL"]
FALLBACKS = {"CREDIT": "CREDIT_ETF", "E_SIZE": "SMB"}
EPISODES = [
    {"id": "covid", "label": "COVID crash", "start": "2020-02-19", "end": "2020-03-18"},
    {"id": "vaccine", "label": "Vaccine rally", "start": "2020-11-06", "end": "2020-11-11"},
    {"id": "rates22", "label": "2022 rate shock", "start": "2021-12-31", "end": "2022-10-12"},
    {"id": "gilts22", "label": "UK mini-budget", "start": "2022-09-22", "end": "2022-09-27"},
    {"id": "banks23", "label": "Bank stress (SVB, Credit Suisse)", "start": "2023-03-08", "end": "2023-03-17"},
    {"id": "rally23", "label": "Late-2023 rates rally", "start": "2023-10-27", "end": "2023-12-27"},
    {"id": "fiscal25", "label": "German fiscal package", "start": "2025-03-04", "end": "2025-03-06"},
    {"id": "tariffs25", "label": "US tariff shock", "start": "2025-04-02", "end": "2025-04-07"},
]
SIGNAL_IDS = ["mom", "momrel", "rev", "lowvol", "high"]


def ffill_inside(a, limit=FFILL_LIMIT):
    """Forward-fill interior gaps of up to `limit` rows; nothing is filled after the last value."""
    a = np.asarray(a, dtype=float)
    out = pd.Series(a).ffill(limit=limit).to_numpy(copy=True)
    valid = np.flatnonzero(np.isfinite(a))
    if valid.size == 0:
        return np.full(a.shape, np.nan)
    out[valid[-1] + 1:] = np.nan
    return out


def nw_lags(n):
    """Newey-West lag rule floor(4 (n/100)^(2/9)), as used with statsmodels."""
    return max(0, int(math.floor(4 * (n / 100) ** (2 / 9))))


def ols_np(y, X):
    """OLS by QR on column-scaled data (X includes the constant). None when too short or collinear."""
    n, k = X.shape
    if n <= k:
        return None
    scale = np.sqrt((X ** 2).mean(axis=0))
    scale[scale == 0] = 1.0
    Q, R = np.linalg.qr(X / scale)
    d = np.abs(np.diag(R))
    if d.min() <= 1e-6 * d.max():
        return None
    coef = np.linalg.solve(R, Q.T @ y) / scale
    resid = y - X @ coef
    ssr = float(resid @ resid)
    sst = float(((y - y.mean()) ** 2).sum())
    Rinv = np.linalg.inv(R)
    cov = (Rinv @ Rinv.T) * (ssr / (n - k)) / np.outer(scale, scale)
    return SimpleNamespace(coef=coef, se=np.sqrt(np.maximum(np.diag(cov), 0)), r2=1 - ssr / sst if sst > 0 else np.nan, resid=resid, n=n, k=k)


def carino(returns, contribs):
    """Carino linking: scale each period's contributions by ln(1+r)/r relative to ln(1+R)/R."""
    r = np.asarray(returns, float)
    Cm = np.asarray(contribs, float)
    R = float(np.prod(1.0 + r) - 1.0)
    K = math.log(1.0 + R) / R if abs(R) > 1e-12 else 1.0
    kt = np.array([math.log(1.0 + v) / v if abs(v) > 1e-12 else 1.0 for v in r])
    return R, (kt[:, None] / K * Cm).sum(axis=0), kt


def carino_path(returns, contribs):
    r = np.asarray(returns, float)
    Cm = np.asarray(contribs, float)
    kt = np.array([math.log(1.0 + v) / v if abs(v) > 1e-12 else 1.0 for v in r])
    acc = np.cumsum(kt[:, None] * Cm, axis=0)
    growth = np.cumprod(1.0 + r) - 1.0
    K = np.array([math.log(1.0 + g) / g if abs(g) > 1e-12 else 1.0 for g in growth])
    return growth, acc / K[:, None]


def conditional_shocks(F, given):
    """E[x_u | x_s] = S_us S_ss^-1 x_s under joint normality; shocked factors keep their value."""
    F = np.asarray(F, float)
    k = F.shape[0]
    s = sorted(given)
    x = np.zeros(k)
    if not s:
        return x
    try:
        w = np.linalg.solve(F[np.ix_(s, s)], np.array([given[i] for i in s], float))
    except np.linalg.LinAlgError:
        w = None
    for u in range(k):
        x[u] = given[u] if u in given else (float(F[u, s] @ w) if w is not None else 0.0)
    return x


def zscores(values, lo=0.05, hi=0.95):
    v = np.asarray(values, float)
    fin = v[np.isfinite(v)]
    if fin.size == 0:
        return np.full(v.shape, np.nan)
    c = np.where(np.isfinite(v), np.clip(v, np.quantile(fin, lo), np.quantile(fin, hi)), np.nan)
    cf = c[np.isfinite(c)]
    m, s = cf.mean(), (cf.std(ddof=1) if cf.size > 1 else np.nan)
    return np.where(np.isfinite(c) & (s > 0), (c - m) / s, np.nan)


def pearson(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3:
        return np.nan
    a, b = a[ok] - a[ok].mean(), b[ok] - b[ok].mean()
    den = math.sqrt((a @ a) * (b @ b))
    return float(a @ b / den) if den > 0 else np.nan


def spearman(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3:
        return np.nan
    return pearson(sps.rankdata(a[ok]), sps.rankdata(b[ok]))


def rsi_wilder(values, n=14):
    """Wilder's RSI: simple average of the first n changes, then (n-1)/n smoothing."""
    out = np.full(len(values), np.nan)
    g_avg = l_avg = 0.0
    seen = 0
    for i in range(1, len(values)):
        a, b = values[i - 1], values[i]
        if not (np.isfinite(a) and np.isfinite(b)):
            continue
        ch = b - a
        g, l = max(ch, 0.0), max(-ch, 0.0)
        seen += 1
        if seen <= n:
            g_avg += g / n
            l_avg += l / n
            if seen < n:
                continue
        else:
            g_avg = (g_avg * (n - 1) + g) / n
            l_avg = (l_avg * (n - 1) + l) / n
        out[i] = 100.0 if l_avg == 0 else 100 - 100 / (1 + g_avg / l_avg)
    return out


def winsorise(v, lo=0.01, hi=0.99):
    v = np.asarray(v, float)
    return np.clip(v, np.quantile(v, lo), np.quantile(v, hi))


def fit_blend_weights(x1, x2, y):
    """Pooled regression y = b1 x1 + b2 x2 on winsorised data (gaps to the peer median), with b1, b2 >= 0 and b1 + b2 <= 1.

    x1: 2-year estimate - peer median, x2: 1-year estimate - peer median, y: realised - peer median.
    """
    w1, w2, wy = winsorise(x1), winsorise(x2), winsorise(y)
    s11, s22, s12, s1y, s2y = w1 @ w1, w2 @ w2, w1 @ w2, w1 @ wy, w2 @ wy
    det = s11 * s22 - s12 * s12
    b1 = (s1y * s22 - s2y * s12) / det if det > 0 else np.nan
    b2 = (s2y * s11 - s1y * s12) / det if det > 0 else np.nan
    if not (b1 >= 0 and b2 >= 0):  # keep the better single estimate, its weight clamped to [0, 1]
        one = lambda s, sy: min(max(sy / s if s > 0 else 0.0, 0.0), 1.0)
        c1, c2 = one(s11, s1y), one(s22, s2y)
        sse = lambda c, w: float(((wy - c * w) ** 2).sum())
        b1, b2 = (c1, 0.0) if sse(c1, w1) <= sse(c2, w2) else (0.0, c2)
    if b1 + b2 > 1:
        b1, b2 = b1 / (b1 + b2), b2 / (b1 + b2)
    rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
    err = wy - b1 * w1 - b2 * w2
    return {"long": float(b1), "short": float(b2), "peers": float(1 - b1 - b2), "n": len(wy), "rmse": rmse(err), "rmseLong": rmse(wy - w1),
            "rmseShort": rmse(wy - w2), "rmsePeers": rmse(wy), "err": err, "errLong": wy - w1}


def max_drawdown(levels):
    v = np.asarray(levels, float)
    v = v[np.isfinite(v)]
    if v.size == 0:
        return 0.0
    return float(min((v / np.maximum.accumulate(v) - 1).min(), 0.0))


def drawdown_series(levels):
    v = np.asarray(levels, float)
    out = np.full(v.shape, np.nan)
    ok = np.isfinite(v)
    out[ok] = v[ok] / np.maximum.accumulate(v[ok]) - 1
    return out

### 4.1 GARCH(1,1), price ranges and the balance-sheet outlook
The GARCH reference maximises the same Gaussian likelihood (variance targeting, first variance = sample variance) with a continuous optimiser from several starting points, so it can be compared with the dashboard's grid search both on the formulas (at the dashboard's own parameters) and on the optimum it finds.

In [ ]:
def garch_path(alpha, beta, e, lr):
    """Conditional variances h_t = w + a e_{t-1}^2 + b h_{t-1}, h_0 = long-run variance."""
    e2 = e * e
    h = np.empty_like(e)
    h[0] = lr
    if len(e) > 1:
        h[1:] = lfilter([1.0], [1.0, -beta], lr * (1 - alpha - beta) + alpha * e2[:-1], zi=[beta * lr])[0]
    return h


def garch_loglik(alpha, beta, e, lr):
    h = garch_path(alpha, beta, e, lr)
    return -0.5 * float(np.sum(np.log(h) + e * e / h))


def garch_fit(returns, alpha_min=0.001):
    r = np.asarray(returns, float)
    r = r[np.isfinite(r)]
    if r.size < 250:
        return None
    e = r - r.mean()
    lr = float(np.mean(e * e))

    def nll(p):
        a, b = p
        if a < alpha_min or b < 0 or a + b >= 0.999:
            return 1e12
        return -garch_loglik(a, b, e, lr)

    best = None
    for x0 in [(0.05, 0.90), (0.10, 0.85), (0.02, 0.96), (0.15, 0.70), (0.30, 0.50), (0.01, 0.50)]:
        res = spo.minimize(nll, x0, method="Nelder-Mead", options={"xatol": 1e-8, "fatol": 1e-10, "maxiter": 5000})
        if best is None or res.fun < best.fun:
            best = res
    a, b = map(float, best.x)
    return garch_state(a, b, e, lr)


def _loop_grid(lo, hi, step, floor):
    """Grid points exactly as a `for (x = max(lo, floor); x <= hi + 1e-12; x += step)` loop produces them."""
    out, x = [], max(lo, floor)
    while x <= hi + 1e-12:
        out.append(x)
        x += step
    return out


def garch_grid_fit(returns):
    """The dashboard's estimator re-implemented: the same likelihood, the same three-stage grid and the
    same first-maximum rule, vectorised over the grid. Used to verify the dashboard's parameters and to
    refit quickly at every backtest date."""
    r = np.asarray(returns, float)
    r = r[np.isfinite(r)]
    n = r.size
    if n < 250:
        return None
    e = r - r.mean()
    lr = float(np.mean((e - e.mean()) ** 2))
    if not lr > 0:
        return None
    best = [0.05, 0.9, -np.inf]

    def search(a0, a1, da, b0, b1, db):
        pts = [(a, b) for a in _loop_grid(a0, a1, da, 0.001) for b in _loop_grid(b0, b1, db, 0.0) if a + b < 0.999]
        if not pts:
            return
        A = np.array([p[0] for p in pts])
        B = np.array([p[1] for p in pts])
        W = lr * (1 - A - B)
        h = np.full(len(pts), lr)
        s = np.log(h) + (e[0] * e[0]) / h
        for t in range(1, n):
            h = W + A * e[t - 1] * e[t - 1] + B * h
            s += np.log(h) + (e[t] * e[t]) / h
        ll = -0.5 * s
        k = int(np.argmax(ll))
        if ll[k] > best[2]:
            best[:] = [pts[k][0], pts[k][1], float(ll[k])]

    search(0.01, 0.3, 0.01, 0.5, 0.98, 0.02)
    search(best[0] - 0.01, best[0] + 0.01, 0.002, best[1] - 0.02, best[1] + 0.02, 0.004)
    search(best[0] - 0.002, best[0] + 0.002, 0.0005, best[1] - 0.004, best[1] + 0.004, 0.001)
    return garch_state(best[0], best[1], e, lr)


def garch_state(a, b, e, lr):
    h = garch_path(a, b, e, lr)
    nxt = lr * (1 - a - b) + a * e[-1] ** 2 + b * h[-1]
    p = a + b
    return SimpleNamespace(alpha=a, beta=b, omega=lr * (1 - a - b), persistence=p, lrVar=lr, nextVar=float(nxt),
                           halfLife=math.log(0.5) / math.log(p) if 0 < p < 1 else np.nan, logLik=garch_loglik(a, b, e, lr), condVar=h, n=len(e))


def garch_cum_var(g, h):
    """Expected variance of the h-day cumulative return: sum over k of lr + p^(k-1) (next - lr)."""
    if h <= 0:
        return 0.0
    p = g.persistence
    decay = h if abs(1 - p) < 1e-12 else (1 - p ** h) / (1 - p)
    return h * g.lrVar + (g.nextVar - g.lrVar) * decay


def garch_var_at(g, h):
    return g.lrVar + g.persistence ** (h - 1) * (g.nextVar - g.lrVar)


def price_range(price, g, h, z):
    s = math.sqrt(garch_cum_var(g, h))
    return {"lo": price * math.exp(-z * s), "hi": price * math.exp(z * s), "sd": s}


def balance_sheet_outlook(f, changes=(-0.2, -0.1, -0.05, 0.0, 0.05, 0.1)):
    """IFRS equity as NAV, total assets less cash as the property portfolio (the dashboard's stated proxies)."""
    need = ["equity", "total_assets", "cash", "net_debt", "shares", "market_cap"]
    if not f or not all(is_num(f.get(k)) for k in need):
        return None
    assets = f["total_assets"] - f["cash"]
    if not (assets > 0 and f["shares"] > 0 and f["equity"] > 0 and f["market_cap"] > 0):
        return None
    price = f["price"] if is_num(f.get("price")) else f["market_cap"] / f["shares"]
    nav0 = f["equity"] / f["shares"]

    def row(ch):
        nav = (f["equity"] + ch * assets) / f["shares"]
        return {"change": ch, "nav": nav, "navChange": nav / nav0 - 1, "pnav": price / nav if nav > 0 else np.nan,
                "ltv": f["net_debt"] / (assets * (1 + ch))}

    implied = (price * f["shares"] - f["equity"]) / assets
    headroom = lambda lvl: f["net_debt"] / (lvl * assets) - 1 if f["net_debt"] > 0 else np.nan
    stressed = np.nan
    if is_num(f.get("ebitda")) and f["ebitda"] > 0 and is_num(f.get("interest_cover")) and f["interest_cover"] > 0 and is_num(f.get("total_debt")):
        stressed = f["ebitda"] / (f["ebitda"] / f["interest_cover"] + 0.01 * f["total_debt"])
    return {"assets": assets, "price": price, "implied": implied, "impliedRow": row(implied), "rows": [row(c) for c in changes],
            "ltvNow": f["net_debt"] / assets, "headroom50": headroom(0.5), "headroom60": headroom(0.6),
            "coverNow": f["interest_cover"] if is_num(f.get("interest_cover")) else np.nan, "coverStressed": stressed}

### 4.2 Returns, factors, regressions and everything built on them

In [ ]:
class Ref:
    """Independent re-implementation of the dashboard's analytics on the same data file."""

    def __init__(self, data, month_arith="js", fx_holiday_fix=False):
        self.month_arith, self.fx_fix = month_arith, fx_holiday_fix
        self.meta = data.get("meta", {})
        self.iso = np.array([str(d) for d in data["dates"]], dtype=object)  # plain str items (pandas 3 rejects numpy.str_)
        self.dates = pd.DatetimeIndex(pd.to_datetime(self.iso))
        self.n = n = len(self.iso)
        self.asof = n - 1
        self.daynum = ((self.dates - pd.Timestamp("1970-01-01")) // pd.Timedelta(days=1)).to_numpy().astype(int)
        self.series = {k: ffill_inside(arr(v["values"])) for k, v in data.get("series", {}).items()}
        wk = self.dates.to_period("W-SUN").asi8  # weeks run Monday to Sunday: sample on each week's last trading day
        mo = self.dates.to_period("M").asi8
        self.week_end = np.r_[wk[:-1] != wk[1:], True]
        self.month_end = np.r_[mo[:-1] != mo[1:], True]
        self.subsectors = self.meta.get("subsectors") or list(dict.fromkeys(s["subsector"] for s in data["stocks"]))
        fx_by = {"USD": "FX_USD", "GBP": "FX_GBP", "SEK": "FX_SEK", "CHF": "FX_CHF", "NOK": "FX_NOK"}
        local_rate = self.meta.get("local_rate_by_country", {})

        def last_valid(a):
            v = np.flatnonzero(np.isfinite(a))
            return int(v[-1]) if v.size else -1

        self.stocks = []
        for i, rec in enumerate(data["stocks"]):
            st = SimpleNamespace(i=i, id=rec["id"], rec=rec, fund=rec.get("fund") or {}, subsector=rec.get("subsector"),
                                 currency=rec.get("currency"), country=rec.get("country"), name=rec.get("name"))
            raw = arr(rec.get("tri", [None] * n))
            st.obs = np.isfinite(raw)
            st.tri = ffill_inside(raw)
            st.px = ffill_inside(arr(rec.get("px", [None] * n)))
            sid = fx_by.get(st.currency) if st.currency != "EUR" else None
            st.fx = self.series.get(sid) if sid else None
            st.triEur = st.tri / st.fx if st.fx is not None else st.tri
            st.pxEur = st.px / st.fx if st.fx is not None else st.px
            rs = local_rate.get(st.country)
            # A home yield that has not updated for about three weeks falls back to the EUR curve.
            st.rate_series = rs if (rs and rs in self.series and last_valid(self.series[rs]) >= n - 15) else "EUR10Y"
            idx = np.flatnonzero(st.obs)
            st.first = int(idx[0]) if idx.size else -1
            st.last = int(idx[-1]) if idx.size else -1
            if idx.size:
                pos = np.searchsorted(idx, np.arange(n), side="left") - 1
                st.prev_obs = np.where(pos >= 0, idx[np.clip(pos, 0, None)], -1)
            else:
                st.prev_obs = np.full(n, -1)
            self.stocks.append(st)
        self.by_id = {s.id: s for s in self.stocks}
        self._peer = {}
        self._cap_weights()

    # ---------------------------------------------------------------- returns and the sector index
    def levels(self, st, ccy):
        return st.triEur if ccy == "EUR" else st.tri

    def day_returns(self, st, ccy):
        """Return on each row: from the previous row (dashboard) or the previous trading day (fx_holiday_fix)."""
        L = self.levels(st, ccy)
        r = np.full(self.n, np.nan)
        r[1:] = L[1:] / L[:-1] - 1
        if self.fx_fix and ccy == "EUR" and st.fx is not None:
            p = st.prev_obs
            ok = (p >= 0) & (np.arange(self.n) - p <= FFILL_LIMIT + 1)
            r = L / np.where(ok, L[np.clip(p, 0, None)], np.nan) - 1
            r[0] = np.nan
        return r

    def _cap_weights(self):
        """Weights for the sector index: current share count times the previous day's EUR price."""
        n, N = self.n, len(self.stocks)
        last_px = np.array([st.pxEur[st.last] if st.last >= 0 else np.nan for st in self.stocks])
        shares = np.full(N, np.nan)
        for j, st in enumerate(self.stocks):
            sh, mc = st.fund.get("shares"), st.fund.get("market_cap_eur")
            if is_num(sh) and sh > 0:
                shares[j] = sh
            elif is_num(mc) and is_num(last_px[j]):
                shares[j] = mc / last_px[j]
        caps = shares * last_px
        med = np.nanmedian(caps) if np.isfinite(caps).any() else np.nan
        self.Rw = np.full((n, N), np.nan)
        self.Cw = np.full((n, N), np.nan)
        for j, st in enumerate(self.stocks):
            st.shares = shares[j] if np.isfinite(shares[j]) else (med / last_px[j] if np.isfinite(last_px[j]) else np.nan)
            r = self.day_returns(st, "EUR")
            c = np.full(n, np.nan)
            c[1:] = st.shares * st.pxEur[:-1]
            ok = st.obs & np.isfinite(r) & np.isfinite(c) & (c > 0)
            ok[0] = False
            self.Rw[ok, j] = r[ok]
            self.Cw[ok, j] = c[ok]

    def peer_index(self, st, scope="all"):
        """Cap-weighted EUR index of the coverage (or sub-sector) excluding the stock, base 100."""
        key = (st.i, scope)
        if key not in self._peer:
            cols = np.arange(len(self.stocks)) if scope == "all" else np.array([s.i for s in self.stocks if s.subsector == st.subsector])
            R, Cw = self.Rw[:, cols], self.Cw[:, cols]
            v = np.isfinite(R)
            S = np.where(v, Cw * R, 0.0).sum(axis=1)
            W = np.where(v, Cw, 0.0).sum(axis=1)
            own = np.isfinite(self.Rw[:, st.i])
            S = S - np.where(own, self.Cw[:, st.i] * self.Rw[:, st.i], 0.0)
            W = W - np.where(own, self.Cw[:, st.i], 0.0)
            r = np.where(W > 1e-9, S / np.where(W > 1e-9, W, 1.0), 0.0)
            r[0] = 0.0
            self._peer[key] = 100.0 * np.cumprod(1.0 + r)
        return self._peer[key]

    # ---------------------------------------------------------------- factors
    def available(self, fid):
        f = FACTORS.get(fid)
        if f is None:
            return False
        if f.kind == "rate" and f.series == "local":
            return "EUR10Y" in self.series
        if f.kind in ("sector", "fxlocal"):
            return True
        need = f.series if isinstance(f.series, tuple) else (f.series,)
        return all(s in self.series for s in need)

    def resolve(self, ids):
        out = []
        for fid in ids:
            use = fid
            if not self.available(use) and use in FALLBACKS and self.available(FALLBACKS[use]):
                use = FALLBACKS[use]
            if self.available(use) and use not in out:
                out.append(use)
        return out

    def applicable(self, st, fid):
        f, S = FACTORS[fid], self.series
        if f.kind == "fxlocal":
            return True if st.fx is not None else "not applicable: euro-denominated stock"
        if f.kind == "sector":
            return True if len(self.stocks) > 2 else "needs at least three stocks"
        if f.kind == "rate":
            sid = st.rate_series if f.series == "local" else f.series
            return True if sid in S else f"{sid} data unavailable"
        if f.kind == "spread":
            return True if all(s in S for s in f.series) else "yield data unavailable"
        if f.kind == "active":
            return True if f.series in S and "MKT" in S else f"{f.series} data unavailable"
        if f.kind == "active2":
            return True if all(s in S for s in f.series) else "bond ETF data unavailable"
        return True if f.series in S else f"{f.series} data unavailable"

    def factor_values(self, st, fid, ccy, A, B):
        """Factor value over (A, B] for arrays of start/end rows: returns, yield changes in bp, or level changes."""
        f, S = FACTORS[fid], self.series
        to_local = ccy == "local" and f.convert and st.fx is not None
        if f.kind == "asset":
            L = S[f.series] * st.fx if to_local else S[f.series]
            return L[B] / L[A] - 1
        if f.kind == "sector":
            L = self.peer_index(st, "all")
            L = L * st.fx if to_local else L
            return L[B] / L[A] - 1
        if f.kind == "ls":
            L = S[f.series]
            return L[B] / L[A] - 1
        if f.kind == "active":
            E, Mk = S[f.series], S["MKT"]
            return E[B] / E[A] - Mk[B] / Mk[A]
        if f.kind == "active2":
            P, Q = S[f.series[0]], S[f.series[1]]
            return P[B] / P[A] - Q[B] / Q[A]
        if f.kind == "rate":
            R = S[st.rate_series if f.series == "local" else f.series]
            return (R[B] - R[A]) * 100
        if f.kind == "spread":
            P, Q = S[f.series[0]], S[f.series[1]]
            return (P[B] - Q[B] - (P[A] - Q[A])) * 100
        if f.kind == "fxlocal":
            return st.fx[A] / st.fx[B] - 1
        if f.kind == "level":
            V = S[f.series]
            return V[B] - V[A]
        raise ValueError(f.kind)

    # ---------------------------------------------------------------- dates and sampling
    def shift_months(self, iso, months):
        d = pd.Timestamp(iso)
        if self.month_arith == "js":  # JavaScript setUTCMonth: day-of-month overflow rolls into the next month
            y, m = divmod(d.year * 12 + d.month - 1 - months, 12)
            return (pd.Timestamp(year=y, month=m + 1, day=1) + pd.Timedelta(days=d.day - 1)).strftime("%Y-%m-%d")
        return (d - pd.DateOffset(months=months)).strftime("%Y-%m-%d")

    def shift_years(self, iso, years):
        return self.shift_months(iso, js_round(years * 12))

    @staticmethod
    def shift_days(iso, days):
        return (pd.Timestamp(iso) - pd.Timedelta(days=days)).strftime("%Y-%m-%d")

    def on_or_before(self, iso):
        return int(np.searchsorted(self.iso, iso, side="right")) - 1

    def window_start(self, end, years):
        if not years or years == "max":
            return 0
        return max(0, self.on_or_before(self.shift_years(self.iso[end], years)))

    def sample_points(self, st, freq, start, end):
        """Sampling rows in [start, end]; the first element is the base (stock's own trading days for D)."""
        if freq == "D":
            idx = np.flatnonzero(st.obs)
            base, pts = idx[idx <= start][-1:], idx[(idx > start) & (idx <= end)]
        else:
            idx = np.flatnonzero(self.month_end if freq == "M" else self.week_end)
            base, pts = idx[idx < start][-1:], idx[(idx >= max(start, 0)) & (idx <= end)]
        return np.r_[base, pts].astype(int)

    # ---------------------------------------------------------------- regressions
    def panel(self, st, fids, freq="W", ccy="local", start=None, end=None, years=None, orth=True):
        end = self.asof if end is None else end
        start = self.window_start(end, years) if start is None else start
        factors, dropped = [], []
        for fid in fids:
            if fid not in FACTORS:
                continue
            ok = self.applicable(st, fid)
            (factors.append(fid) if ok is True else dropped.append({"id": fid, "reason": ok}))
        Y = self.levels(st, ccy)
        pts = self.sample_points(st, freq, start, end)
        A, B = pts[:-1], pts[1:]
        y = Y[B] / Y[A] - 1
        X = np.column_stack([self.factor_values(st, f, ccy, A, B) for f in factors]) if factors else np.empty((len(B), 0))
        keep = np.isfinite(y) & np.isfinite(X).all(axis=1)
        rows, y, X = B[keep], y[keep], X[keep]
        for j in reversed(range(len(factors))):  # a factor that does not move (e.g. a pegged currency) is dropped
            v = np.var(X[:, j], ddof=1) if len(y) > 1 else np.nan
            if not (v > 1e-14):
                dropped.append({"id": factors[j], "reason": "no variation in the window"})
                del factors[j]
                X = np.delete(X, j, axis=1)
        gammas = {}
        if orth and "MKT" in factors:
            mi = factors.index("MKT")
            m = X[:, mi]
            vm = np.var(m, ddof=1)
            for j, fid in enumerate(factors):
                if FACTORS[fid].orth and j != mi:
                    g = float(np.cov(X[:, j], m, ddof=1)[0, 1] / vm) if vm > 0 else 0.0
                    gammas[fid] = g
                    X[:, j] = X[:, j] - g * m
        return SimpleNamespace(factors=factors, X=X, y=y, rows=rows, dropped=dropped, gammas=gammas, start=start, end=end, points=len(pts) - 1)

    def fit(self, st, fids, freq="W", years=3, ccy="local", end=None, start=None, hac=True, fast=False):
        P = self.panel(st, fids, freq, ccy, start=start, end=end, years=years)
        n, kf = len(P.y), len(P.factors)
        out = {"ok": False, "n": n, "factors": list(P.factors), "dropped": P.dropped}
        if n < max(MIN_OBS[freq], kf + 5):
            out["error"] = f"Only {n} overlapping {FREQ_LABEL[freq]} observations"
            return out
        X = np.column_stack([np.ones(n), P.X])
        base = ols_np(P.y, X)
        if base is None:
            out["error"] = "The factors are collinear in this window"
            return out
        coef, se, resid, r2, lags = base.coef, base.se, base.resid, base.r2, 0
        if not fast:  # statsmodels: the textbook reference
            res = sm.OLS(P.y, X).fit()
            coef, resid, r2, se = res.params, res.resid, res.rsquared, res.bse
            if hac:
                lags = nw_lags(n)
                se = sm.OLS(P.y, X).fit(cov_type="HAC", cov_kwds={"maxlags": lags, "use_correction": True}).bse
        k = X.shape[1]
        t = np.where(se > 0, coef / np.where(se > 0, se, 1), np.nan)
        F = np.atleast_2d(np.cov(P.X, rowvar=False, ddof=1)) if kf else np.zeros((0, 0))
        spec_var = float(np.var(resid, ddof=1) * (n - 1) / max(n - k, 1))  # residual variance with n - k degrees of freedom
        b = np.asarray(coef[1:], float)
        contrib = b * (F @ b) if kf else np.array([])
        systematic = float(contrib.sum())
        total = systematic + spec_var
        ppy = PPY[freq]
        out.update({
            "ok": True, "start": self.iso[P.rows[0]], "end": self.iso[P.rows[-1]], "alpha": float(coef[0]), "alphaT": float(t[0]),
            "alphaAnn": float(coef[0]) * ppy, "betas": dict(zip(P.factors, b.tolist())), "se": dict(zip(P.factors, np.asarray(se[1:], float).tolist())),
            "t": dict(zip(P.factors, np.asarray(t[1:], float).tolist())), "r2": float(r2), "adjR2": 1 - (1 - float(r2)) * (n - 1) / (n - k),
            "lags": lags, "factorStd": dict(zip(P.factors, np.sqrt(np.diag(F)).tolist())), "factorMean": dict(zip(P.factors, P.X.mean(axis=0).tolist())),
            "gammas": P.gammas, "risk": {"total": total, "systematic": systematic, "specific": spec_var, "contrib": contrib.tolist(),
                                         "share": (contrib / total).tolist(), "specificShare": spec_var / total},
            "volAnn": math.sqrt(float(np.var(P.y, ddof=1)) * ppy), "specificVolAnn": math.sqrt(spec_var * ppy),
            "systematicVolAnn": math.sqrt(max(systematic, 0.0) * ppy), "ppy": ppy, "rows": P.rows, "resid": np.asarray(resid, float),
            "y": P.y, "X": P.X, "factorCov": F,
        })
        return out

    def rolling_fit(self, st, fids, freq="W", ccy="local", window=None, step=1):
        P = self.panel(st, fids, freq, ccy, start=0, end=self.asof, orth=False)
        W = window or ROLLING_WINDOW[freq]
        if len(P.y) < W + 2:
            return {"ok": False, "error": "Not enough history for a rolling window", "factors": []}
        mi = P.factors.index("MKT") if "MKT" in P.factors else -1
        orth_idx = [j for j, f in enumerate(P.factors) if FACTORS[f].orth and mi >= 0 and j != mi]
        ends, coefs, ses, r2s = [], [], [], []
        for e in range(W - 1, len(P.y), step):
            lo = e - W + 1
            ys, Xs = P.y[lo:e + 1], P.X[lo:e + 1].copy()
            if orth_idx:  # re-orthogonalise inside each window
                m = Xs[:, mi]
                vm = np.var(m, ddof=1)
                for j in orth_idx:
                    Xs[:, j] -= (np.cov(Xs[:, j], m, ddof=1)[0, 1] / vm if vm > 0 else 0.0) * m
            res = ols_np(ys, np.column_stack([np.ones(len(ys)), Xs]))
            if res is None:
                continue
            ends.append(e)
            coefs.append(res.coef)
            ses.append(res.se)
            r2s.append(res.r2)
        k = len(P.factors) + 1
        coefs, ses = np.array(coefs).reshape(-1, k), np.array(ses).reshape(-1, k)
        return {"ok": True, "window": W, "freq": freq, "factors": P.factors, "dates": [self.iso[P.rows[e]] for e in ends],
                "beta": {f: coefs[:, j + 1] for j, f in enumerate(P.factors)}, "se": {f: ses[:, j + 1] for j, f in enumerate(P.factors)},
                "alpha": coefs[:, 0], "r2": np.array(r2s)}

    def macro_sens(self, st, freq="W", years=3, ccy="local"):
        rows = []
        capm = self.fit(st, ["MKT"], freq, years, ccy)
        if capm["ok"]:
            rows.append({"id": "MKT", "total": capm["betas"]["MKT"], "totalT": capm["t"]["MKT"], "r2": capm["r2"], "partial": None, "partialT": None})
        sec = self.fit(st, ["SECTOR"], freq, years, ccy)
        if sec["ok"]:
            rows.append({"id": "SECTOR", "total": sec["betas"].get("SECTOR", np.nan), "totalT": sec["t"].get("SECTOR", np.nan), "r2": sec["r2"], "partial": None, "partialT": None})
        for fid in MACRO_DRIVERS:
            if self.applicable(st, fid) is not True:
                continue
            if fid == "CREDIT_ETF" and self.available("CREDIT"):
                continue  # the ETF proxy only stands in when the spread itself is missing
            tot = self.fit(st, [fid], freq, years, ccy)
            par = self.fit(st, ["MKT", fid], freq, years, ccy)
            if not tot["ok"]:
                continue
            rows.append({"id": fid, "total": tot["betas"].get(fid, np.nan), "totalT": tot["t"].get(fid, np.nan), "r2": tot["r2"],
                         "partial": par["betas"].get(fid, np.nan) if par["ok"] else None, "partialT": par["t"].get(fid, np.nan) if par["ok"] else None})
        for r in rows:
            sh = FACTORS[r["id"]].shock
            r["totalImpact"] = r["total"] * sh
            r["partialImpact"] = None if r["partial"] is None else r["partial"] * sh
        return rows

    # ---------------------------------------------------------------- returns, risk, technicals
    def stock_stats(self, st, ccy="local"):
        L = self.levels(st, ccy)
        end = st.last if st.last >= 0 else self.asof
        d = self.iso[end]
        at = self.on_or_before
        prev = end - 1
        while prev > 0 and not st.obs[prev]:
            prev -= 1
        H = {"d1": prev, "w1": at(self.shift_days(d, 7)), "m1": at(self.shift_years(d, 1 / 12)), "m3": at(self.shift_years(d, 0.25)),
             "m6": at(self.shift_years(d, 0.5)), "ytd": at(f"{int(d[:4]) - 1}-12-31"), "y1": at(self.shift_years(d, 1)),
             "y3": at(self.shift_years(d, 3)), "y5": at(self.shift_years(d, 5))}
        fv = lambda i: i if i >= st.first else -1
        pr = lambda Lv, a, b: float(Lv[b] / Lv[a] - 1) if (a >= 0 and b > a) else np.nan
        cov, sub, mkt = self.peer_index(st, "all"), self.peer_index(st, "sub"), self.series.get("MKT")
        m_end = end
        while mkt is not None and m_end > 0 and not np.isfinite(mkt[m_end]):
            m_end -= 1
        ret = {k: pr(L, fv(i), end) for k, i in H.items()}
        rel = {}
        for k, i in H.items():
            s = pr(st.triEur, fv(i), end)
            rel[k] = {"coverage": s - pr(cov, i, end), "subsector": s - pr(sub, i, end),
                      "market": pr(st.triEur, fv(i), m_end) - pr(mkt, i, m_end) if mkt is not None else np.nan}
        y_start = max(H["y1"], st.first)
        r_all = self.day_returns(st, ccy)
        tt = np.arange(y_start + 1, end + 1)
        daily = r_all[tt[st.obs[tt]]]  # stock's trading days in the last year (NaN kept, as the dashboard does)
        fin = daily[np.isfinite(daily)]
        srt = np.sort(fin)
        q = int(math.floor(0.05 * len(daily)))
        px = st.px[:end + 1]
        return {
            "end": end, "date": d, "horizons": H, "ret": ret, "rel": rel,
            "vol1y": float(np.std(fin, ddof=1) * math.sqrt(252)) if fin.size > 1 else np.nan,
            "mom12_1": pr(L, fv(at(self.shift_years(d, 1))), at(self.shift_years(d, 1 / 12))),
            "maxDD1y": max_drawdown(L[max(y_start, 0):end + 1]),
            "price": float(px[end]), "sma50": float(pd.Series(px).rolling(50).mean().iloc[-1]), "sma200": float(pd.Series(px).rolling(200).mean().iloc[-1]),
            "rsi14": float(rsi_wilder(px, 14)[end]),
            "var95": float(-srt[q]) if len(daily) > 50 and not np.isnan(daily).any() else np.nan,
            "es95": float(-srt[:q + 1].mean()) if len(daily) > 50 and not np.isnan(daily).any() else np.nan,
            "daily_has_nan": bool(np.isnan(daily).any()), "daily": daily,
        }

    # ---------------------------------------------------------------- attribution
    def attribution(self, st, fids, freq, years, ccy, start, end):
        Y = self.levels(st, ccy)
        prev = next((t for t in range(start, -1, -1) if st.obs[t]), -1)
        if prev < 0:
            return {"ok": False, "error": "No price history before the period"}
        days = []
        for t in range(start + 1, end + 1):
            if st.obs[t]:
                days.append((prev, t))
                prev = t
        if not days:
            return {"ok": False, "error": "No trading days in the period"}
        cache = {}

        def fit_for(t):  # exposures re-estimated at each month start from data up to the previous day
            key = self.iso[t][:7]
            if key not in cache:
                fom = t
                while fom > 0 and self.iso[fom - 1][:7] == key:
                    fom -= 1
                cache[key] = self.fit(st, fids, freq, years, ccy, end=max(fom - 1, 0), hac=False, fast=True)
            return cache[key]

        ids = [f for f in fids if f in FACTORS and self.applicable(st, f) is True]
        rets, rows, dts, used_pairs = [], [], [], []
        missing = {f: 0 for f in ids}
        beta_sum = {f: 0.0 for f in ids}
        last_fit = None
        for a, b in days:
            r = Y[b] / Y[a] - 1
            if not np.isfinite(r):
                continue
            fit = fit_for(b)
            if not fit["ok"]:
                continue
            last_fit = fit
            m = float(self.factor_values(st, "MKT", ccy, a, b))
            row, explained = [], 0.0
            for fid in ids:
                c = 0.0
                if fid in fit["betas"]:
                    v = float(self.factor_values(st, fid, ccy, a, b))
                    if np.isfinite(v) and fid in fit["gammas"]:
                        v = v - fit["gammas"][fid] * m if np.isfinite(m) else np.nan
                    if np.isfinite(v):
                        c = fit["betas"][fid] * v
                    else:
                        missing[fid] += 1
                    beta_sum[fid] += fit["betas"][fid]
                row.append(c)
                explained += c
            row.append(r - explained)  # stock-specific, including alpha
            rets.append(r)
            rows.append(row)
            dts.append(self.iso[b])
            used_pairs.append((a, b))
        if not rets:
            err = fit_for(days[0][1])
            return {"ok": False, "error": err.get("error") or "Not enough history to estimate exposures before this period"}
        total, linked, _ = carino(rets, rows)
        growth, path = carino_path(rets, rows)
        a0, b0 = days[0][0], days[-1][1]
        moves, moves_to = {}, {}
        for fid in ids:
            b = b0
            while b > a0 and not np.isfinite(self.factor_values(st, fid, ccy, a0, b)):
                b -= 1
            moves[fid] = float(self.factor_values(st, fid, ccy, a0, b)) if b > a0 else np.nan
            moves_to[fid] = self.iso[b] if b > a0 else None
        return {"ok": True, "factors": ids, "start": self.iso[a0], "end": self.iso[b0], "days": len(rets), "trading_days": len(days),
                "first_used": self.iso[used_pairs[0][0]], "total": total, "contributions": dict(zip(ids, linked[:len(ids)].tolist())),
                "specific": float(linked[len(ids)]), "avgBeta": {f: beta_sum[f] / len(rets) for f in ids}, "moves": moves, "movesTo": moves_to,
                "missing": missing, "lastFitSpecificVolAnn": last_fit["specificVolAnn"], "lastFitEnd": last_fit["end"],
                "path": {"date": dts, "total": growth, "factors": path[:, :len(ids)].sum(axis=1), "specific": path[:, len(ids)]},
                "period_return": float(Y[b0] / Y[a0] - 1)}

    # ---------------------------------------------------------------- scenarios and episodes
    @staticmethod
    def scenario(fit, shocks, conditional):
        ids = fit["factors"]
        given = {j: shocks[f] for j, f in enumerate(ids) if f in shocks and is_num(shocks[f])}
        x = conditional_shocks(fit["factorCov"], given) if conditional else np.array([given.get(j, 0.0) for j in range(len(ids))])
        impacts = {f: fit["betas"][f] * x[j] for j, f in enumerate(ids)}
        return {"total": float(sum(impacts.values())), "impacts": impacts, "moves": dict(zip(ids, x.tolist()))}

    def episode_impact(self, st, fit, ep, ccy):
        a, b = self.on_or_before(ep["start"]), self.on_or_before(ep["end"])
        if a < 0 or b <= a or not fit or not fit["ok"]:
            return None
        Y = self.levels(st, ccy)
        actual = Y[b] / Y[a] - 1
        mkt = float(self.factor_values(st, "MKT", ccy, a, b)) if "MKT" in fit["factors"] else np.nan
        moves, implied, complete = {}, 0.0, True
        for fid in fit["factors"]:
            v = float(self.factor_values(st, fid, ccy, a, b))
            if fid in fit["gammas"] and np.isfinite(mkt):
                v -= fit["gammas"][fid] * mkt
            moves[fid] = v
            if np.isfinite(v):
                implied += fit["betas"][fid] * v
            else:
                complete = False
        return {"actual": float(actual) if np.isfinite(actual) else np.nan, "implied": implied, "moves": moves, "complete": complete,
                "startDate": self.iso[a], "endDate": self.iso[b], "a": a, "b": b}

    # ---------------------------------------------------------------- cross-section
    def characteristics(self, stats, capm_betas):
        S = self.stocks
        g = lambda fn: np.array([_as_float(fn(st, i)) for i, st in enumerate(S)], dtype=float)
        f = lambda st, k: st.fund.get(k) if is_num(st.fund.get(k)) else np.nan
        defs = {
            "value": g(lambda st, i: 1 / f(st, "pb") if f(st, "pb") > 0 else np.nan),
            "yield": g(lambda st, i: f(st, "dividend_yield")),
            "momentum": g(lambda st, i: stats[i]["mom12_1"]),
            "size": g(lambda st, i: math.log(f(st, "market_cap_eur")) if f(st, "market_cap_eur") > 0 else np.nan),
            "volatility": g(lambda st, i: stats[i]["vol1y"]),
            "beta": np.asarray(capm_betas, float),
            "leverage": g(lambda st, i: f(st, "ltv")),
            "liquidity": g(lambda st, i: math.log(f(st, "adv_3m_eur")) if f(st, "adv_3m_eur") > 0 else np.nan),
        }
        return {k: {"raw": v, "z": zscores(v)} for k, v in defs.items()}

    def weekly_eur(self, st, freq, years):
        end = self.asof
        start = self.window_start(end, years)
        pts = self.sample_points(st, "W" if freq == "D" else freq, start, end)
        r = st.triEur[pts[1:]] / st.triEur[pts[:-1]] - 1
        return pd.Series(r, index=pts[1:]).dropna()

    def correlations(self, st, fits, freq, years):
        base = self.weekly_eur(st, freq, years)
        f0 = fits[st.i]
        base_res = pd.Series(f0["resid"], index=f0["rows"]) if f0["ok"] else pd.Series(dtype=float)
        out = []
        for o in self.stocks:
            if o.i == st.i:
                out.append({"id": o.id, "ret": 1.0, "resid": 1.0, "n": None})
                continue
            other = self.weekly_eur(o, freq, years)
            common = base.index.intersection(other.index)
            fo = fits[o.i]
            resid = np.nan
            if fo["ok"] and f0["ok"]:
                ores = pd.Series(fo["resid"], index=fo["rows"])
                ci = base_res.index.intersection(ores.index)
                resid = pearson(base_res[ci], ores[ci]) if len(ci) > 20 else np.nan
            out.append({"id": o.id, "ret": pearson(base[common], other[common]), "resid": resid, "n": len(common)})
        return out

    def pair_spread(self, a, b):
        end = self.asof
        start = self.window_start(end, 1)
        v = np.log(a.triEur[start:end + 1] / b.triEur[start:end + 1])
        v = v[np.isfinite(v)]
        if len(v) < 60:
            return None
        s = v.std(ddof=1)
        return {"z": (v[-1] - v.mean()) / s if s > 0 else np.nan, "change3m": math.exp(v[-1] - v[max(0, len(v) - 64)]) - 1}

    # ---------------------------------------------------------------- outlook: exposure forecasts
    def exposure_forecast(self, fids, ccy="local", long_w=104, short_w=52, horizon_w=52, step_w=4):
        ids = [f for f in fids if f in FACTORS and not FACTORS[f].orth]
        week_ends = np.flatnonzero(self.week_end)
        origins = week_ends[len(week_ends) - 1::-step_w][::-1]  # every four weeks back from the latest week
        now_k = len(origins) - 1
        est = []
        for st in self.stocks:
            P = self.panel(st, ids, "W", ccy, start=0, end=self.asof)
            rows, last = P.rows, len(P.rows) - 1
            Xc = np.column_stack([np.ones(len(P.y)), P.X])

            def betas(lo, hi):
                if lo < 0 or hi > last:
                    return None
                if self.daynum[rows[hi]] - self.daynum[rows[lo]] > (hi - lo + 1) * 7 * 1.25:
                    return None  # a window with missing weeks spans too many calendar days
                res = ols_np(P.y[lo:hi + 1], Xc[lo:hi + 1])
                return dict(zip(P.factors, res.coef[1:])) if res is not None else None

            e, per = -1, []
            for T in origins:
                while e + 1 <= last and rows[e + 1] <= T:
                    e += 1
                if e < 0 or self.daynum[T] - self.daynum[rows[e]] > 7:
                    per.append(None)
                    continue
                lg, sh, rl = betas(e - long_w + 1, e), betas(e - short_w + 1, e), betas(e + 1, e + horizon_w)
                per.append({"long": lg, "short": sh, "realised": rl} if (lg or sh) else None)
            est.append(per)
        members = {ss: [s.i for s in self.stocks if s.subsector == ss] for ss in self.subsectors}
        cache = {}

        def values_at(k, fid):
            if (k, fid) not in cache:
                vals = []
                for e_ in est:
                    x = e_[k]
                    v = np.nan if not x else (x["long"].get(fid, np.nan) if x["long"] else x["short"].get(fid, np.nan) if x["short"] else np.nan)
                    vals.append(v if is_num(v) else np.nan)
                cache[(k, fid)] = np.array(vals, float)
            return cache[(k, fid)]

        def target_for(st, k, fid):  # sub-sector peer median of the 2-year estimates (1-year when a peer is too new)
            vals = values_at(k, fid)
            sub = [vals[i] for i in members.get(st.subsector, []) if i != st.i and np.isfinite(vals[i])]
            if len(sub) >= 2:
                return float(np.median(sub))
            rest = np.delete(vals, st.i)
            rest = rest[np.isfinite(rest)]
            return float(np.median(rest)) if rest.size else np.nan

        weights, pooled, pairs = {}, {}, []
        for fid in ids:
            x1, x2, yy, who, kk = [], [], [], [], []
            for st in self.stocks:
                for k in range(now_k):
                    x = est[st.i][k]
                    if not x or not x["long"] or not x["short"] or not x["realised"]:
                        continue
                    a, b, r = x["long"].get(fid, np.nan), x["short"].get(fid, np.nan), x["realised"].get(fid, np.nan)
                    if not (is_num(a) and is_num(b) and is_num(r)):
                        continue
                    m = target_for(st, k, fid)
                    if not np.isfinite(m):
                        continue
                    x1.append(a - m)
                    x2.append(b - m)
                    yy.append(r - m)
                    who.append(st.i)
                    kk.append(k)
                    pairs.append({"factor": fid, "stock": st.id, "k": k, "origin": self.iso[origins[k]], "long": a, "short": b, "realised": r, "peers": m})
            if len(yy) < 30:
                continue
            w = fit_blend_weights(np.array(x1), np.array(x2), np.array(yy))
            weights[fid] = {k: w[k] for k in ("long", "short", "peers", "n", "rmse", "rmseLong", "rmseShort", "rmsePeers")}
            pooled[fid] = {"who": np.array(who), "k": np.array(kk), "err": w["err"], "errLong": w["errLong"]}
        stocks = []
        for st in self.stocks:
            x = est[st.i][now_k]
            out = {}
            for fid in weights:
                w = weights[fid]
                a = x["long"].get(fid, np.nan) if x and x["long"] else np.nan
                b = x["short"].get(fid, np.nan) if x and x["short"] else np.nan
                m = target_for(st, now_k, fid)
                if not np.isfinite(m) or (not is_num(a) and not is_num(b)):
                    continue
                if is_num(a) and is_num(b):
                    fc, basis = m + w["long"] * (a - m) + w["short"] * (b - m), "full"
                else:
                    fc, basis = m + (w["long"] + w["short"]) * ((b if is_num(b) else a) - m), "short"
                mine = np.flatnonzero(pooled[fid]["who"] == st.i)
                own = lambda e: float(np.sqrt(np.mean(np.square(e[mine])))) if len(mine) >= 6 else np.nan
                out[fid] = {"long": a, "short": b, "peers": m, "forecast": fc, "lo": fc - w["rmse"], "hi": fc + w["rmse"], "basis": basis,
                            "ownRmse": own(pooled[fid]["err"]), "ownRmseLong": own(pooled[fid]["errLong"]), "ownN": len(mine)}
            stocks.append(out)
        tested = [k for k in range(now_k) if any(e_[k] and e_[k]["long"] and e_[k]["realised"] for e_ in est)]
        return {"ok": bool(weights), "factors": list(weights), "weights": weights, "stocks": stocks,
                "firstOrigin": self.iso[origins[tested[0]]] if tested else None, "lastOrigin": self.iso[origins[tested[-1]]] if tested else None,
                "origins": len(tested), "origin_rows": origins, "pairs": pd.DataFrame(pairs), "est": est}

    # ---------------------------------------------------------------- outlook: price signals
    def signal_values(self, k, ends):
        S = len(self.stocks)
        out = {s: np.full(S, np.nan) for s in SIGNAL_IDS}
        if k < 12:
            return out
        t = ends[k]
        for st in self.stocks:
            if st.first > ends[k - 12] or not st.obs[t]:
                continue
            L, i = st.triEur, st.i
            out["mom"][i] = L[ends[k - 1]] / L[ends[k - 12]] - 1
            out["rev"][i] = L[t] / L[ends[k - 1]] - 1
            us = np.arange(max(t - 251, 1), t + 1)
            us = us[st.obs[us]]
            r = self.day_returns(st, "EUR")[us]
            r = r[np.isfinite(r)]
            if len(r) >= 200:
                out["lowvol"][i] = -np.std(r, ddof=1) * math.sqrt(252)
            p = st.px[us]
            p = p[np.isfinite(p)]
            peak = p.max() if p.size else -np.inf
            if peak > 0 and np.isfinite(st.px[t]):
                out["high"][i] = st.px[t] / peak - 1
        for ss in self.subsectors:
            mem = [s.i for s in self.stocks if s.subsector == ss]
            vals = out["mom"][mem]
            med = np.median(vals[np.isfinite(vals)]) if np.isfinite(vals).any() else np.nan
            out["momrel"][mem] = out["mom"][mem] - med
        return out

    def signal_backtest(self, min_stocks=15):
        ends = np.flatnonzero(self.month_end)
        ics = {s: [] for s in SIGNAL_IDS}
        spreads = {s: [] for s in SIGNAL_IDS}
        months = []
        for k in range(12, len(ends) - 1):
            vals = self.signal_values(k, ends)
            fwd = np.array([st.triEur[ends[k + 1]] / st.triEur[ends[k]] - 1 if st.obs[ends[k]] else np.nan for st in self.stocks])
            months.append(self.iso[ends[k]])
            for s in SIGNAL_IDS:
                x = vals[s]
                ok = np.isfinite(x) & np.isfinite(fwd)
                if ok.sum() < min_stocks:
                    ics[s].append(np.nan)
                    spreads[s].append(np.nan)
                    continue
                ics[s].append(spearman(x[ok], fwd[ok]))
                order = np.argsort(x[ok], kind="stable")
                fr = fwd[ok][order]
                third = len(fr) // 3
                spreads[s].append(fr[len(fr) - third:].mean() - fr[:third].mean())
        now = self.signal_values(len(ends) - 1, ends)
        signals = {}
        for s in SIGNAL_IDS:
            ic = np.array(ics[s], float)
            fin = ic[np.isfinite(ic)]
            m, sd, T = fin.mean(), fin.std(ddof=1), len(fin)
            cur = now[s]
            rk = pd.Series(cur).rank(method="average").to_numpy()
            cnt = np.isfinite(cur).sum()
            recent = ic[-36:]
            signals[s] = {"months": T, "ic": m, "t": m / (sd / math.sqrt(T)) if sd > 0 else np.nan, "hit": (fin > 0).mean() if T else np.nan,
                          "spread": np.nanmean(spreads[s]), "icRecent": np.nanmean(recent), "current": cur,
                          "percentile": np.where(np.isfinite(rk) & (cnt > 1), (rk - 1) / max(cnt - 1, 1), np.nan),
                          "ic_series": pd.Series(ic, index=pd.to_datetime(months)), "spread_series": pd.Series(spreads[s], index=pd.to_datetime(months))}
        return {"ok": len(months) > 0, "signals": signals, "first": months[0] if months else None, "last": months[-1] if months else None, "months": len(months)}


def js_fit_like(ref_fit):
    """Drop the arrays from a reference fit (for display)."""
    return {k: v for k, v in ref_fit.items() if k not in ("rows", "resid", "y", "X")}


t0 = time.time()
REF = Ref(DATA)                                   # mirrors every dashboard convention
REF_CAL = Ref(DATA, month_arith="calendar")       # calendar month arithmetic (section 8)
REF_FX = Ref(DATA, fx_holiday_fix=True)           # no FX move lost on local holidays (section 8)
print(f"Reference contexts prepared in {time.time() - t0:.1f}s")

## 5. Layer 2 — reconcile every number: dashboard engine vs independent reference
Tolerances are set far below what the screen shows (the dashboard displays two to three significant figures; the tolerances are seven to nine), so a PASS means the two implementations agree to machine precision. A difference beyond tolerance that disappears when both values are formatted the way the dashboard formats them is a WARN; one that changes the on-screen text is a FAIL.

In [ ]:
DV = dict(DEFAULT_VIEW)
DV_KEY = f"{DV['freq']}|{DV['years']}|{DV['ccy']}"
pct0, pct1, pct1s, pct2s, pct0s = (lambda v: js_pct(v, 0)), (lambda v: js_pct(v, 1)), (lambda v: js_pct(v, 1, True)), (lambda v: js_pct(v, 2, True)), (lambda v: js_pct(v, 0, True))
num0, num1, num2, num3 = (lambda v: js_num(v, 0)), (lambda v: js_num(v, 1)), (lambda v: js_num(v, 2)), (lambda v: js_num(v, 3))
ppf = lambda v: js_pp(v)


def exposure_fmt(fid):
    """How an exposure is shown: % move per standard shock for yield/level factors, a beta otherwise."""
    f = FACTORS.get(fid)
    if f is not None and f.unit in ("bp", "pts"):
        return lambda v: js_pct(v * f.shock, 2, True)
    return num2


def setting_label(s, model=None):
    return f"{s['freq']}|{s['years']}Y|{s['ccy']}" + (f"|{model}" if model else "")


def batch(rows, section, classes, layer="2. Engine"):
    """Split rows by their 'cls' and add each class with its own tolerance."""
    frames = []
    for cls, (tol_abs, tol_rel) in classes.items():
        sub = [r for r in rows if r.get("cls") == cls]
        if sub:
            frames.append(add_checks(sub, section, tol_abs=tol_abs, tol_rel=tol_rel, layer=layer))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


TOL = {  # (absolute, relative)
    "coef": (1e-12, 1e-7), "stat": (1e-9, 1e-6), "fit": (1e-9, 0.0), "exact": (1e-12, 1e-10), "level": (1e-12, 1e-9),
    "return": (1e-12, 1e-9), "count": (0.0, 0.0), "garch": (1e-12, 1e-8), "optimizer": (0.003, 0.0), "series": (1e-10, 1e-8),
}
REF_STATS = {c: [REF.stock_stats(st, c) for st in REF.stocks] for c in ("local", "EUR")}
_def_fits = {}


def ref_fits(model, s=None, hac=True):
    """Reference fits for every stock at a setting (cached for the default view)."""
    s = s or DV
    key = (model, s["freq"], s["years"], s["ccy"], hac)
    if key not in _def_fits:
        ids = REF.resolve(MODEL_SPECS[model])
        _def_fits[key] = [REF.fit(st, ids, s["freq"], s["years"], s["ccy"], hac=hac) for st in REF.stocks]
    return _def_fits[key]

### 5.1 Set-up: factor catalogue, resolved models, sampling calendar, sector-index weights

In [ ]:
if ENGINE:
    E = ENGINE
    rows = []
    cat = {c["id"]: c for c in E["catalogue"]}
    for f in FACTOR_SPEC:
        c = cat.get(f.id)
        ser = tuple(c["series"]) if c and isinstance(c["series"], list) else (c["series"] if c else None)
        same = c is not None and (c["kind"], ser, c["convert"], c["orth"], c["unit"], float(c["shock"])) == (f.kind, f.series, f.convert, f.orth, f.unit, float(f.shock))
        rows.append({"item": "factor definition matches the documented specification", "stock": f.id, "passed": same,
                     "dashboard": None if c is None else f"{c['kind']}, {c['series']}, {c['unit']}, shock {c['shock']}", "reference": f"{f.kind}, {f.series}, {f.unit}, shock {f.shock}"})
    rows.append({"item": "no factor in the engine that the specification lacks", "passed": set(cat) == set(FACTORS), "dashboard": ", ".join(sorted(set(cat) - set(FACTORS))) or "none"})
    for k in MODEL_KEYS + ["mktrates"]:
        rows.append({"item": "model after fallbacks", "stock": k, "passed": E["models"][k] == REF.resolve(MODEL_SPECS[k]),
                     "dashboard": ", ".join(E["models"][k]), "reference": ", ".join(REF.resolve(MODEL_SPECS[k]))})
    rows.append({"item": "week-end sampling rows", "passed": E["prep"]["weekEnds"] == np.flatnonzero(REF.week_end).tolist(), "dashboard": len(E["prep"]["weekEnds"]), "reference": int(REF.week_end.sum())})
    rows.append({"item": "month-end sampling rows", "passed": E["prep"]["monthEnds"] == np.flatnonzero(REF.month_end).tolist(), "dashboard": len(E["prep"]["monthEnds"]), "reference": int(REF.month_end.sum())})
    for fid, av in E["prep"]["available"].items():
        rows.append({"item": "factor available in this build", "stock": fid, "passed": bool(av) == REF.available(fid), "dashboard": av, "reference": REF.available(fid)})
    for j, st in enumerate(REF.stocks):
        p = E["prep"]["stocks"][j]
        rows += [
            {"item": "home 10Y series used", "stock": st.id, "passed": p["rateSeries"] == st.rate_series, "dashboard": p["rateSeries"], "reference": st.rate_series},
            {"item": "first / last trading row", "stock": st.id, "passed": (p["first"], p["last"]) == (st.first, st.last), "dashboard": (p["first"], p["last"]), "reference": (st.first, st.last)},
            {"item": "number of trading days", "stock": st.id, "passed": p["obsCount"] == int(st.obs.sum()), "dashboard": p["obsCount"], "reference": int(st.obs.sum())},
            {"item": "converted with an FX series", "stock": st.id, "passed": p["hasFx"] == (st.fx is not None), "dashboard": p["hasFx"], "reference": st.fx is not None},
        ]
    add_bool_checks(rows, "Set-up")
    add_checks([{"item": "sector-index share count", "stock": st.id, "dashboard": E["prep"]["stocks"][j]["shares"], "reference": st.shares} for j, st in enumerate(REF.stocks)],
               "Set-up", tol_rel=1e-12)
    rows = []
    for j, st in enumerate(REF.stocks):
        for scope in ("all", "sub"):
            js = np.asarray(E["peerIndex"][j][scope], float)
            py = REF.peer_index(st, scope)
            i, a, b = worst_point(js, py, 1e-10)
            rows.append({"item": f"peer index ({'coverage' if scope == 'all' else 'sub-sector'}), worst date", "stock": st.id, "dashboard": a, "reference": b, "note": f"{REF.iso[i]} of {len(js)} dates"})
    add_checks(rows, "Sector & peer indices", tol_rel=1e-10, display=lambda v: js_num(v, 1))
    display(summarise(checks_frame().query("layer == '2. Engine'")))

### 5.2 Regressions: every model × frequency × window × currency
Coefficients, Newey–West t-statistics, R², alpha, volatilities, the risk decomposition and the exact sample (which weeks entered) for every stock. This is the engine behind the factor profile, the regression table, the exposure strip, the coverage heatmap and every "rate sensitivity" and "beta" on the page.

In [ ]:
def compare_fit(js, py, stock, setting):
    b, n = [], []
    b.append({"item": "estimable", "stock": stock, "setting": setting, "passed": bool(js["ok"]) == bool(py["ok"]), "dashboard": js["ok"], "reference": py["ok"],
              "note": js.get("error") or py.get("error")})
    b.append({"item": "observations", "stock": stock, "setting": setting, "passed": js["n"] == py["n"], "dashboard": js["n"], "reference": py["n"]})
    if not (js["ok"] and py["ok"]):
        return b, n
    rows_py = np.asarray(py["rows"])
    b += [
        {"item": "same sample rows", "stock": stock, "setting": setting, "passed": (js["rowsFirst"], js["rowsLast"], js["rowsSum"]) == (int(rows_py[0]), int(rows_py[-1]), int(rows_py.sum())),
         "dashboard": f"{js['start']}..{js['end']}", "reference": f"{py['start']}..{py['end']}"},
        {"item": "factors kept", "stock": stock, "setting": setting, "passed": js["factors"] == py["factors"], "dashboard": ",".join(js["factors"]), "reference": ",".join(py["factors"])},
        {"item": "factors dropped", "stock": stock, "setting": setting, "passed": sorted(d["id"] for d in js["dropped"]) == sorted(d["id"] for d in py["dropped"]),
         "dashboard": ",".join(sorted(d["id"] for d in js["dropped"])), "reference": ",".join(sorted(d["id"] for d in py["dropped"]))},
        {"item": "Newey-West lags", "stock": stock, "setting": setting, "passed": js["lags"] == py["lags"], "dashboard": js["lags"], "reference": py["lags"]},
    ]
    common = dict(stock=stock, setting=setting)
    n += [
        {**common, "item": "alpha (per period)", "dashboard": js["alpha"], "reference": py["alpha"], "cls": "coef"},
        {**common, "item": "alpha annualised", "dashboard": js["alphaAnn"], "reference": py["alphaAnn"], "cls": "coef", "fmt": pct1s},
        {**common, "item": "alpha t", "dashboard": js["alphaT"], "reference": py["alphaT"], "cls": "stat", "fmt": num1},
        {**common, "item": "R2", "dashboard": js["r2"], "reference": py["r2"], "cls": "fit", "fmt": num2},
        {**common, "item": "adjusted R2", "dashboard": js["adjR2"], "reference": py["adjR2"], "cls": "fit", "fmt": num2},
        {**common, "item": "volatility (window)", "dashboard": js["volAnn"], "reference": py["volAnn"], "cls": "coef", "fmt": pct1},
        {**common, "item": "specific volatility", "dashboard": js["specificVolAnn"], "reference": py["specificVolAnn"], "cls": "coef", "fmt": pct1},
        {**common, "item": "systematic volatility", "dashboard": js["systematicVolAnn"], "reference": py["systematicVolAnn"], "cls": "coef", "fmt": pct1},
        {**common, "item": "specific share of variance", "dashboard": js["risk"]["specificShare"], "reference": py["risk"]["specificShare"], "cls": "fit", "fmt": pct0},
    ]
    for j, f in enumerate(js["factors"]):
        if f not in py["betas"]:
            continue
        n += [
            {**common, "item": f"beta {f}", "dashboard": js["betas"][f], "reference": py["betas"][f], "cls": "coef", "fmt": exposure_fmt(f)},
            {**common, "item": f"t {f}", "dashboard": js["t"][f], "reference": py["t"][f], "cls": "stat", "fmt": num2},
            {**common, "item": f"se {f}", "dashboard": js["se"][f], "reference": py["se"][f], "cls": "stat"},
            {**common, "item": f"factor sd {f}", "dashboard": js["factorStd"][f], "reference": py["factorStd"][f], "cls": "coef"},
            {**common, "item": f"1-sd impact {f}", "dashboard": js["betas"][f] * js["factorStd"][f], "reference": py["betas"][f] * py["factorStd"][f], "cls": "coef", "fmt": pct2s},
            {**common, "item": f"variance share {f}", "dashboard": js["risk"]["share"][j], "reference": py["risk"]["share"][j], "cls": "fit", "fmt": pct0},
        ]
        if f in js["gammas"]:
            n.append({**common, "item": f"orthogonalisation gamma {f}", "dashboard": js["gammas"][f], "reference": py["gammas"].get(f), "cls": "coef"})
    return b, n


if ENGINE:
    t0 = time.time()
    bool_rows, num_rows = [], []
    for key, results in E["fits"].items():
        freq, years, ccy, model = key.split("|")
        s = {"freq": freq, "years": int(years), "ccy": ccy}
        ids = REF.resolve(MODEL_SPECS[model])
        is_def = s == DV
        for j, st in enumerate(REF.stocks):
            py = ref_fits(model)[j] if is_def else REF.fit(st, ids, freq, int(years), ccy, hac=True)
            bb, nn = compare_fit(results[j], py, st.id, setting_label(s, model))
            bool_rows += bb
            num_rows += nn
    add_bool_checks(bool_rows, "Regressions")
    REG = batch(num_rows, "Regressions", {k: TOL[k] for k in ("coef", "stat", "fit")})
    print(f"{len(E['fits'])} model x setting combinations, {len(bool_rows) + len(num_rows):,} comparisons in {time.time() - t0:.0f}s")
    display(summarise(checks_frame().query("section == 'Regressions'"), by=("item",)).sort_values("FAIL", ascending=False).head(25))

### 5.3 Returns, relative returns, volatility, drawdown, VaR and technicals (both currency views)

In [ ]:
HORIZONS = ["d1", "w1", "m1", "m3", "m6", "ytd", "y1", "y3", "y5"]
if ENGINE:
    rows = []
    for ccy in ("local", "EUR"):
        for j, st in enumerate(REF.stocks):
            js, py = E["stats"][ccy][j], REF_STATS[ccy][j]
            c = dict(stock=st.id, setting=ccy)
            for k in HORIZONS:
                rows.append({**c, "item": f"return {k}", "dashboard": js["ret"][k], "reference": py["ret"][k], "cls": "return", "fmt": pct1s})
                for w in ("subsector", "coverage", "market"):
                    rows.append({**c, "item": f"relative {k} vs {w}", "dashboard": js["rel"][k][w], "reference": py["rel"][k][w], "cls": "return", "fmt": ppf})
            note = "daily returns contain a gap longer than the forward-fill (see section 8)" if py["daily_has_nan"] else None
            rows += [
                {**c, "item": "volatility 1Y (daily)", "dashboard": js["vol1y"], "reference": py["vol1y"], "cls": "return", "fmt": pct1},
                {**c, "item": "momentum 12-1", "dashboard": js["mom12_1"], "reference": py["mom12_1"], "cls": "return", "fmt": pct1s},
                {**c, "item": "max drawdown 1Y", "dashboard": js["maxDD1y"], "reference": py["maxDD1y"], "cls": "return", "fmt": pct1},
                {**c, "item": "1-day VaR 95%", "dashboard": js["var95"], "reference": py["var95"], "cls": "return", "fmt": pct1, "note": note},
                {**c, "item": "1-day expected shortfall 95%", "dashboard": js["es95"], "reference": py["es95"], "cls": "return", "fmt": pct1, "note": note},
            ]
            if ccy == "local":
                rows += [
                    {**c, "item": "last price", "dashboard": js["price"], "reference": py["price"], "cls": "exact", "fmt": num2},
                    {**c, "item": "price vs 50-day average", "dashboard": js["price"] / js["sma50"] - 1 if is_num(js["sma50"]) else None, "reference": py["price"] / py["sma50"] - 1, "cls": "return", "fmt": pct1s},
                    {**c, "item": "price vs 200-day average", "dashboard": js["price"] / js["sma200"] - 1 if is_num(js["sma200"]) else None, "reference": py["price"] / py["sma200"] - 1, "cls": "return", "fmt": pct1s},
                    {**c, "item": "RSI (14)", "dashboard": js["rsi14"], "reference": py["rsi14"], "cls": "level", "fmt": num0},
                ]
    STATS_CHECKS = batch(rows, "Returns & risk", {k: TOL[k] for k in ("return", "exact", "level")})
    display(summarise(STATS_CHECKS, by=("item",)).sort_values("FAIL", ascending=False).head(12))

### 5.4 Macro sensitivities and rolling exposures

In [ ]:
if ENGINE:
    rows, brows = [], []
    for key, per_stock in E["macroSens"].items():
        freq, years, ccy = key.split("|")
        for j, st in enumerate(REF.stocks):
            py_rows = {r["id"]: r for r in REF.macro_sens(st, freq, int(years), ccy)}
            js_rows = {r["id"]: r for r in per_stock[j]}
            brows.append({"item": "drivers listed", "stock": st.id, "setting": key, "passed": list(js_rows) == list(py_rows), "dashboard": ",".join(js_rows), "reference": ",".join(py_rows)})
            for fid, jr in js_rows.items():
                pr_ = py_rows.get(fid)
                if pr_ is None:
                    continue
                c = dict(stock=st.id, setting=f"{key}|{fid}")
                rows += [
                    {**c, "item": "total effect", "dashboard": jr["totalImpact"], "reference": pr_["totalImpact"], "cls": "coef", "fmt": pct2s},
                    {**c, "item": "total effect t", "dashboard": jr["totalT"], "reference": pr_["totalT"], "cls": "stat", "fmt": num1},
                    {**c, "item": "R2 alone", "dashboard": jr["r2"], "reference": pr_["r2"], "cls": "fit", "fmt": num2},
                    {**c, "item": "market held constant", "dashboard": jr["partialImpact"], "reference": pr_["partialImpact"], "cls": "coef", "fmt": pct2s},
                    {**c, "item": "market held constant t", "dashboard": jr["partialT"], "reference": pr_["partialT"], "cls": "stat", "fmt": num1},
                ]
    add_bool_checks(brows, "Macro sensitivities")
    batch(rows, "Macro sensitivities", {k: TOL[k] for k in ("coef", "stat", "fit")})

    rows, brows = [], []
    specs = {"model": ("full", DV["freq"], 1), "rates": ("mktrates", DV["freq"], 5 if DV["freq"] == "D" else 1), "macroW": ("macro", "W", 1)}
    for name, (model, freq, step) in specs.items():
        for j, st in enumerate(REF.stocks):
            js = E["rolling"][name][j]
            py = REF.rolling_fit(st, REF.resolve(MODEL_SPECS[model]), freq, DV["ccy"], step=step)
            brows.append({"item": "rolling fit available", "stock": st.id, "setting": name, "passed": bool(js["ok"]) == bool(py["ok"])})
            if not (js["ok"] and py["ok"]):
                continue
            same_dates = js["dates"] == py["dates"]
            brows.append({"item": "rolling window end dates", "stock": st.id, "setting": name, "passed": same_dates, "dashboard": len(js["dates"]), "reference": len(py["dates"])})
            if not same_dates:
                continue
            for f in js["factors"]:
                for what, scale_fmt in (("beta", exposure_fmt(f)), ("se", None)):
                    i, a, b = worst_point(js[what][f], py[what][f], 1e-7, 1e-12)
                    rows.append({"item": f"rolling {what} {f} (worst window)", "stock": st.id, "setting": name, "dashboard": a, "reference": b,
                                 "cls": "coef" if what == "beta" else "stat", "fmt": scale_fmt, "note": js["dates"][i] if js["dates"] else None})
    add_bool_checks(brows, "Rolling exposures")
    batch(rows, "Rolling exposures", {k: TOL[k] for k in ("coef", "stat")})
    display(summarise(checks_frame().query("section in ['Macro sensitivities', 'Rolling exposures']")))

### 5.5 Attribution (Carino-linked, exposures re-estimated monthly)

In [ ]:
ATTR_KEYS = ["1M", "3M", "6M", "YTD", "1Y", "3Y"]


def attr_period(ref, key):
    end = ref.asof
    if key == "YTD":
        return ref.on_or_before(f"{int(ref.iso[end][:4]) - 1}-12-31"), end
    return ref.window_start(end, {"1M": 1 / 12, "3M": 0.25, "6M": 0.5, "1Y": 1, "3Y": 3}[key]), end


REF_ATTR = {}
if ENGINE:
    rows, brows = [], []
    for mode, model in (("sector", "attrib"), ("model", "full")):
        ids = REF.resolve(MODEL_SPECS[model])
        for key in ATTR_KEYS:
            start, end = attr_period(REF, key)
            for j, st in enumerate(REF.stocks):
                js = E["attribution"][f"{mode}|{key}"][j]
                py = REF.attribution(st, ids, DV["freq"], DV["years"], DV["ccy"], start, end)
                REF_ATTR[(mode, key, st.id)] = py
                c = dict(stock=st.id, setting=f"{mode}|{key}")
                brows.append({**c, "item": "period start/end rows", "passed": (js["startIdx"], js["endIdx"]) == (start, end), "dashboard": (js["startIdx"], js["endIdx"]), "reference": (start, end)})
                brows.append({**c, "item": "available", "passed": bool(js["ok"]) == bool(py["ok"]), "dashboard": js.get("error") or js["ok"], "reference": py.get("error") or py["ok"]})
                if not (js["ok"] and py["ok"]):
                    continue
                brows.append({**c, "item": "days used / dates", "passed": (js["days"], js["start"], js["end"]) == (py["days"], py["start"], py["end"]),
                              "dashboard": (js["days"], js["start"], js["end"]), "reference": (py["days"], py["start"], py["end"])})
                brows.append({**c, "item": "missing factor days", "passed": js["missing"] == py["missing"], "dashboard": js["missing"], "reference": py["missing"]})
                sd = lambda a: a["specific"] / (a["lastFitSpecificVolAnn"] * math.sqrt(a["days"] / 252)) if a.get("lastFitSpecificVolAnn") else np.nan
                rows += [
                    {**c, "item": "total return", "dashboard": js["total"], "reference": py["total"], "cls": "return", "fmt": pct1s},
                    {**c, "item": "stock-specific", "dashboard": js["specific"], "reference": py["specific"], "cls": "coef", "fmt": pct1s},
                    {**c, "item": "stock-specific z", "dashboard": sd(js), "reference": sd(py), "cls": "coef", "fmt": num1},
                    {**c, "item": "explained by factors", "dashboard": js["total"] - js["specific"], "reference": py["total"] - py["specific"], "cls": "coef", "fmt": pct1s},
                    {**c, "item": "Carino additivity: sum of contributions + specific = total (dashboard's own figures)",
                     "dashboard": sum(js["contributions"].values()) + js["specific"], "reference": js["total"], "cls": "exact"},
                ]
                for f in js["factors"]:
                    fm = FACTORS[f]
                    move_fmt = (lambda v: js_bp(v, 0)) if fm.unit == "bp" else (lambda v: js_num(v, 1, True) + " pts") if fm.unit == "pts" else pct1s
                    rows += [
                        {**c, "item": f"contribution {f}", "dashboard": js["contributions"][f], "reference": py["contributions"][f], "cls": "coef", "fmt": pct2s},
                        {**c, "item": f"average exposure {f}", "dashboard": js["avgBeta"][f], "reference": py["avgBeta"][f], "cls": "coef", "fmt": exposure_fmt(f)},
                        {**c, "item": f"factor move {f}", "dashboard": js["moves"][f], "reference": py["moves"][f], "cls": "return", "fmt": move_fmt},
                    ]
                if js.get("path"):
                    for part in ("total", "factors", "specific"):
                        i, a, b = worst_point([p[part] for p in js["path"]], py["path"][part], 1e-7, 1e-12)
                        rows.append({**c, "item": f"cumulative path: {part} (worst day)", "dashboard": a, "reference": b, "cls": "coef", "fmt": pct1s, "note": py["path"]["date"][i]})
    add_bool_checks(brows, "Attribution")
    batch(rows, "Attribution", {k: TOL[k] for k in ("return", "coef", "exact")})
    display(summarise(checks_frame().query("section == 'Attribution'"), by=("item",)).sort_values("FAIL", ascending=False).head(12))

### 5.6 Scenario builder and historical stress episodes

In [ ]:
def scenario_presets(credit_id):
    risk_credit = -3 if credit_id == "CREDIT_ETF" else 150
    stag_credit = -2 if credit_id == "CREDIT_ETF" else 100
    return [("Rates +50bp", {"MKT": 0, "RATES": 50, "CREDIT": 0, "FXL": 0}), ("Rates −50bp", {"MKT": 0, "RATES": -50, "CREDIT": 0, "FXL": 0}),
            ("Equity sell-off", {"MKT": -10, "RATES": 0, "CREDIT": 0, "FXL": 0}), ("Risk-off", {"MKT": -10, "RATES": -25, "CREDIT": risk_credit, "FXL": 0}),
            ("Stagflation", {"MKT": -5, "RATES": 75, "CREDIT": stag_credit, "FXL": 0})]


def shocks_for(fit, scen):
    """Slider values (%, bp) to factor units for the factors in the stock's model."""
    s = {}
    if scen.get("MKT"):
        s["MKT"] = scen["MKT"] / 100
    if scen.get("RATES"):
        s["RATES"] = scen["RATES"]
    if scen.get("CREDIT"):
        if "CREDIT" in fit["factors"]:
            s["CREDIT"] = scen["CREDIT"]
        if "CREDIT_ETF" in fit["factors"]:
            s["CREDIT_ETF"] = scen["CREDIT"] / 100
    if scen.get("FXL") and "FXL" in fit["factors"]:
        s["FXL"] = scen["FXL"] / 100
    return s


MACRO_IDS = REF.resolve(MACRO_SPEC)
CREDIT_ID = "CREDIT" if "CREDIT" in MACRO_IDS else "CREDIT_ETF" if "CREDIT_ETF" in MACRO_IDS else None
if ENGINE:
    rows, brows = [], []
    mf = ref_fits("macro")
    brows.append({"item": "credit factor behind the scenario slider", "passed": E["scenario"]["creditId"] == CREDIT_ID, "dashboard": E["scenario"]["creditId"], "reference": CREDIT_ID})
    for label, scen in scenario_presets(CREDIT_ID):
        for cond in (True, False):
            res = E["scenario"]["results"][f"{label}|{'true' if cond else 'false'}"]
            for j, st in enumerate(REF.stocks):
                js = res[j]
                if js is None or not mf[j]["ok"]:
                    brows.append({"item": "scenario available", "stock": st.id, "setting": label, "passed": (js is None) == (not mf[j]["ok"])})
                    continue
                py = Ref.scenario(mf[j], shocks_for(mf[j], scen), cond)
                c = dict(stock=st.id, setting=f"{label}|{'correlated' if cond else 'shocked only'}")
                rows.append({**c, "item": "model-implied move", "dashboard": js["total"], "reference": py["total"], "cls": "coef", "fmt": pct1s})
                for f in mf[j]["factors"]:
                    rows.append({**c, "item": f"contribution {f}", "dashboard": js["impacts"][f], "reference": py["impacts"][f], "cls": "coef", "fmt": pct1s})
                    rows.append({**c, "item": f"factor move {f}", "dashboard": js["moves"][f], "reference": py["moves"][f], "cls": "coef"})
    for j, st in enumerate(REF.stocks):
        f = mf[j]
        rows.append({"item": "typical stock-specific noise per period", "stock": st.id, "dashboard": E["scenario"]["residSd"][j],
                     "reference": f["specificVolAnn"] / math.sqrt(f["ppy"]) if f["ok"] else None, "cls": "coef", "fmt": pct1})
    add_bool_checks(brows, "Scenarios")
    batch(rows, "Scenarios", {"coef": TOL["coef"]})

    rows, brows = [], []
    for j, st in enumerate(REF.stocks):
        peers = [p for p in REF.stocks if p.subsector == st.subsector and p is not st]
        for e_i, ep in enumerate(EPISODES):
            js = E["episodes"][j][e_i]
            py = REF.episode_impact(st, mf[j], ep, DV["ccy"])
            brows.append({"item": "episode shown", "stock": st.id, "setting": ep["id"], "passed": (js is None) == (py is None)})
            if js is None or py is None:
                continue
            a, b = py["a"], py["b"]
            vals = [float(REF.levels(p, DV["ccy"])[b] / REF.levels(p, DV["ccy"])[a] - 1) if a >= p.first else np.nan for p in peers]
            vals = [v for v in vals if np.isfinite(v)]
            c = dict(stock=st.id, setting=ep["id"])
            rows += [
                {**c, "item": "implied today", "dashboard": js["implied"], "reference": py["implied"], "cls": "coef", "fmt": pct1s},
                {**c, "item": "actual then", "dashboard": js["actual"], "reference": py["actual"], "cls": "return", "fmt": pct1s},
                {**c, "item": "peer median then", "dashboard": js["peerActual"], "reference": float(np.median(vals)) if vals else np.nan, "cls": "return", "fmt": pct1s},
            ]
            for f, v in js["moves"].items():
                rows.append({**c, "item": f"episode move {f}", "dashboard": v, "reference": py["moves"].get(f), "cls": "return",
                             "fmt": (lambda x: js_bp(x, 0)) if FACTORS[f].unit == "bp" else pct1s})
    add_bool_checks(brows, "Stress episodes")
    batch(rows, "Stress episodes", {k: TOL[k] for k in ("coef", "return")})
    display(summarise(checks_frame().query("section in ['Scenarios', 'Stress episodes']")))

### 5.7 Outlook: exposure forecasts, GARCH volatility and price ranges, balance-sheet stress, price signals

In [ ]:
REF_FC = {c: REF.exposure_forecast(MACRO_IDS, c) for c in ("local", "EUR")}
if ENGINE:
    rows, brows = [], []
    for ccy in ("local", "EUR"):
        js, py = E["forecast"][ccy], REF_FC[ccy]
        brows += [
            {"item": "factors forecast", "setting": ccy, "passed": js["factors"] == py["factors"], "dashboard": ",".join(js["factors"]), "reference": ",".join(py["factors"])},
            {"item": "back-test origins (first / last / count)", "setting": ccy, "passed": (js["firstOrigin"], js["lastOrigin"], js["origins"]) == (py["firstOrigin"], py["lastOrigin"], py["origins"]),
             "dashboard": (js["firstOrigin"], js["lastOrigin"], js["origins"]), "reference": (py["firstOrigin"], py["lastOrigin"], py["origins"])},
        ]
        for f in js["factors"]:
            if f not in py["weights"]:
                continue
            for k in ("long", "short", "peers", "rmse", "rmseLong", "rmseShort", "rmsePeers"):
                rows.append({"item": f"pooled weight/error {k}", "stock": f, "setting": ccy, "dashboard": js["weights"][f][k], "reference": py["weights"][f][k], "cls": "coef"})
            brows.append({"item": "pooled observations", "stock": f, "setting": ccy, "passed": js["weights"][f]["n"] == py["weights"][f]["n"], "dashboard": js["weights"][f]["n"], "reference": py["weights"][f]["n"]})
            rows.append({"item": "accuracy gain vs 2-year estimate (shown in the note)", "stock": f, "setting": ccy,
                         "dashboard": 1 - js["weights"][f]["rmse"] / js["weights"][f]["rmseLong"], "reference": 1 - py["weights"][f]["rmse"] / py["weights"][f]["rmseLong"], "cls": "coef", "fmt": pct0})
        for j, st in enumerate(REF.stocks):
            jsx, pyx = js["stocks"][j], py["stocks"][j]
            brows.append({"item": "exposures forecast", "stock": st.id, "setting": ccy, "passed": sorted(jsx) == sorted(pyx), "dashboard": ",".join(sorted(jsx)), "reference": ",".join(sorted(pyx))})
            for f in jsx:
                if f not in pyx:
                    continue
                c = dict(stock=st.id, setting=f"{ccy}|{f}")
                brows.append({**c, "item": "basis (2-year or 1-year)", "passed": jsx[f]["basis"] == pyx[f]["basis"], "dashboard": jsx[f]["basis"], "reference": pyx[f]["basis"]})
                for k in ("long", "short", "peers", "forecast", "lo", "hi", "ownRmse", "ownRmseLong"):
                    rows.append({**c, "item": f"forecast {k}", "dashboard": jsx[f][k], "reference": pyx[f][k], "cls": "coef", "fmt": exposure_fmt(f)})
    add_bool_checks(brows, "Exposure forecasts")
    batch(rows, "Exposure forecasts", {"coef": TOL["coef"]})

In [ ]:
REF_GARCH = []
for st in REF.stocks:
    r = []
    for t in range(max(st.last - 756, st.first + 1), st.last + 1):
        if st.obs[t]:
            v = math.log(st.tri[t] / st.tri[t - 1]) if st.tri[t] > 0 and st.tri[t - 1] > 0 else np.nan
            if np.isfinite(v):
                r.append(v)
    REF_GARCH.append(np.array(r))
if ENGINE:
    rows, orows, brows = [], [], []
    for j, st in enumerate(REF.stocks):
        js = E["garch"][j]
        r = REF_GARCH[j]
        brows.append({"item": "GARCH estimable", "stock": st.id, "passed": (js is not None) == (len(r) >= 250), "dashboard": js is not None, "reference": len(r) >= 250})
        if js is None or len(r) < 250:
            continue
        e = r - r.mean()
        lr = float(np.mean(e * e))
        at_js = garch_state(js["alpha"], js["beta"], e, lr)  # formula fidelity at the dashboard's own parameters
        price = js["price"]
        cns = FUND[st.id].get("consensus") or {}
        vol = lambda g, h: math.sqrt(garch_cum_var(g, h) / h * 252)
        c = dict(stock=st.id)
        rows += [
            {**c, "item": "sample size (returns)", "dashboard": js["n"], "reference": len(r), "cls": "exact"},
            {**c, "item": "log-likelihood at the dashboard's parameters", "dashboard": js["logLik"], "reference": at_js.logLik, "cls": "garch"},
            {**c, "item": "long-run variance", "dashboard": js["lrVar"], "reference": at_js.lrVar, "cls": "garch"},
            {**c, "item": "next-day variance", "dashboard": js["nextVar"], "reference": at_js.nextVar, "cls": "garch"},
            {**c, "item": "half-life (days)", "dashboard": js["halfLife"], "reference": at_js.halfLife, "cls": "garch", "fmt": num0},
            {**c, "item": "volatility next month", "dashboard": js["vol21"], "reference": vol(at_js, 21), "cls": "garch", "fmt": pct0},
            {**c, "item": "volatility next 3 months", "dashboard": js["vol63"], "reference": vol(at_js, 63), "cls": "garch", "fmt": pct0},
            {**c, "item": "volatility next year", "dashboard": js["vol252"], "reference": vol(at_js, 252), "cls": "garch", "fmt": pct0},
            {**c, "item": "long-run volatility", "dashboard": js["lrVol"], "reference": math.sqrt(at_js.lrVar * 252), "cls": "garch", "fmt": pct0},
        ]
        for h in ("21", "63", "252"):
            pr_ = price_range(price, at_js, int(h), 1)
            rows += [{**c, "item": f"price range {h}d low", "dashboard": js["range"][h]["lo"], "reference": pr_["lo"], "cls": "garch", "fmt": num2},
                     {**c, "item": f"price range {h}d high", "dashboard": js["range"][h]["hi"], "reference": pr_["hi"], "cls": "garch", "fmt": num2}]
        if is_num(cns.get("target_mean")):
            z = math.log(cns["target_mean"] / price) / math.sqrt(garch_cum_var(at_js, 252))
            rows += [{**c, "item": "consensus target in standard deviations", "dashboard": js["zTarget"], "reference": z, "cls": "garch", "fmt": lambda v: js_num(v, 1, True)},
                     {**c, "item": "probability of ending a year above target", "dashboard": js["probAbove"], "reference": 1 - sps.norm.cdf(z), "cls": "optimizer", "fmt": pct0,
                      "note": "the dashboard uses the Abramowitz-Stegun normal CDF (error < 1e-7)"}]
        grid = garch_grid_fit(r)  # the dashboard's own grid search, re-implemented
        rows += [{**c, "item": "grid-search alpha (re-implemented)", "dashboard": js["alpha"], "reference": grid.alpha, "cls": "garch"},
                 {**c, "item": "grid-search beta (re-implemented)", "dashboard": js["beta"], "reference": grid.beta, "cls": "garch"}]
        best = garch_fit(r)  # the continuous maximum-likelihood optimum
        orows.append({"stock": st.id, "alpha_dashboard": js["alpha"], "alpha_mle": best.alpha, "beta_dashboard": js["beta"], "beta_mle": best.beta,
                      "loglik_dashboard": js["logLik"], "loglik_mle": best.logLik, "loglik_gap": best.logLik - js["logLik"],
                      "vol21_dashboard": js["vol21"], "vol21_mle": vol(best, 21), "vol252_dashboard": js["vol252"], "vol252_mle": vol(best, 252),
                      "at_search_edge": js["alpha"] <= 0.0015 or js["alpha"] >= 0.31 or js["beta"] <= 0.48 or js["alpha"] + js["beta"] >= 0.998})
    add_bool_checks(brows, "GARCH & price ranges")
    batch(rows, "GARCH & price ranges", {k: TOL[k] for k in ("garch", "exact", "optimizer")})
    GARCH_OPT = pd.DataFrame(orows).set_index("stock")
    add_checks([{"item": "grid optimum vs continuous MLE: volatility next month", "stock": s, "dashboard": r.vol21_dashboard, "reference": r.vol21_mle,
                 "note": f"alpha {r.alpha_dashboard:.4f} vs {r.alpha_mle:.4f}, beta {r.beta_dashboard:.4f} vs {r.beta_mle:.4f}, log-lik gap {r.loglik_gap:.3f}"}
                for s, r in GARCH_OPT.iterrows()], "GARCH optimiser", tol_abs=0.0025, display=pct0)
    add_checks([{"item": "grid optimum vs continuous MLE: volatility next year", "stock": s, "dashboard": r.vol252_dashboard, "reference": r.vol252_mle}
                for s, r in GARCH_OPT.iterrows()], "GARCH optimiser", tol_abs=0.0025, display=pct0)
    display(GARCH_OPT.sort_values("loglik_gap", ascending=False).head(12))

In [ ]:
if ENGINE:
    rows, brows = [], []
    for j, st in enumerate(REF.stocks):
        js, py = E["balance"][j], balance_sheet_outlook(FUND[st.id])
        brows.append({"item": "balance-sheet outlook available", "stock": st.id, "passed": (js is None) == (py is None)})
        if js is None or py is None:
            continue
        c = dict(stock=st.id)
        for k, fmt in (("implied", pct0s), ("ltvNow", pct0), ("headroom50", pct0), ("headroom60", pct0), ("coverNow", num1), ("coverStressed", num1), ("assets", None)):
            rows.append({**c, "item": k, "dashboard": js[k], "reference": py[k], "fmt": fmt})
        for jr, prow in zip(js["rows"] + [js["impliedRow"]], py["rows"] + [py["impliedRow"]]):
            for k, fmt in (("nav", num2), ("navChange", pct0s), ("pnav", lambda v: js_mult(v)), ("ltv", pct0)):
                rows.append({**c, "setting": f"values {prow['change']:+.3f}", "item": f"row {k}", "dashboard": jr[k], "reference": prow[k], "fmt": fmt})
    add_bool_checks(brows, "Balance-sheet outlook")
    add_checks(rows, "Balance-sheet outlook", tol_rel=1e-10, tol_abs=1e-12)

    REF_SIG = REF.signal_backtest()
    rows, brows = [], []
    brows.append({"item": "months tested (first / last / count)", "passed": (E["signals"]["first"], E["signals"]["last"], E["signals"]["months"]) == (REF_SIG["first"], REF_SIG["last"], REF_SIG["months"]),
                  "dashboard": (E["signals"]["first"], E["signals"]["last"], E["signals"]["months"]), "reference": (REF_SIG["first"], REF_SIG["last"], REF_SIG["months"])})
    for sig in E["signals"]["signals"]:
        py = REF_SIG["signals"][sig["id"]]
        c = dict(stock=sig["id"])
        brows.append({**c, "item": "months with an IC", "passed": sig["months"] == py["months"], "dashboard": sig["months"], "reference": py["months"]})
        for k, fmt in (("ic", lambda v: js_num(v, 2, True)), ("t", num1), ("hit", pct0), ("spread", pct1s), ("icRecent", lambda v: js_num(v, 2, True))):
            rows.append({**c, "item": k, "dashboard": sig[k], "reference": py[k], "fmt": fmt})
        for j, st in enumerate(REF.stocks):
            rows.append({"item": f"{sig['id']} value today", "stock": st.id, "dashboard": sig["current"][j], "reference": py["current"][j]})
            rows.append({"item": f"{sig['id']} percentile today", "stock": st.id, "dashboard": sig["percentile"][j], "reference": py["percentile"][j],
                         "fmt": lambda v: js_ordinal(js_round(v * 100)) if is_num(v) else DASH})
    add_bool_checks(brows, "Price signals")
    add_checks(rows, "Price signals", tol_rel=1e-9, tol_abs=1e-12)
    display(summarise(checks_frame().query("section in ['Exposure forecasts', 'GARCH & price ranges', 'GARCH optimiser', 'Balance-sheet outlook', 'Price signals']")))

### 5.8 Peers, correlations, pair spreads, style characteristics, relative-value lines and chart series

In [ ]:
if ENGINE:
    capm_def = ref_fits("capm", hac=False)
    full_def = ref_fits("full")
    macro_def = ref_fits("macro")
    stats_def = REF_STATS[DV["ccy"]]
    REF_CHARS = REF.characteristics(stats_def, [f["betas"]["MKT"] if f["ok"] else np.nan for f in capm_def])
    rows = []
    for d in E["chars"]:
        py = REF_CHARS[d["id"]]
        for j, st in enumerate(REF.stocks):
            rows.append({"item": f"{d['id']} raw", "stock": st.id, "dashboard": d["raw"][j], "reference": py["raw"][j]})
            rows.append({"item": f"{d['id']} z-score", "stock": st.id, "dashboard": d["z"][j], "reference": py["z"][j], "fmt": lambda v: js_num(v, 2, True)})
    add_checks(rows, "Style characteristics", tol_rel=1e-9, tol_abs=1e-12)

    rows, prow, mrow = [], [], []
    for j, st in enumerate(REF.stocks):
        py = {r["id"]: r for r in REF.correlations(st, full_def, DV["freq"], DV["years"])}
        for r in E["corr"][j]:
            if r["id"] == st.id:
                continue
            rows.append({"item": "return correlation", "stock": st.id, "setting": r["id"], "dashboard": r["ret"], "reference": py[r["id"]]["ret"], "fmt": num2})
            rows.append({"item": "specific (residual) correlation", "stock": st.id, "setting": r["id"], "dashboard": r["resid"], "reference": py[r["id"]]["resid"], "fmt": num2})
        for k, other in enumerate(REF.stocks):
            if k == j:
                continue
            js, pyp = E["pairs"][j][k], REF.pair_spread(st, other)
            if (js is None) != (pyp is None):
                prow.append({"item": "pair spread available", "stock": st.id, "setting": other.id, "dashboard": 1.0 if js else None, "reference": 1.0 if pyp else None})
            elif js is not None:
                prow.append({"item": "pair spread z (1Y)", "stock": st.id, "setting": other.id, "dashboard": js["z"], "reference": pyp["z"], "fmt": lambda v: js_num(v, 1, True)})
                prow.append({"item": "pair relative 3M", "stock": st.id, "setting": other.id, "dashboard": js["change3m"], "reference": pyp["change3m"], "fmt": pct1s})
    add_checks(rows, "Correlations & pairs", tol_rel=1e-9, tol_abs=1e-12)
    add_checks(prow, "Correlations & pairs", tol_rel=1e-9, tol_abs=1e-12)
    wk = {st.i: REF.weekly_eur(st, "W", DV["years"]) for st in REF.stocks}
    for j, st in enumerate(REF.stocks):
        for k, other in enumerate(REF.stocks):
            if k <= j:
                continue
            common = wk[j].index.intersection(wk[k].index)
            pyv = pearson(wk[j][common], wk[k][common]) if len(common) > 20 else np.nan
            mrow.append({"item": "correlation matrix cell", "stock": st.id, "setting": other.id, "dashboard": E["corrMatrix"][j][k], "reference": pyv, "fmt": num2})
    add_checks(mrow, "Correlations & pairs", tol_rel=1e-9, tol_abs=1e-12)

    s_def = stats_def
    METRIC_REF = {
        "pb": [FUND[s].get("pb") for s in STOCK_IDS], "dy": [FUND[s].get("dividend_yield") for s in STOCK_IDS], "ltv": [FUND[s].get("ltv") for s in STOCK_IDS],
        "pe": [FUND[s].get("pe_forward") for s in STOCK_IDS], "upside": [(FUND[s].get("consensus") or {}).get("upside") for s in STOCK_IDS],
        "mcap": [FUND[s]["market_cap_eur"] / 1e9 if is_num(FUND[s].get("market_cap_eur")) else None for s in STOCK_IDS],
        "beta": [f["betas"]["MKT"] if f["ok"] else None for f in capm_def],
        "rate10": [f["betas"]["RATES"] * 1000 if f["ok"] and "RATES" in f["betas"] else None for f in macro_def],
        "specvol": [f["specificVolAnn"] if f["ok"] else None for f in macro_def], "vol": [s["vol1y"] for s in s_def],
        "r1y": [s["ret"]["y1"] for s in s_def], "r3m": [s["ret"]["m3"] for s in s_def], "mom": [s["mom12_1"] for s in s_def],
    }
    rows = []
    for m, vals in METRIC_REF.items():
        for j, st in enumerate(REF.stocks):
            rows.append({"item": f"metric {m}", "stock": st.id, "dashboard": E["relval"]["values"][m][j], "reference": vals[j]})
    for key, line in E["relval"]["lines"].items():
        kx, ky = key.split("|")
        x, y = np.array([_as_float(v) for v in METRIC_REF[kx]]), np.array([_as_float(v) for v in METRIC_REF[ky]])
        ok = np.isfinite(x) & np.isfinite(y)
        if ok.sum() >= 3 and line is not None:
            res = ols_np(y[ok], np.column_stack([np.ones(ok.sum()), x[ok]]))
            rows += [{"item": "fitted line slope", "setting": key, "dashboard": line["slope"], "reference": res.coef[1]},
                     {"item": "fitted line intercept", "setting": key, "dashboard": line["intercept"], "reference": res.coef[0]},
                     {"item": "fitted line R2", "setting": key, "dashboard": line["r2"], "reference": res.r2, "fmt": pct0}]
    add_checks(rows, "Relative value", tol_rel=1e-8, tol_abs=1e-12)

    rows = []
    end = REF.asof
    start = REF.window_start(end, DV["years"])
    for j, st in enumerate(REF.stocks):
        js = E["charts"][j]
        L = REF.levels(st, DV["ccy"])
        r = REF.day_returns(st, DV["ccy"])
        rv = []
        for t in range(max(start - 63, 1), end + 1):
            if st.obs[t] and np.isfinite(r[t]):
                rv.append((t, r[t]))
        vols = []
        for q in range(len(rv)):
            t = rv[q][0]
            window = [v for _, v in rv[max(0, q - 62):q + 1]]
            if t >= start and len(window) >= 40:
                vols.append((REF.iso[t], np.std(window, ddof=1) * math.sqrt(252) * 100))
        ok_dates = [d for d, _ in vols] == [p[0] for p in js["rollingVol"]]
        if ok_dates and vols:
            i, a, b = worst_point([p[1] for p in js["rollingVol"]], [v for _, v in vols], 1e-9)
            rows.append({"item": "63-day rolling volatility (worst day)", "stock": st.id, "dashboard": a, "reference": b, "note": vols[i][0], "fmt": num1})
        else:
            rows.append({"item": "63-day rolling volatility dates", "stock": st.id, "dashboard": len(js["rollingVol"]), "reference": len(vols)})
        seg = np.arange(max(start, st.first), end + 1)
        seg = seg[np.isfinite(L[seg])]
        dd = drawdown_series(L[seg]) * 100
        if [p[0] for p in js["drawdown"]] == [REF.iso[t] for t in seg]:
            i, a, b = worst_point([p[1] for p in js["drawdown"]], dd, 1e-9, 1e-12)
            rows.append({"item": "drawdown chart (worst day)", "stock": st.id, "dashboard": a, "reference": b, "fmt": num1})
            i, a, b = worst_point([p[1] for p in js["perf"]], L[seg] / L[seg[0]] * 100, 1e-10)
            rows.append({"item": "performance chart, rebased (worst day)", "stock": st.id, "dashboard": a, "reference": b, "fmt": num1})
    add_checks(rows, "Chart series", tol_rel=1e-9, tol_abs=1e-10)

    rows = []
    for t in E["formatTests"]:
        x = t["x"]
        expect = {"num0": js_num(x, 0), "num1": js_num(x, 1), "num2": js_num(x, 2), "num3": js_num(x, 3), "num1s": js_num(x, 1, True), "num2s": js_num(x, 2, True),
                  "pct0": js_pct(x, 0), "pct0s": js_pct(x, 0, True), "pct1": js_pct(x, 1), "pct1s": js_pct(x, 1, True), "pct2s": js_pct(x, 2, True), "pp": js_pp(x),
                  "bp": js_bp(x, 0), "mult": js_mult(x), "mult1": js_mult(x, 1), "compact": js_compact(x), "moneyGBP": js_money(x, "GBP"), "moneySEK": js_money(x, "SEK"),
                  "mcEUR": js_money_compact(x, "EUR"), "mcSEK": js_money_compact(x, "SEK"), "sig": js_sig(x)}
        for k, v in expect.items():
            rows.append({"item": f"format {k}", "stock": repr(x), "passed": t[k] == v, "dashboard": t[k], "reference": v})
    rows += [{"item": "ordinal", "stock": str(i), "passed": s == js_ordinal(i), "dashboard": s, "reference": js_ordinal(i)} for i, s in enumerate(E["ordinalTests"])]
    for (d, dn, my), iso in zip(E["dateTests"], ["2026-09-25", "2026-01-01", "2025-12-31"]):
        rows.append({"item": "date format", "stock": iso, "passed": (d, dn, my) == (js_date(iso), js_date(iso, False), js_month_year(iso)), "dashboard": (d, dn, my)})
    FMT_CHECKS = add_bool_checks(rows, "Number formatting (ui.js port)")
    print(f"Formatter port: {int((FMT_CHECKS.status == 'PASS').sum())}/{len(FMT_CHECKS)} strings identical to the dashboard's own formatting.")

### 5.9 Reconciliation summary

In [ ]:
if ENGINE:
    CK = checks_frame()
    ENG = CK[CK.layer == "2. Engine"]
    SUM = summarise(ENG)
    display(style_status(SUM.reset_index()))
    fig, axes = plt.subplots(1, 2, figsize=(13, 0.34 * len(SUM) + 1.8), gridspec_kw={"width_ratios": [1.2, 1]})
    ax = axes[0]
    y = np.arange(len(SUM))
    left = np.zeros(len(SUM))
    for s in ("PASS", "WARN", "FAIL"):
        ax.barh(y, SUM[s], left=left, color=STATUS_COLOR[s], height=0.62, label=s, edgecolor=C.surface, linewidth=2)
        left += SUM[s].to_numpy()
    for yi, (tot, w, f_) in enumerate(zip(SUM["total"], SUM["WARN"], SUM["FAIL"])):
        tag = f"  {tot:,}" + (f"  ({w} warn" if w else "") + (f", {f_} fail)" if f_ and w else f"  ({f_} fail)" if f_ else ")" if w else "")
        ax.text(tot, yi, tag, va="center", fontsize=8.5, color=C.ink2)
    ax.set_yticks(y)
    ax.set_yticklabels(SUM.index)
    ax.invert_yaxis()
    ax.set_xscale("symlog", linthresh=10)
    ax.set_xlabel("comparisons (log scale)")
    ax.set_title("Engine vs reference: comparisons by section")
    ax.legend(loc="lower right", ncols=3)
    ax.grid(axis="y", visible=False)
    ax = axes[1]
    num = ENG.dropna(subset=["dashboard", "reference"])
    for yi, sec in enumerate(SUM.index):
        if sec == "GARCH optimiser":  # two different estimators, judged in absolute vol points (section 5.6), not an implementation check
            ax.text(-16.8, yi, "different estimator: see 5.6", va="center", fontsize=8, color=C.muted)
            continue
        v = num.loc[num.section == sec, "rel_diff"].to_numpy()
        v = np.log10(np.clip(v, 1e-17, None))
        jitter = (np.random.default_rng(yi).random(len(v)) - 0.5) * 0.5
        ax.scatter(v, yi + jitter, s=6, color=C.s1, alpha=0.25, linewidth=0)
    ax.axvline(-6, color=C.axis, lw=1)
    ax.text(-6, -0.7, " loosest tolerance (1e-6)", fontsize=8, color=C.muted)
    ax.set_yticks(y)
    ax.set_yticklabels([])
    ax.set_ylim(len(SUM) - 0.5, -1)
    ax.set_xlabel("relative difference, log10 (values at -17 agree to the last bit)")
    ax.set_title("Size of every difference")
    ax.grid(axis="y", visible=False)
    fig.tight_layout()
    show(fig, "Reconciliation by section")

In [ ]:
if ENGINE:
    pick = [("Regressions", "beta MKT", "W|3Y|local|full", "Market beta (default view)"),
            ("Regressions", "beta RATES", "W|3Y|local|macro", "Rate beta, macro model"),
            ("Regressions", "t RATES", "W|3Y|local|macro", "Rate t-statistic (Newey-West)"),
            ("Regressions", "R2", "W|3Y|local|full", "R² (default model)"),
            ("Returns & risk", "volatility 1Y (daily)", "local", "Volatility 1Y"),
            ("Returns & risk", "return y1", "local", "1-year total return"),
            ("GARCH & price ranges", "volatility next month", None, "GARCH vol, next month"),
            ("Attribution", "stock-specific", "sector|3M", "3M stock-specific return")]
    fig, axes = plt.subplots(2, 4, figsize=(14, 6.6))
    for ax, (sec, item, setting, title) in zip(axes.ravel(), pick):
        q = ENG[(ENG.section == sec) & (ENG["item"] == item)]
        if setting is not None:
            q = q[q.setting == setting]
        q = q.dropna(subset=["dashboard", "reference"])
        if q.empty:
            ax.set_visible(False)
            continue
        lo, hi = np.nanmin(q[["dashboard", "reference"]].to_numpy()), np.nanmax(q[["dashboard", "reference"]].to_numpy())
        pad = (hi - lo) * 0.06 or 1e-6
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color=C.axis, lw=1)
        bad = q.status != "PASS"
        ax.scatter(q.reference[~bad], q.dashboard[~bad], s=34, color=C.s1, edgecolor=C.surface, linewidth=1.2, zorder=3)
        ax.scatter(q.reference[bad], q.dashboard[bad], s=40, color=C.critical, edgecolor=C.surface, linewidth=1.2, zorder=4)
        ax.set_title(f"{title}\n{int((~bad).sum())}/{len(q)} agree, max rel. diff {q.rel_diff.max():.0e}", fontsize=9.5)
        ax.set_xlabel("independent (Python)", fontsize=8.5)
        ax.set_ylabel("dashboard (JS)", fontsize=8.5)
    fig.suptitle("Dashboard vs independent value, one dot per stock (on the line = identical)", x=0.01, ha="left", fontsize=11.5, fontweight="bold")
    fig.tight_layout()
    show(fig, "45-degree reconciliation plots")

In [ ]:
if ENGINE:
    heat = ENG.groupby(["stock", "section"])["rel_diff"].max().unstack()
    heat = heat.reindex([s for s in order if s in heat.index])
    fails = ENG[ENG.status == "FAIL"].groupby(["stock", "section"]).size().unstack().reindex_like(heat)
    vals = np.log10(np.clip(heat.to_numpy(dtype=float), 1e-17, 1))
    fig, ax = plt.subplots(figsize=(1.0 * heat.shape[1] + 3, 0.17 * heat.shape[0] + 2.2))
    im = ax.imshow(vals, aspect="auto", cmap=SEQ_BLUE, vmin=-17, vmax=0, interpolation="nearest")
    for (r_, c_), v in np.ndenumerate(fails.to_numpy(dtype=float)):
        if np.isfinite(v) and v > 0:
            ax.text(c_, r_, f"{int(v)} FAIL", ha="center", va="center", fontsize=6.5, color="white", fontweight="bold")
    ax.set_xticks(range(heat.shape[1]))
    ax.set_xticklabels(heat.columns, rotation=35, ha="right", fontsize=8)
    ax.set_yticks(range(heat.shape[0]))
    ax.set_yticklabels(heat.index, fontsize=7)
    ax.grid(False)
    cb = fig.colorbar(im, ax=ax, fraction=0.02, pad=0.01)
    cb.set_label("largest relative difference (log10)", color=C.ink2)
    cb.outline.set_visible(False)
    ax.set_title("Where differences sit: largest relative difference by stock and section (darker = larger)")
    show(fig, "Differences by stock and section")
    shown = ENG[ENG.status != "PASS"].sort_values(["status", "rel_diff"], ascending=[True, False])
    display(Markdown(f"**{len(shown):,} engine comparisons outside tolerance** ({int((shown.status == 'FAIL').sum()):,} FAIL, {int((shown.status == 'WARN').sum()):,} WARN). The first 40:"))
    display(shown.head(40)[["section", "item", "stock", "setting", "dashboard", "reference", "rel_diff", "shown_dashboard", "shown_reference", "status", "note"]])

## 6. Fundamentals: arithmetic, units and plausibility
The valuation, balance-sheet and consensus figures are built once, in the Python pipeline (`factor_dashboard/fundamentals.py`), from Yahoo Finance statements. The data file stores both the inputs and the ratios, so every ratio can be recomputed from its stored inputs. Plausibility bands then catch the unit errors Yahoo is known for (pence vs pounds, statements in a different currency from the quote). A value outside its band is a WARN to look at, not proof of an error.

In [ ]:
fx_latest = META.get("fx_latest", {})
rows, brows, frows = [], [], []
for sid in STOCK_IDS:
    f = FUND[sid]
    ccy = STOCK_META.loc[sid, "currency"]
    price, sh = f.get("price"), f.get("shares")
    c = dict(stock=sid)
    if is_num(price) and is_num(sh):
        rows.append({**c, "item": "market cap = shares x price", "dashboard": f.get("market_cap"), "reference": sh * price, "fmt": js_compact})
    if is_num(f.get("market_cap")) and ccy in fx_latest:
        rows.append({**c, "item": "market cap in EUR = local / FX", "dashboard": f.get("market_cap_eur"), "reference": f["market_cap"] / fx_latest[ccy], "fmt": lambda v: js_money_compact(v, "EUR")})
    if is_num(price) and is_num(f.get("book_value_per_share")) and f["book_value_per_share"] > 0:
        rows.append({**c, "item": "P/B = price / book value per share", "dashboard": f.get("pb"), "reference": price / f["book_value_per_share"], "fmt": js_mult})
    if all(is_num(f.get(k)) for k in ("total_debt", "cash")):
        rows.append({**c, "item": "net debt = total debt - cash", "dashboard": f.get("net_debt"), "reference": f["total_debt"] - f["cash"], "fmt": js_compact})
    if all(is_num(f.get(k)) for k in ("net_debt", "total_assets", "cash")) and f["total_assets"] - f["cash"] > 0:
        rows.append({**c, "item": "LTV proxy = net debt / (assets - cash)", "dashboard": f.get("ltv"), "reference": f["net_debt"] / (f["total_assets"] - f["cash"]), "fmt": pct1})
    if is_num(f.get("dps_ttm")) and is_num(price) and price > 0:
        rows.append({**c, "item": "dividend yield = trailing DPS / price", "dashboard": f.get("dividend_yield"), "reference": f["dps_ttm"] / price, "fmt": pct1})
    if is_num(f.get("ebitda")) and is_num(f.get("revenue")) and f["revenue"] > 0:
        rows.append({**c, "item": "EBITDA margin", "dashboard": f.get("ebitda_margin"), "reference": f["ebitda"] / f["revenue"], "fmt": pct1})
    if is_num(f.get("nd_ebitda")):
        rows.append({**c, "item": "net debt / EBITDA", "dashboard": f.get("nd_ebitda"), "reference": f["net_debt"] / f["ebitda"], "fmt": lambda v: js_mult(v, 1)})
    cns = f.get("consensus") or {}
    if is_num(cns.get("target_mean")) and is_num(price):
        rows.append({**c, "item": "consensus upside = target / price - 1", "dashboard": cns.get("upside"), "reference": cns["target_mean"] / price - 1, "fmt": pct0s})
    if is_num(cns.get("target_low")) and is_num(cns.get("target_high")) and is_num(cns.get("target_mean")):
        brows.append({**c, "item": "target low <= mean <= high", "passed": cns["target_low"] <= cns["target_mean"] <= cns["target_high"],
                      "dashboard": f"{cns['target_low']:.4g} / {cns['target_mean']:.4g} / {cns['target_high']:.4g}"})
    brows.append({**c, "item": "price date is the as-of date", "passed": f.get("price_date") == AS_OF, "dashboard": f.get("price_date"), "reference": AS_OF})
    if f.get("stale"):
        brows.append({**c, "item": "fundamentals refreshed in this build", "passed": False, "dashboard": "stale"})
    # plausibility bands (WARN: look at these by eye)
    bands = {"pb": (0.2, 3.0, js_mult), "dividend_yield": (0.0, 0.12, pct1), "ltv": (0.0, 0.65, pct1), "pe_forward": (5.0, 40.0, lambda v: js_mult(v, 1)),
             "ev_ebitda": (5.0, 50.0, lambda v: js_mult(v, 1)), "nd_ebitda": (0.0, 20.0, lambda v: js_mult(v, 1))}
    for k, (lo, hi, fm) in bands.items():
        v = f.get(k)
        if is_num(v) and not (lo <= v <= hi):
            frows.append({"stock": sid, "field": k, "value": fm(v), "band": f"{fm(lo)} to {fm(hi)}"})
    up = cns.get("upside")
    if is_num(up) and not (-0.4 <= up <= 0.8):
        frows.append({"stock": sid, "field": "consensus upside", "value": pct0s(up), "band": "-40% to +80%"})
    if cns.get("analysts") is not None and cns["analysts"] < 3:
        frows.append({"stock": sid, "field": "analysts", "value": cns["analysts"], "band": ">= 3 for a meaningful consensus"})
    bsd = f.get("balance_sheet_date")
    if bsd and (pd.Timestamp(AS_OF) - pd.Timestamp(bsd)).days > 270:
        frows.append({"stock": sid, "field": "balance sheet date", "value": bsd, "band": "within 9 months of the as-of date"})
    if f.get("financial_currency") and f.get("financial_currency") != f.get("quote_currency"):
        frows.append({"stock": sid, "field": "reporting currency differs from quote", "value": f"{f['financial_currency']} vs {f['quote_currency']}",
                      "band": "check P/B, P/E and dividends by hand (converted with today's FX)"})
    for note in f.get("notes") or []:
        frows.append({"stock": sid, "field": "pipeline note", "value": note, "band": ""})
FUND_CHECKS = add_checks(rows, "Fundamentals arithmetic", tol_rel=2e-5, layer="1. Data")
add_bool_checks(brows, "Fundamentals arithmetic", layer="1. Data")
FUND_FLAGS = pd.DataFrame(frows, columns=["stock", "field", "value", "band"])
flag_rows = [{"item": f"plausibility: {r.field}", "stock": r.stock, "passed": False, "dashboard": r.value, "reference": r.band}
             for _, r in FUND_FLAGS.iterrows() if r.field != "pipeline note"]  # pipeline notes are information, not findings
if flag_rows:
    add_bool_checks(flag_rows, "Fundamentals plausibility", layer="1. Data")
    CHECK_FRAMES[-1].loc[:, "status"] = "WARN"  # outside a band means 'look at this', not 'wrong'
CK = checks_frame()
bad_fund = CK[(CK.section.str.startswith("Fundamentals")) & (CK.status != "PASS")]
print(f"Fundamentals: {len(FUND_CHECKS)} recomputed ratios, {int((FUND_CHECKS.status != 'PASS').sum())} outside tolerance; {len(FUND_FLAGS)} values outside plausibility bands.")
display(FUND_CHECKS[FUND_CHECKS.status != "PASS"][["item", "stock", "dashboard", "reference", "rel_diff", "shown_dashboard", "shown_reference", "status"]])
display(Markdown("**Values to look at by eye** (outside the plausibility band, a reporting-currency mismatch, or a pipeline correction):"))
display(FUND_FLAGS)
FUND_TABLE = pd.DataFrame({sid: {k: FUND[sid].get(k) for k in ("price", "market_cap_eur", "pb", "dividend_yield", "ltv", "pe_forward", "nd_ebitda", "interest_cover",
                                                                  "ev_ebitda", "balance_sheet_date", "financial_currency", "quote_currency")} for sid in STOCK_IDS}).T
FUND_TABLE["consensus_upside"] = [(FUND[s].get("consensus") or {}).get("upside") for s in STOCK_IDS]
FUND_TABLE["analysts"] = [(FUND[s].get("consensus") or {}).get("analysts") for s in STOCK_IDS]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.4))
for ax, (k, lab, fm, band) in zip(axes, [("pb", "P/B (IFRS book)", 1, (0.2, 3.0)), ("dividend_yield", "Dividend yield (%)", 100, (0, 12)), ("ltv", "LTV proxy (%)", 100, (0, 65))]):
    v = pd.Series({s: FUND[s].get(k) for s in STOCK_IDS}, dtype=float) * fm
    v = v.dropna().sort_values()
    outside = (v < band[0]) | (v > band[1])
    ax.scatter(v[~outside], np.arange(len(v))[~outside], s=26, color=C.s1, edgecolor=C.surface, linewidth=1)
    ax.scatter(v[outside], np.arange(len(v))[outside], s=32, color=C.s2, edgecolor=C.surface, linewidth=1)
    for i, (sid, val) in enumerate(v.items()):
        if outside[sid] or i in (0, len(v) - 1):
            ax.annotate(sid, (val, i), xytext=(4, -3), textcoords="offset points", fontsize=7.5, color=C.ink2)
    ax.axvspan(band[0], band[1], color=C.neutral, zorder=0)
    ax.set_yticks([])
    ax.set_title(f"{lab}: {int(outside.sum())} outside the band")
axes[0].set_ylabel("stocks, sorted")
fig.suptitle("Fundamentals against plausibility bands (shaded); orange = outside, label = stock", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
show(fig, "Fundamentals plausibility")

## 6b. What the input-data errors change on screen
The data errors found in section 2 are corrected in a copy of the data file: total-return indices of the stocks whose dividends were applied 100× too small are rebuilt with the dividends at full size, and probable bad prints are removed (the previous close carries over, as on a holiday). The reference implementation then recomputes the default view, and every number whose on-screen text changes is listed. This turns "the data has an error" into "these figures on screen are wrong, and by this much".

In [ ]:
import copy


def corrected_data(data, div_fix=(), bad_prints=()):
    d = copy.deepcopy(data)
    by_id = {s["id"]: s for s in d["stocks"]}
    for sid in div_fix:
        s = by_id[sid]
        tri, px = arr(s["tri"]), arr(s["px"])
        idx = np.flatnonzero(np.isfinite(tri) & np.isfinite(px))
        ratio = tri[idx] / px[idx]
        step = ratio[1:] / ratio[:-1] - 1
        micro = (step > 1e-5) & (step < 1e-3)
        y = np.where(micro, 100 * step / (1 + step), 0.0)            # the dividend as a share of the previous close
        fixed_step = np.where(micro, 1 / (1 - y) - 1, 0.0)          # Yahoo's multiplicative adjustment at full size
        new_ratio = np.r_[1.0, np.cumprod(1 + fixed_step)]
        new_ratio /= new_ratio[-1]
        tri2 = tri.copy()
        tri2[idx] = px[idx] * new_ratio
        s["tri"] = [None if not np.isfinite(v) else float(f"{v:.7g}") for v in tri2]
    for sid, dt in bad_prints:
        k = d["dates"].index(dt)
        by_id[sid]["tri"][k] = None
        by_id[sid]["px"][k] = None
    return d


ERR_ROWS = []
if DIV_100X or BAD_PRINTS:
    t0 = time.time()
    FIXED = corrected_data(DATA, DIV_100X, BAD_PRINTS)
    REF_FIXED = Ref(FIXED)
    stats_now, stats_fix = REF_STATS[DV["ccy"]], [REF_FIXED.stock_stats(st, DV["ccy"]) for st in REF_FIXED.stocks]
    capm_now, capm_fix = ref_fits("capm", hac=False), [REF_FIXED.fit(st, ["MKT"], DV["freq"], DV["years"], DV["ccy"], hac=False, fast=True) for st in REF_FIXED.stocks]
    macro_now = ref_fits("macro")
    macro_fix = [REF_FIXED.fit(st, MACRO_IDS, DV["freq"], DV["years"], DV["ccy"], hac=False, fast=True) for st in REF_FIXED.stocks]
    for j, st in enumerate(REF.stocks):
        a, b = stats_now[j], stats_fix[j]
        c = dict(stock=st.id)
        for lab, k in (("1M", "m1"), ("3M", "m3"), ("6M", "m6"), ("YTD", "ytd"), ("1Y", "y1"), ("3Y", "y3"), ("5Y", "y5")):
            ERR_ROWS.append({**c, "item": f"{lab} total return", "dashboard": a["ret"][k], "reference": b["ret"][k], "fmt": pct1s})
            ERR_ROWS.append({**c, "item": f"{lab} vs sub-sector", "dashboard": a["rel"][k]["subsector"], "reference": b["rel"][k]["subsector"], "fmt": ppf})
        ERR_ROWS += [{**c, "item": "momentum 12-1", "dashboard": a["mom12_1"], "reference": b["mom12_1"], "fmt": pct1s},
                     {**c, "item": "volatility 1Y", "dashboard": a["vol1y"], "reference": b["vol1y"], "fmt": pct0},
                     {**c, "item": "1-day VaR 95%", "dashboard": a["var95"], "reference": b["var95"], "fmt": pct1},
                     {**c, "item": "max drawdown 1Y", "dashboard": a["maxDD1y"], "reference": b["maxDD1y"], "fmt": pct1}]
        if capm_now[j]["ok"] and capm_fix[j]["ok"]:
            ERR_ROWS.append({**c, "item": "beta vs STOXX 600", "dashboard": capm_now[j]["betas"]["MKT"], "reference": capm_fix[j]["betas"]["MKT"], "fmt": num2})
        if macro_now[j]["ok"] and macro_fix[j]["ok"] and "RATES" in macro_now[j]["betas"]:
            ERR_ROWS.append({**c, "item": "rate sensitivity (header)", "dashboard": macro_now[j]["betas"]["RATES"] * 10, "reference": macro_fix[j]["betas"]["RATES"] * 10, "fmt": pct2s})
    for sid in sorted({s for s, _ in BAD_PRINTS}):  # the GARCH forecasts are the most exposed to a bad print
        j = STOCK_IDS.index(sid)
        st, st2 = REF.stocks[j], REF_FIXED.stocks[j]
        rets = lambda s_: np.array([math.log(s_.tri[t] / s_.tri[t - 1]) for t in range(max(s_.last - 756, s_.first + 1), s_.last + 1) if s_.obs[t]])
        g0, g1 = garch_grid_fit(rets(st)), garch_grid_fit(rets(st2))
        if g0 and g1:
            vol = lambda g, h: math.sqrt(garch_cum_var(g, h) / h * 252)
            price = float(st.px[st.last])
            ERR_ROWS += [{"stock": sid, "item": "volatility next month (GARCH)", "dashboard": vol(g0, 21), "reference": vol(g1, 21), "fmt": pct0},
                         {"stock": sid, "item": "volatility next year (GARCH)", "dashboard": vol(g0, 252), "reference": vol(g1, 252), "fmt": pct0},
                         {"stock": sid, "item": "12-month price range, low", "dashboard": price_range(price, g0, 252, 1)["lo"], "reference": price_range(price, g1, 252, 1)["lo"], "fmt": num1},
                         {"stock": sid, "item": "12-month price range, high", "dashboard": price_range(price, g0, 252, 1)["hi"], "reference": price_range(price, g1, 252, 1)["hi"], "fmt": num1}]
        for freq_, yrs in (("D", 3), ("D", 5)):
            f0 = REF.fit(st, ["MKT"], freq_, yrs, "local", hac=False, fast=True)
            f1 = REF_FIXED.fit(st2, ["MKT"], freq_, yrs, "local", hac=False, fast=True)
            if f0["ok"] and f1["ok"]:
                ERR_ROWS.append({"stock": sid, "setting": f"{freq_}|{yrs}Y", "item": "beta vs STOXX 600 (daily view)", "dashboard": f0["betas"]["MKT"], "reference": f1["betas"]["MKT"], "fmt": num2})
    IMPACT = add_checks(ERR_ROWS, "Impact of data errors", tol_abs=1e-12, tol_rel=1e-9, layer="1. Data")
    visible = IMPACT[IMPACT.status == "FAIL"]
    print(f"Correcting {len(DIV_100X)} dividend series and {len(BAD_PRINTS)} bad print(s) changes {len(visible)} numbers on screen "
          f"({visible.stock.nunique()} stocks) in the default view; recomputed in {time.time() - t0:.0f}s.")
    display(visible.sort_values("abs_diff", ascending=False)[["stock", "item", "setting", "shown_dashboard", "shown_reference"]]
            .rename(columns={"shown_dashboard": "shown today", "shown_reference": "with the data corrected"}).head(40))
    if DIV_100X:
        d1 = pd.DataFrame([{"stock": s, "shown": stats_now[STOCK_IDS.index(s)]["ret"]["y1"], "corrected": stats_fix[STOCK_IDS.index(s)]["ret"]["y1"]} for s in DIV_100X]).set_index("stock").sort_values("shown")
        fig, ax = plt.subplots(figsize=(8, 0.42 * len(d1) + 1.3))
        y_ = np.arange(len(d1))
        ax.hlines(y_, d1.shown * 100, d1.corrected * 100, color=C.axis, lw=1.4)
        ax.scatter(d1.shown * 100, y_, s=44, color=C.s2, zorder=3, label="1Y total return shown today")
        ax.scatter(d1.corrected * 100, y_, s=44, color=C.s1, zorder=3, label="with dividends at full size")
        for yi, (sid, r_) in enumerate(d1.iterrows()):
            ax.text(max(r_.shown, r_.corrected) * 100, yi, f"  +{(r_.corrected - r_.shown) * 100:.1f}pp", va="center", fontsize=8.5, color=C.ink2)
        ax.set_yticks(y_)
        ax.set_yticklabels([f"{NAME[s]} ({s})" for s in d1.index], fontsize=8.5)
        ax.axvline(0, color=C.axis, lw=1)
        ax.set_xlabel("1-year total return, %")
        ax.set_title("London stocks whose dividends are missing: 1-year total return")
        ax.legend(loc="lower right", fontsize=8)
        ax.grid(axis="y", visible=False)
        show(fig, "Missing dividends: 1Y total return")
else:
    print("No data errors to correct.")

## 7. Presentation: does the same label mean the same number on every tab?
A number can be computed correctly and still mislead if the same label shows different numbers in different places. These checks use the values the dashboard itself computes (from the engine run) for the default view.

In [ ]:
def dv_fits(model, hac=True):
    """Default-view fits: the dashboard's own when the engine ran, else the reference."""
    if ENGINE:
        return E["fits"][f"{DV['freq']}|{DV['years']}|{DV['ccy']}|{model}"]
    return ref_fits(model, hac=hac)


def dv_stats(ccy=None):
    ccy = ccy or DV["ccy"]
    return E["stats"][ccy] if ENGINE else REF_STATS[ccy]


def dv_macro_sens():
    if ENGINE:
        return E["macroSens"][DV_KEY]
    return [REF.macro_sens(st, DV["freq"], DV["years"], DV["ccy"]) for st in REF.stocks]


def dv_attr(mode, key, j):
    if ENGINE:
        return E["attribution"][f"{mode}|{key}"][j]
    return REF_ATTR.get((mode, key, STOCK_IDS[j])) or REF.attribution(REF.stocks[j], REF.resolve(MODEL_SPECS["attrib" if mode == "sector" else "full"]), DV["freq"], DV["years"], DV["ccy"], *attr_period(REF, key))


macro_f, full_f, sens = dv_fits("macro"), dv_fits("full"), dv_macro_sens()
rate_rows = []
for j, sid in enumerate(STOCK_IDS):
    head = macro_f[j]["betas"].get("RATES") if macro_f[j]["ok"] else None
    tab = next((r for r in sens[j] if r["id"] == "RATES"), None)  # partialImpact = beta x the +10bp shock
    expo = full_f[j]["betas"].get("RATES") if full_f[j]["ok"] else None
    vals = {"header, peer & coverage tables (macro model)": head * 10 if is_num(head) else None,
            "Macro tab, 'market held constant'": tab["partialImpact"] if tab and is_num(tab.get("partialImpact")) else None,
            "Exposures tab (selected model)": expo * 10 if is_num(expo) else None}
    shown = {k: js_pct(v, 2, True) for k, v in vals.items()}
    same_label = [shown[k] for k in list(vals)[:2] if shown[k] != DASH]  # both of these say 'market held constant'
    v2 = [v for v in list(vals.values())[:2] if is_num(v)]
    rate_rows.append({"stock": sid, **{f"value: {k}": v for k, v in vals.items()}, **{f"shown: {k}": s for k, s in shown.items()},
                      "same label, different text": len(set(same_label)) > 1,
                      "sign_disagrees": len({np.sign(v) for v in vals.values() if is_num(v) and abs(v) > 5e-5}) > 1,
                      "gap_pp (header vs Macro tab)": abs(v2[0] - v2[1]) * 100 if len(v2) == 2 else np.nan,
                      "spread_pp": (max(v for v in vals.values() if is_num(v)) - min(v for v in vals.values() if is_num(v))) * 100 if any(is_num(v) for v in vals.values()) else np.nan})
RATE_LABELS = pd.DataFrame(rate_rows).set_index("stock")
n_diff = int(RATE_LABELS["same label, different text"].sum())
n_sign = int(RATE_LABELS.sign_disagrees.sum())
add_bool_checks([{"item": "rate sensitivity identical wherever it is labelled 'market held constant' (header tile vs Macro tab)", "stock": s,
                  "passed": not r["same label, different text"], "dashboard": " | ".join(str(r[c]) for c in RATE_LABELS.columns if c.startswith("shown"))}
                 for s, r in RATE_LABELS.iterrows()], "Label consistency", layer="3. Presentation")
if n_diff:
    gap = RATE_LABELS["gap_pp (header vs Macro tab)"]
    finding("HIGH", "Presentation", "Two different 'rate sensitivity, market held constant' numbers for the same stock",
            f"For {n_diff} of {len(RATE_LABELS)} stocks the header tile and the Macro tab's 'market held constant' column show different rate sensitivities "
            f"under the same description (median gap {gap.median():.2f}pp, largest {gap.max():.2f}pp per +10bp, {gap.idxmax()}). The header uses the macro model, "
            "which also holds credit and FX constant; the Macro tab holds only the market constant. The Exposures tab adds style factors and shows a third number"
            + (f"; for {n_sign} stocks the three disagree on the sign." if n_sign else ".")
            + " Suggested fix: label the header 'holding market, credit and FX constant', or show the Macro tab's figure in the header.",
            stocks=list(RATE_LABELS.index[RATE_LABELS["same label, different text"]]))
display(RATE_LABELS[[c for c in RATE_LABELS.columns if c.startswith("shown")] + ["spread_pp", "sign_disagrees"]].sort_values("spread_pp", ascending=False).head(15))

In [ ]:
d = RATE_LABELS.sort_values(RATE_LABELS.columns[0])
cols = [c for c in d.columns if c.startswith("value")]
fig, ax = plt.subplots(figsize=(8.5, 0.19 * len(d) + 1.6))
y = np.arange(len(d))
lo, hi = d[cols].min(axis=1) * 100, d[cols].max(axis=1) * 100
ax.hlines(y, lo, hi, color=C.axis, lw=1.2, zorder=1)
for c_, col, mk in zip([C.s1, C.s2, C.s3], cols, ["o", "D", "s"]):
    ax.scatter(d[col] * 100, y, s=30, color=c_, marker=mk, edgecolor=C.surface, linewidth=1, zorder=3, label=col.replace("value: ", ""))
ax.axvline(0, color=C.axis, lw=1)
ax.set_yticks(y)
ax.set_yticklabels(d.index, fontsize=7)
ax.set_xlabel("% share move per +10bp in the home 10Y yield")
ax.set_title("'Rate sensitivity' as shown in three places of the dashboard (grey bar = spread)")
ax.legend(loc="upper left", fontsize=8)
ax.grid(axis="y", visible=False)
show(fig, "Rate sensitivity across tabs")

In [ ]:
rows = []
for j, sid in enumerate(STOCK_IDS):
    a, b = macro_f[j], full_f[j]
    if a["ok"] and b["ok"]:
        rows.append({"stock": sid, "header 'stock-specific' (macro model)": js_pct(a["specificVolAnn"], 0), "Risk tab 'Stock-specific' (selected model)": js_pct(b["specificVolAnn"], 1),
                     "same at 0dp": js_pct(a["specificVolAnn"], 0) == js_pct(b["specificVolAnn"], 0)})
SPEC_LABELS = pd.DataFrame(rows).set_index("stock")
n_spec = int((~SPEC_LABELS["same at 0dp"]).sum())
if n_spec:
    finding("LOW", "Presentation", "Two different 'stock-specific volatility' figures",
            f"The header's 'x% stock-specific' uses the macro model while the Overview, Exposures and Risk tabs use the selected model; they differ at the header's precision for {n_spec} stocks.",
            stocks=list(SPEC_LABELS.index[~SPEC_LABELS["same at 0dp"]]))

ATTR_TO_STATS = {"1M": "m1", "3M": "m3", "6M": "m6", "YTD": "ytd", "1Y": "y1", "3Y": "y3"}
rows = []
st_local = dv_stats()
for key, hk in ATTR_TO_STATS.items():
    for j, sid in enumerate(STOCK_IDS):
        a = dv_attr("sector", key, j)
        ref_a = REF_ATTR.get(("sector", key, sid)) or (a if "trading_days" in a else None)
        r_hdr = st_local[j]["ret"][hk]
        if not a.get("ok"):
            continue
        rows.append({"stock": sid, "period": key, "returns table": r_hdr, "attribution tab total": a["total"],
                     "shown returns table": js_pct(r_hdr, 1, True), "shown attribution": js_pct(a["total"], 1, True),
                     "days used": a["days"], "trading days": ref_a["trading_days"] if ref_a else None,
                     "attribution starts": ref_a["first_used"] if ref_a else a["start"], "label says": a["start"]})
RET_VS_ATTR = pd.DataFrame(rows)
RET_VS_ATTR["same on screen"] = RET_VS_ATTR["shown returns table"] == RET_VS_ATTR["shown attribution"]
bad = RET_VS_ATTR[~RET_VS_ATTR["same on screen"]]
add_bool_checks([{"item": f"{r.period} return identical on Overview and Attribution", "stock": r.stock, "passed": bool(r["same on screen"]),
                  "dashboard": r["shown attribution"], "reference": r["shown returns table"],
                  "note": f"attribution used {r['days used']} of {r['trading days']} trading days" if r["days used"] != r["trading days"] else None}
                 for _, r in RET_VS_ATTR.iterrows()], "Label consistency", layer="3. Presentation")
if len(bad):
    partial = bad[bad["days used"] != bad["trading days"]]
    finding("HIGH", "Presentation", "Attribution 'Total return' differs from the Returns table for the same period",
            f"{len(bad)} stock-period pair{'s show' if len(bad) > 1 else ' shows'} a different total return on the Attribution tab than in the Overview returns table: "
            + "; ".join(f"{r.stock} {r.period} {r['shown returns table']} in the returns table vs {r['shown attribution']} on the Attribution tab" for _, r in bad.head(6).iterrows())
            + f". {'In all of them' if len(partial) == len(bad) else f'In {len(partial)} of them'} exposures could not yet be estimated for the first months of the period "
            "(recent listing), so those days are skipped and the 'Total return' tile covers a shorter period than the dates printed under it. "
            "Suggested fix: start the period at the first estimable day and print that date, or show the full-period return and attribute the unexplained days to 'stock-specific'.",
            stocks=sorted(set(bad.stock)), evidence=bad[["stock", "period", "shown returns table", "shown attribution", "days used", "trading days", "attribution starts"]].to_string(index=False)[:3000])
display(bad if len(bad) else Markdown("Returns table and attribution totals agree on screen for every stock and period."))

## 8. Layer 4 — backtests of every forward-looking number
The Outlook, Risk and Macro tabs make statements about the future: a volatility forecast and price ranges that should hold "two in three" and "19 in 20" outcomes, a 1-day VaR, 12-month exposure forecasts with a stated accuracy gain and a range that "about two in three past outcomes fell inside", signals labelled tailwind or headwind, and stress replays. Each is tested here on data the forecast could not have seen. For the volatility forecasts the backtest refits the dashboard's own estimator at each historical quarter-end (section 8.1).

Pass bands are about materiality: a ±1σ range that holds 64% instead of 68% is fine; one that holds 50% makes "two in three" untrue.

In [ ]:
BT_ROWS = []  # one row per backtest verdict


def bt_verdict(test, measured, target, pass_band, warn_band, detail, fmt=lambda v: f"{v:.1%}"):
    gap = abs(measured - target) if is_num(measured) else np.inf
    status = "PASS" if gap <= pass_band else "WARN" if gap <= warn_band else "FAIL"
    BT_ROWS.append({"test": test, "measured": fmt(measured) if is_num(measured) else DASH, "target": fmt(target), "status": status, "detail": detail})
    add_bool_checks([{"item": test, "passed": status == "PASS", "dashboard": fmt(measured) if is_num(measured) else DASH, "reference": fmt(target), "note": detail}],
                    "Backtests", layer="4. Backtest")
    if status == "WARN":
        CHECK_FRAMES[-1].loc[:, "status"] = "WARN"
    return status


OBS_IDX = [np.flatnonzero(st.obs) for st in REF.stocks]

### 8.1 GARCH volatility forecasts and the price ranges
At every quarter-end (or month-end, see the configuration) the dashboard's volatility model is refitted on the three years of returns available then, with the dashboard's own estimator: its grid search, re-implemented in section 4.1 and shown in section 5.7 to reproduce the dashboard's parameters exactly (or the JavaScript itself, with `GARCH_BACKTEST_IN_JS = True`). Each forecast is scored against what followed over 21, 63 and 252 trading days: the share of outcomes inside ±1σ (the "two in three" band) and ±1.96σ (the "19 in 20" band) for the share **price** (what the chart shows) and the total return, and the accuracy of the variance forecast against two naive alternatives (last year's variance, and the three-year average variance) using the QLIKE loss.

In [ ]:
if RUN_BACKTESTS:
    H_EVAL = [21, 63, 252]
    origins = []
    t0 = time.time()
    if ENGINE and E.get("garchBacktest"):
        Hs = E["garchHorizons"]
        for j in range(len(REF.stocks)):
            for o in E["garchBacktest"][j]:
                origins.append((j, o["T"], dict(zip(Hs, o["cum"])), o["lrVar"]))
        GARCH_BT_SOURCE = "the dashboard's JavaScript"
    else:
        months = ("03", "06", "09", "12") if GARCH_BACKTEST_DATES == "Q" else None
        for j, st in enumerate(REF.stocks):
            for T in np.flatnonzero(REF.month_end)[:-1]:
                if T > st.last or not st.obs[T] or (months and REF.iso[T][5:7] not in months):
                    continue
                t_idx = OBS_IDX[j][(OBS_IDX[j] >= max(T - 756, st.first + 1)) & (OBS_IDX[j] <= T)]
                r = np.log(st.tri[t_idx] / st.tri[t_idx - 1])
                g = garch_grid_fit(r)
                if g is not None:
                    origins.append((j, T, {h: garch_cum_var(g, h) for h in H_EVAL}, g.lrVar))
        GARCH_BT_SOURCE = "the re-implemented dashboard estimator"
    print(f"{len(origins):,} volatility forecasts refitted with {GARCH_BT_SOURCE} in {time.time() - t0:.0f}s")
    rows = []
    for j, T, cum, lrv in origins:
        st, oi = REF.stocks[j], OBS_IDX[j]
        pos = int(np.searchsorted(oi, T))
        if pos >= len(oi) or oi[pos] != T:
            continue
        past = oi[max(0, pos - 252):pos + 1]
        lr_past = np.diff(np.log(st.tri[past]))
        naive = float(np.var(lr_past, ddof=1)) if len(lr_past) > 20 else np.nan
        for h in H_EVAL:
            if pos + h >= len(oi) or h not in cum:
                continue
            Th = oi[pos + h]
            path = st.tri[oi[pos:pos + h + 1]]
            rv = float(np.sum(np.diff(np.log(path)) ** 2))
            sd = math.sqrt(cum[h])
            rows.append({"stock": st.id, "date": REF.iso[T], "h": h, "forecast_var": cum[h], "z_price": math.log(st.px[Th] / st.px[T]) / sd,
                         "z_tri": math.log(st.tri[Th] / st.tri[T]) / sd, "rv": rv, "naive_var": naive * h, "longrun_var": lrv * h})
    GARCH_BT = pd.DataFrame(rows)
    qlike = lambda rv, f: np.mean(rv / f - np.log(rv / f) - 1)
    summ = []
    for h, g in GARCH_BT.groupby("h"):
        g = g[(g.rv > 0) & np.isfinite(g.naive_var)]
        summ.append({"horizon (trading days)": h, "forecasts": len(g), "stocks": g.stock.nunique(), "first origin": g.date.min(), "last origin": g.date.max(),
                     "price inside ±1σ": (g.z_price.abs() <= 1).mean(), "price inside ±1.96σ": (g.z_price.abs() <= 1.96).mean(),
                     "total return inside ±1σ": (g.z_tri.abs() <= 1).mean(), "total return inside ±1.96σ": (g.z_tri.abs() <= 1.96).mean(),
                     "realised / forecast variance": float(g.rv.sum() / g.forecast_var.sum()),
                     "QLIKE GARCH": qlike(g.rv, g.forecast_var), "QLIKE last-year variance": qlike(g.rv, g.naive_var), "QLIKE 3-year variance": qlike(g.rv, g.longrun_var)})
    GARCH_SUMMARY = pd.DataFrame(summ).set_index("horizon (trading days)")
    display(GARCH_SUMMARY.style.format({c: "{:.1%}" for c in GARCH_SUMMARY.columns if "inside" in c} | {c: "{:.3f}" for c in GARCH_SUMMARY.columns if "QLIKE" in c or "realised" in c}))
    for h, r in GARCH_SUMMARY.iterrows():
        bt_verdict(f"price range {h}d: share inside the 'two in three' band", r["price inside ±1σ"], 0.6827, 0.05, 0.10,
                   f"{r.forecasts:,} forecasts made at {'quarter' if GARCH_BACKTEST_DATES == 'Q' else 'month'}-ends, {r.stocks} stocks, {r['first origin']} to {r['last origin']}. Overlapping horizons: treat as indicative.")
        bt_verdict(f"price range {h}d: share inside the '19 in 20' band", r["price inside ±1.96σ"], 0.95, 0.03, 0.06, "same forecasts, ±1.96σ")
        better = r["QLIKE GARCH"] <= min(r["QLIKE last-year variance"], r["QLIKE 3-year variance"])
        BT_ROWS.append({"test": f"volatility forecast {h}d beats naive alternatives (QLIKE)", "measured": f"{r['QLIKE GARCH']:.3f}",
                        "target": f"<= {min(r['QLIKE last-year variance'], r['QLIKE 3-year variance']):.3f}", "status": "PASS" if better else "WARN",
                        "detail": "Lower QLIKE is better. WARN means a naive forecast did as well, so the GARCH adds little at this horizon."})
    wide = GARCH_SUMMARY[(GARCH_SUMMARY["price inside ±1σ"] - 0.6827).abs() > 0.05]
    if len(wide):
        over, under = (wide["price inside ±1σ"] > 0.6827).all(), (wide["price inside ±1σ"] < 0.6827).all()
        c196 = GARCH_SUMMARY["price inside ±1.96σ"]
        ratio = GARCH_SUMMARY["realised / forecast variance"]
        finding("MEDIUM", "Backtest", "Price ranges hold more than 'two in three' outcomes" if over else "Price ranges hold fewer than 'two in three' outcomes" if under
                else "Price ranges miss 'two in three' at some horizons",
                "Refitted at each past date, the ±1σ price ranges contained "
                + ", ".join(f"{r['price inside ±1σ']:.0%} of outcomes at {h} days" for h, r in wide.iterrows())
                + f" (promised about 68%); realised variance was {ratio.min():.0%}-{ratio.max():.0%} of the forecast across horizons. "
                + ("The ranges are conservative rather than wrong; say 'at least two in three' or recalibrate. " if over
                   else "The ranges are too narrow; recalibrate before quoting 'two in three'. " if under else "Recalibrate before quoting 'two in three'. ")
                + "The ±1.96σ ranges held " + ", ".join(f"{v:.0%} at {h} days" for h, v in c196.items()) + " (promised 19 in 20).")

In [ ]:
if RUN_BACKTESTS and len(GARCH_BT):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.3))
    ax = axes[0]
    x = np.arange(len(GARCH_SUMMARY))
    w = 0.36
    for k, (col, nominal, colr) in enumerate([("price inside ±1σ", 0.6827, C.s1), ("price inside ±1.96σ", 0.95, C.s3)]):
        vals = GARCH_SUMMARY[col].to_numpy()
        ax.bar(x + (k - 0.5) * w, vals * 100, width=w * 0.92, color=colr, label=col.replace("price ", "share of prices "))
        for xi, v in zip(x + (k - 0.5) * w, vals):
            ax.text(xi, v * 100 + 1, f"{v:.0%}", ha="center", fontsize=8.5, color=C.ink2)
        ax.hlines(nominal * 100, x[0] - 0.6, x[-1] + 0.6, color=colr, lw=1)
        ax.text(x[-1] + 0.62, nominal * 100, f" promised {nominal:.0%}", va="center", fontsize=8, color=C.ink2)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{h} days" for h in GARCH_SUMMARY.index])
    ax.set_ylim(0, 108)
    ax.set_ylabel("% of outcomes inside the band")
    ax.set_title("Price ranges: promised vs realised coverage")
    ax.legend(loc="lower left", fontsize=8)
    ax.grid(axis="x", visible=False)
    ax = axes[1]
    g = GARCH_BT[(GARCH_BT.h == 21) & (GARCH_BT.rv > 0)]
    fv = np.sqrt(g.forecast_var / 21 * 252) * 100
    rvv = np.sqrt(g.rv / 21 * 252) * 100
    ax.scatter(fv, rvv, s=8, color=C.s1, alpha=0.25, linewidth=0)
    bins = pd.qcut(fv, 10, duplicates="drop")
    means = pd.DataFrame({"f": fv, "r": rvv}).groupby(bins, observed=True).mean()
    ax.plot(means.f, means.r, color=C.s2, marker="o", ms=5, lw=2, label="average by forecast decile")
    lim = np.nanpercentile(np.r_[fv, rvv], 99.5)
    ax.plot([0, lim], [0, lim], color=C.axis, lw=1)
    ax.set_xlim(0, lim)
    ax.set_ylim(0, lim)
    ax.set_xlabel("forecast volatility for the next month (%, annualised)")
    ax.set_ylabel("realised volatility that month (%)")
    ax.set_title("Next-month volatility: forecast vs realised")
    ax.legend(loc="upper left", fontsize=8)
    fig.tight_layout()
    show(fig, "GARCH backtest")
    by_year = GARCH_BT[GARCH_BT.h == 252].assign(year=lambda d: d.date.str[:4]).groupby("year").agg(
        forecasts=("z_price", "size"), inside_1sd=("z_price", lambda z: (z.abs() <= 1).mean()), inside_196=("z_price", lambda z: (z.abs() <= 1.96).mean()),
        mean_z=("z_price", "mean"))
    display(Markdown("**12-month price ranges by the year the forecast was made** (a mean z below zero means prices mostly ended in the lower half of the range; the ranges assume no drift and prices fall by the dividend on ex-dates):"))
    display(by_year.style.format({"inside_1sd": "{:.0%}", "inside_196": "{:.0%}", "mean_z": "{:+.2f}"}))

### 8.2 One-day VaR (95%)
The Risk tab's historical VaR uses the last year of daily returns. Re-computing it every day since 2020 and counting how often the next day's loss exceeded it gives a direct test: the rate should be 5%, and exceedances should not cluster. Kupiec's test checks the rate, Christoffersen's the clustering.

In [ ]:
def kupiec(x, n, p=0.05):
    if n == 0:
        return np.nan
    ph = x / n
    ll0 = (n - x) * math.log(1 - p) + x * math.log(p)
    ll1 = (n - x) * math.log(1 - ph) + x * math.log(ph) if 0 < x < n else 0.0
    return float(1 - sps.chi2.cdf(-2 * (ll0 - ll1), 1))


def christoffersen(hits):
    h = np.asarray(hits, int)
    a, b = h[:-1], h[1:]
    n00, n01, n10, n11 = [int(((a == i) & (b == k)).sum()) for i in (0, 1) for k in (0, 1)]
    p0 = n01 / max(n00 + n01, 1)
    p1 = n11 / max(n10 + n11, 1)
    p = (n01 + n11) / max(len(a), 1)
    lg = lambda q, k: k * math.log(q) if q > 0 else 0.0
    l0 = lg(1 - p, n00 + n10) + lg(p, n01 + n11)
    l1 = lg(1 - p0, n00) + lg(p0, n01) + lg(1 - p1, n10) + lg(p1, n11)
    return float(1 - sps.chi2.cdf(-2 * (l0 - l1), 1))


if RUN_BACKTESTS:
    y1_idx = np.array([REF.on_or_before(REF.shift_years(REF.iso[t], 1)) for t in range(REF.n)])
    start_bt = REF.on_or_before("2020-01-01") + 1
    var_rows, var_obs = [], []
    for j, st in enumerate(REF.stocks):
        r = REF.day_returns(st, DV["ccy"])
        oi = OBS_IDX[j]
        hits, fc_var, fc_es, loss = [], [], [], []
        for p in range(len(oi) - 1):
            t = oi[p]
            if t < start_bt:
                continue
            lo = int(np.searchsorted(oi, max(y1_idx[t], st.first), side="right"))
            w = r[oi[lo:p + 1]]
            if len(w) <= 50 or np.isnan(w).any():
                continue
            q = int(math.floor(0.05 * len(w)))
            part = np.partition(w, q)
            v, es = -part[q], -part[:q + 1].mean()
            nxt = r[oi[p + 1]]
            if not np.isfinite(nxt):
                continue
            hits.append(nxt < -v)
            fc_var.append(v)
            fc_es.append(es)
            loss.append(-nxt)
            var_obs.append((st.id, REF.iso[oi[p + 1]], v, es, -nxt))
        hits = np.array(hits)
        if len(hits) < 100:
            continue
        loss, fc_es = np.array(loss), np.array(fc_es)
        var_rows.append({"stock": st.id, "days": len(hits), "exceedances": int(hits.sum()), "rate": hits.mean(), "kupiec_p": kupiec(int(hits.sum()), len(hits)),
                         "christoffersen_p": christoffersen(hits), "loss_on_exceedance / ES": float(loss[hits].mean() / fc_es[hits].mean()) if hits.any() else np.nan})
    VAR_BT = pd.DataFrame(var_rows).set_index("stock")
    pooled_rate = VAR_BT.exceedances.sum() / VAR_BT.days.sum()
    display(VAR_BT.sort_values("rate", ascending=False).style.format({"rate": "{:.2%}", "kupiec_p": "{:.3f}", "christoffersen_p": "{:.3f}", "loss_on_exceedance / ES": "{:.2f}"}))
    bt_verdict("1-day VaR 95%: pooled exceedance rate", pooled_rate, 0.05, 0.005, 0.01,
               f"{VAR_BT.days.sum():,} stock-days since 2020; {int((VAR_BT.kupiec_p < 0.05).sum())} of {len(VAR_BT)} stocks fail Kupiec at 5%, "
               f"{int((VAR_BT.christoffersen_p < 0.05).sum())} show clustered exceedances (Christoffersen).", fmt=lambda v: f"{v:.2%}")
    bt_verdict("1-day expected shortfall: average loss beyond VaR vs ES", float(VAR_BT["loss_on_exceedance / ES"].median()), 1.0, 0.10, 0.25,
               "median across stocks of (average loss on exceedance days) / (ES forecast on those days); above 1 means ES understates tail losses", fmt=lambda v: f"{v:.2f}")
    fig, ax = plt.subplots(figsize=(11, 3.8))
    d = VAR_BT.sort_values("rate")
    n_med = int(d.days.median())
    band = sps.binom.ppf([0.025, 0.975], n_med, 0.05) / n_med
    ax.axhspan(band[0] * 100, band[1] * 100, color=C.neutral, zorder=0)
    ax.axhline(5, color=C.axis, lw=1)
    colors = [C.critical if p < 0.05 else C.s1 for p in d.kupiec_p]
    ax.scatter(np.arange(len(d)), d.rate * 100, s=30, color=colors, edgecolor=C.surface, linewidth=1, zorder=3)
    for i, (sid, r_) in enumerate(d.iterrows()):
        if r_.kupiec_p < 0.05:
            ax.annotate(sid, (i, r_.rate * 100), xytext=(0, 6), textcoords="offset points", ha="center", fontsize=7.5, color=C.ink2)
    ax.set_xticks([])
    ax.set_xlabel("stocks, sorted by exceedance rate (red = fails Kupiec at 5%, labelled)")
    ax.set_ylabel("% of days losing more than VaR")
    ax.set_title(f"1-day 95% VaR backtest since 2020: pooled rate {pooled_rate:.2%} (target 5%, shaded = 95% band for one stock)")
    ax.grid(axis="x", visible=False)
    show(fig, "VaR backtest")

### 8.3 Exposure forecasts: is the stated accuracy real out of sample?
The dashboard fits its blend weights (2-year estimate, last year, sub-sector median) on **all** past forecast dates at once and then reports how much smaller the errors were than with the 2-year estimate alone. That figure is in-sample: the weights have seen the outcomes they are scored on. Here the weights are re-fitted **walk-forward**, using only forecasts whose 52-week outcome was known at the time, and scored on the next forecast date. The same test checks the claim that "about two in three past outcomes fell inside" the likely range.

In [ ]:
if RUN_BACKTESTS:
    fc = REF_FC[DV["ccy"]]
    pairs = fc["pairs"]
    wf_rows, cov_rows = [], []
    for f_id in fc["factors"]:
        pf = pairs[pairs.factor == f_id].copy()
        w_is = fc["weights"][f_id]
        pf["fc_insample"] = pf.peers + w_is["long"] * (pf.long - pf.peers) + w_is["short"] * (pf.short - pf.peers)
        parts = []
        for k in sorted(pf.k.unique()):
            train = pf[pf.k <= k - 13]  # outcomes of these forecasts were complete 52 weeks later
            if len(train) < 30:
                continue
            w = fit_blend_weights((train.long - train.peers).to_numpy(), (train.short - train.peers).to_numpy(), (train.realised - train.peers).to_numpy())
            test = pf[pf.k == k].copy()
            test["fc_walkforward"] = test.peers + w["long"] * (test.long - test.peers) + w["short"] * (test.short - test.peers)
            test["w_long"], test["w_short"] = w["long"], w["short"]
            parts.append(test)
        if not parts:
            continue
        oos = pd.concat(parts)
        rm = lambda e: float(np.sqrt(np.mean(np.square(e))))
        unit = FACTORS[f_id].shock * 100 if FACTORS[f_id].unit in ("bp", "pts") else 1.0
        res = {"factor": f_id, "forecasts scored": len(oos), "from": oos.origin.min(), "to": oos.origin.max(),
               "RMSE 2-year estimate": rm(oos.realised - oos.long) * unit, "RMSE last year": rm(oos.realised - oos.short) * unit,
               "RMSE peer median": rm(oos.realised - oos.peers) * unit, "RMSE blend, walk-forward": rm(oos.realised - oos.fc_walkforward) * unit,
               "RMSE blend, dashboard weights (in-sample)": rm(oos.realised - oos.fc_insample) * unit}
        res["gain vs 2-year, walk-forward"] = 1 - res["RMSE blend, walk-forward"] / res["RMSE 2-year estimate"]
        res["gain vs 2-year, as stated (in-sample, winsorised)"] = 1 - w_is["rmse"] / w_is["rmseLong"]
        wf_rows.append(res)
        inside = (pf.realised - pf.fc_insample).abs() <= w_is["rmse"]
        cov_rows.append({"factor": f_id, "share inside the likely range": inside.mean(), "forecasts": len(pf)})
    FC_BT = pd.DataFrame(wf_rows).set_index("factor")
    FC_COV = pd.DataFrame(cov_rows).set_index("factor")
    display(Markdown("RMSE in display units (beta, or % share move per standard shock for rate-type factors):"))
    display(FC_BT.style.format({c: "{:.4f}" for c in FC_BT.columns if c.startswith("RMSE")} | {c: "{:.1%}" for c in FC_BT.columns if c.startswith("gain")}))
    display(FC_COV.style.format({"share inside the likely range": "{:.1%}"}))
    for f_id, r in FC_BT.iterrows():
        bt_verdict(f"exposure forecast {f_id}: out-of-sample gain vs the stated in-sample gain", r["gain vs 2-year, walk-forward"], r["gain vs 2-year, as stated (in-sample, winsorised)"],
                   0.03, 0.08, f"walk-forward {r['gain vs 2-year, walk-forward']:.1%} vs stated {r['gain vs 2-year, as stated (in-sample, winsorised)']:.1%}; {r['forecasts scored']:,} forecasts")
    for f_id, r in FC_COV.iterrows():
        bt_verdict(f"exposure forecast {f_id}: 'about two in three' inside the likely range", r["share inside the likely range"], 2 / 3, 0.05, 0.10,
                   "range = forecast ± the pooled (winsorised) RMSE, as displayed; outcomes not winsorised")
    short_ = FC_BT[FC_BT["gain vs 2-year, walk-forward"] < FC_BT["gain vs 2-year, as stated (in-sample, winsorised)"] - 0.03]
    if len(short_):
        peers_close = FC_BT[(FC_BT["RMSE peer median"] - FC_BT["RMSE blend, walk-forward"]) / FC_BT["RMSE blend, walk-forward"] < 0.02]
        finding("MEDIUM", "Backtest", "The exposure forecasts' stated accuracy gain is an in-sample figure",
                "Re-fitting the blend weights only on outcomes known at the time, the gain over the 2-year estimate is "
                + ", ".join(f"{f} {r['gain vs 2-year, walk-forward']:.0%} (stated {r['gain vs 2-year, as stated (in-sample, winsorised)']:.0%})" for f, r in FC_BT.iterrows())
                + ". The note under the forecast table (and the README's range) should quote the out-of-sample figures."
                + (f" For {', '.join(peers_close.index)} the sub-sector median alone is within 2% of the blend's accuracy." if len(peers_close) else ""))
    readme = (REPO / "README.md").read_text(encoding="utf-8")
    m = re.search(r"\((\d+)[–-](\d+)% smaller in ([A-Za-z]+ \d{4})\)", readme)
    gains = [1 - fc["weights"][f]["rmse"] / fc["weights"][f]["rmseLong"] for f in fc["factors"]]
    if m and gains:
        lo_, hi_ = int(m.group(1)), int(m.group(2))
        now_lo, now_hi = round(min(gains) * 100), round(max(gains) * 100)
        ok = (lo_, hi_) == (now_lo, now_hi)
        add_bool_checks([{"item": f"README claim '{m.group(0)}' matches this data file", "passed": ok, "dashboard": f"{now_lo}-{now_hi}%", "reference": f"{lo_}-{hi_}%"}],
                        "Documentation", layer="3. Presentation")
        if not ok:
            finding("LOW", "Documentation", "README accuracy claim is out of date", f"README says {lo_}-{hi_}%; this data file gives {now_lo}-{now_hi}% (in-sample).")
    fig, axes = plt.subplots(1, len(FC_BT), figsize=(3.4 * len(FC_BT), 3.4), squeeze=False)
    methods = ["RMSE 2-year estimate", "RMSE last year", "RMSE peer median", "RMSE blend, walk-forward", "RMSE blend, dashboard weights (in-sample)"]
    short = ["2-year estimate", "last year", "peer median", "blend, walk-forward", "blend, as shown"]
    for ax, (f_id, r) in zip(axes[0], FC_BT.iterrows()):
        vals = [r[m_] for m_ in methods]
        colors = [C.deemph, C.deemph, C.deemph, C.s1, C.s2]
        ax.barh(short[::-1], vals[::-1], color=colors[::-1], height=0.6)
        for yi, v in enumerate(vals[::-1]):
            ax.text(v, yi, f" {v:.3f}", va="center", fontsize=8, color=C.ink2)
        ax.set_title(f"{f_id}: out-of-sample RMSE", fontsize=10)
        ax.set_xlim(0, max(vals) * 1.3)
        ax.grid(axis="y", visible=False)
    fig.suptitle("Exposure forecasts scored out of sample (lower is better; blue = honest walk-forward blend)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    show(fig, "Exposure forecast backtest")

### 8.4 Price signals: how reliable is the record the Outlook tab quotes?
The dashboard calls a signal reliable when the average monthly rank IC is at least two standard errors from zero (simple t-statistic). Below: the same record with Newey–West errors, a block-bootstrap interval, a split into halves, and the multiple-testing view (five signals tested at once).

In [ ]:
if RUN_BACKTESTS:
    REF_SIG = REF.signal_backtest() if "REF_SIG" not in globals() else REF_SIG
    rng = np.random.default_rng(42)
    rows = []
    for s_id, s in REF_SIG["signals"].items():
        ic = s["ic_series"].dropna()
        T = len(ic)
        nw = sm.OLS(ic.to_numpy(), np.ones(T)).fit(cov_type="HAC", cov_kwds={"maxlags": 3}).tvalues[0]
        block = 6
        means = []
        for _ in range(2000):
            starts = rng.integers(0, T - block + 1, size=int(math.ceil(T / block)))
            idx = np.concatenate([np.arange(a, a + block) for a in starts])[:T]
            means.append(ic.to_numpy()[idx].mean())
        half = T // 2
        rows.append({"signal": s_id, "months": T, "mean IC": ic.mean(), "t (as shown)": s["t"], "t (Newey-West, 3 lags)": nw,
                     "95% bootstrap CI low": np.percentile(means, 2.5), "95% bootstrap CI high": np.percentile(means, 97.5),
                     "first half IC": ic.iloc[:half].mean(), "second half IC": ic.iloc[half:].mean(),
                     "IC without the partial last month": ic.iloc[:-1].mean(), "hit rate": (ic > 0).mean(),
                     "reliable as shown (|t| >= 2)": abs(s["t"]) >= 2, "survives Bonferroni (|t_NW| >= 2.58)": abs(nw) >= 2.58})
    SIG_BT = pd.DataFrame(rows).set_index("signal")
    display(SIG_BT.style.format({c: "{:+.3f}" for c in SIG_BT.columns if "IC" in c} | {c: "{:+.2f}" for c in SIG_BT.columns if c.startswith("t ")} | {"hit rate": "{:.0%}"}))
    for s_id, r in SIG_BT.iterrows():
        shown, robust = r["reliable as shown (|t| >= 2)"], r["survives Bonferroni (|t_NW| >= 2.58)"]
        status = "PASS" if shown == robust or not shown else "WARN"
        BT_ROWS.append({"test": f"signal '{s_id}': reliability label holds under robust errors", "measured": f"t_NW {r['t (Newey-West, 3 lags)']:+.2f}",
                        "target": "same verdict as |t| >= 2", "status": status,
                        "detail": f"shown t {r['t (as shown)']:+.2f}; halves {r['first half IC']:+.3f} / {r['second half IC']:+.3f}; CI [{r['95% bootstrap CI low']:+.3f}, {r['95% bootstrap CI high']:+.3f}]"})
    first_ic = min(s["ic_series"].dropna().index.min() for s in REF_SIG["signals"].values())
    finding("MEDIUM", "Methodology", "Signal record uses today's coverage list (survivorship)",
            f"The monthly rank ICs are computed on the {len(REF.stocks)} stocks in today's coverage list, back to {js_month_year(str(first_ic)[:10])}. "
            "Companies that were taken over, delisted or dropped from coverage in that period are not in the data file, which tends to flatter "
            "momentum and low-volatility records. Quote the signal record as indicative.")
    fig, axes = plt.subplots(1, len(SIG_BT), figsize=(3.0 * len(SIG_BT), 2.8), sharey=True)
    for ax, (s_id, s) in zip(axes, REF_SIG["signals"].items()):
        ic = s["ic_series"].dropna()
        ax.plot(ic.index, ic.cumsum(), color=C.s1, lw=2)
        ax.axhline(0, color=C.axis, lw=1)
        ax.set_title(f"{s_id}: t {s['t']:+.1f} (NW {SIG_BT.loc[s_id, 't (Newey-West, 3 lags)']:+.1f})", fontsize=9.5)
        ax.tick_params(axis="x", labelsize=7.5, rotation=45)
    axes[0].set_ylabel("cumulative monthly rank IC")
    fig.suptitle("Price signals: cumulative rank IC (a steady climb = a consistent record)", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    show(fig, "Signal records")

### 8.5 Stress episodes: would the model have called them in advance?
The Macro tab replays each episode with **today's** exposures, which is a what-if rather than a test. Here each stock's macro model is estimated with data up to the day before each episode (default weekly 3-year window), and the implied move is compared with what the stock actually did, across the coverage.

In [ ]:
if RUN_BACKTESTS:
    ep_rows, ep_pts = [], []
    for ep in EPISODES:
        a, b = REF.on_or_before(ep["start"]), REF.on_or_before(ep["end"])
        if a < 0 or b <= a:
            continue
        for st in REF.stocks:
            if a < st.first:
                continue
            fit = REF.fit(st, MACRO_IDS, "W", 3, DV["ccy"], end=a, hac=False, fast=True)
            if not fit["ok"]:
                continue
            imp = REF.episode_impact(st, fit, ep, DV["ccy"])
            if imp and np.isfinite(imp["actual"]):
                ep_pts.append({"episode": ep["label"], "stock": st.id, "implied ex-ante": imp["implied"], "actual": imp["actual"]})
    EP_BT = pd.DataFrame(ep_pts)
    for lab, g in EP_BT.groupby("episode", sort=False):
        ep_rows.append({"episode": lab, "stocks": len(g), "correlation implied vs actual": pearson(g["implied ex-ante"], g["actual"]),
                        "same sign": float((np.sign(g["implied ex-ante"]) == np.sign(g["actual"])).mean()),
                        "median actual": g.actual.median(), "median implied": g["implied ex-ante"].median(),
                        "mean abs error (pp)": float((g.actual - g["implied ex-ante"]).abs().mean() * 100)})
    EP_SUMMARY = pd.DataFrame(ep_rows).set_index("episode")
    display(EP_SUMMARY.style.format({"correlation implied vs actual": "{:+.2f}", "same sign": "{:.0%}", "median actual": "{:+.1%}", "median implied": "{:+.1%}", "mean abs error (pp)": "{:.1f}"}))
    for lab, r in EP_SUMMARY.iterrows():
        BT_ROWS.append({"test": f"stress replay '{lab}': ex-ante model ranks stocks correctly", "measured": f"corr {r['correlation implied vs actual']:+.2f}",
                        "target": "> 0.3", "status": "PASS" if r["correlation implied vs actual"] > 0.3 else "WARN",
                        "detail": f"{r.stocks} stocks; median implied {r['median implied']:+.1%} vs actual {r['median actual']:+.1%}; same sign {r['same sign']:.0%}"})
    big_eps = EP_SUMMARY[EP_SUMMARY["median actual"].abs() > 0.10]
    if len(big_eps):
        finding("MEDIUM", "Backtest", "Stress replays understate the size of large real-estate moves",
                "Estimated with data from before each episode, the macro model (market, home 10Y, credit, FX) implied "
                + "; ".join(f"{lab}: {r['median implied']:+.0%} vs {r['median actual']:+.0%} actual (median stock)" for lab, r in big_eps.iterrows())
                + ". The macro model has no real-estate sector factor, so the sector-wide part of these moves is left out and 'Implied today' in the stress table "
                "understates tail moves. The table already shows the actual move alongside; the note under it should say this.")
    n_ep = EP_BT.episode.nunique()
    cols_ = 4
    fig, axes = plt.subplots(int(math.ceil(n_ep / cols_)), cols_, figsize=(14, 3.3 * math.ceil(n_ep / cols_)), squeeze=False)
    for ax, (lab, g) in zip(axes.ravel(), EP_BT.groupby("episode", sort=False)):
        v = np.r_[g["implied ex-ante"], g.actual] * 100
        lo, hi = np.nanmin(v), np.nanmax(v)
        pad = (hi - lo) * 0.08 + 0.5
        ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], color=C.axis, lw=1)
        ax.scatter(g["implied ex-ante"] * 100, g.actual * 100, s=22, color=C.s1, edgecolor=C.surface, linewidth=1)
        ax.set_title(f"{lab}\ncorr {EP_SUMMARY.loc[lab, 'correlation implied vs actual']:+.2f}", fontsize=9.5)
        ax.set_xlabel("implied from exposures before (%)", fontsize=8)
        ax.set_ylabel("actual (%)", fontsize=8)
    for ax in axes.ravel()[n_ep:]:
        ax.set_visible(False)
    fig.suptitle("Stress episodes: ex-ante model move vs actual move, one dot per stock", x=0.01, ha="left", fontweight="bold")
    fig.tight_layout()
    show(fig, "Stress episode backtest")

### 8.6 How robust are the headline exposures?
Three questions a sceptical reader will ask: does a single week drive the estimate (leave-one-out influence), does the answer depend on return frequency (daily betas of less liquid stocks are biased towards zero when prices react late), and how many of the "significant" labels survive a false-discovery-rate correction across all the coefficients on screen.

In [ ]:
if RUN_BACKTESTS:
    rows = []
    for j, (st, fit) in enumerate(zip(REF.stocks, ref_fits("full"))):
        if not fit["ok"]:
            continue
        X = np.column_stack([np.ones(len(fit["y"])), fit["X"]])
        XtXi = np.linalg.inv(X.T @ X)
        h = np.einsum("ij,jk,ik->i", X, XtXi, X)
        dfb = -(XtXi @ X.T).T * (fit["resid"] / (1 - h))[:, None]  # change in each coefficient when one week is left out
        for f in ("MKT", "RATES"):
            if f not in fit["factors"]:
                continue
            k = fit["factors"].index(f) + 1
            i = int(np.argmax(np.abs(dfb[:, k])))
            b, db = fit["betas"][f], dfb[i, k]
            rows.append({"stock": st.id, "factor": f, "estimate": b, "largest one-week change": db, "relative change": abs(db) / abs(b) if b else np.inf,
                         "week": REF.iso[fit["rows"][i]], "sign flips": np.sign(b + db) != np.sign(b), "shown": exposure_fmt(f)(b), "shown without that week": exposure_fmt(f)(b + db)})
    INFLUENCE = pd.DataFrame(rows)
    fragile = INFLUENCE[(INFLUENCE["relative change"] > 0.25) | INFLUENCE["sign flips"]]
    display(Markdown(f"**{len(fragile)} of {len(INFLUENCE)} headline exposures move by more than 25% (or change sign) when one week is dropped:**"))
    display(fragile.sort_values("relative change", ascending=False).head(20))
    if len(fragile):
        n_by = fragile.factor.value_counts()
        finding("MEDIUM", "Robustness", "Some exposures hinge on a single week",
                f"{len(fragile)} market or rate exposures in the default model change by more than 25% (or flip sign) when one week is left out "
                f"({int(n_by.get('RATES', 0))} rate sensitivities, {int(n_by.get('MKT', 0))} market betas; {int(fragile['sign flips'].sum())} flip sign). "
                "Show these with their confidence interval, or mark them as not significant, before quoting them.",
                stocks=sorted(set(fragile.stock)))
    freq_rows = []
    for j, st in enumerate(REF.stocks):
        vals = {}
        for fq in ("D", "W", "M"):
            f = E["fits"].get(f"{fq}|3|{DV['ccy']}|capm", [None] * len(REF.stocks))[j] if ENGINE and f"{fq}|3|{DV['ccy']}|capm" in E["fits"] else REF.fit(st, ["MKT"], fq, 3, DV["ccy"], hac=False, fast=True)
            vals[fq] = f["betas"]["MKT"] if f and f["ok"] else np.nan
        freq_rows.append({"stock": st.id, "daily": vals["D"], "weekly": vals["W"], "monthly": vals["M"], "adv_3m_eur": FUND[st.id].get("adv_3m_eur")})
    FREQ_BETA = pd.DataFrame(freq_rows).set_index("stock")
    FREQ_BETA["weekly - daily"] = FREQ_BETA.weekly - FREQ_BETA.daily
    fig, ax = plt.subplots(figsize=(6.4, 5.2))
    lim = [np.nanmin(FREQ_BETA[["daily", "weekly"]].to_numpy()) - 0.1, np.nanmax(FREQ_BETA[["daily", "weekly"]].to_numpy()) + 0.1]
    ax.plot(lim, lim, color=C.axis, lw=1)
    big = FREQ_BETA["weekly - daily"].abs() > 0.25
    ax.scatter(FREQ_BETA.daily[~big], FREQ_BETA.weekly[~big], s=30, color=C.s1, edgecolor=C.surface, linewidth=1)
    ax.scatter(FREQ_BETA.daily[big], FREQ_BETA.weekly[big], s=34, color=C.s2, edgecolor=C.surface, linewidth=1)
    for sid, r in FREQ_BETA[big].iterrows():
        ax.annotate(sid, (r.daily, r.weekly), xytext=(4, 3), textcoords="offset points", fontsize=8, color=C.ink2)
    ax.set_xlabel("market beta from daily returns (3Y)")
    ax.set_ylabel("market beta from weekly returns (3Y)")
    ax.set_title(f"Beta depends on frequency: {int(big.sum())} stocks differ by more than 0.25")
    show(fig, "Daily vs weekly beta")
    if big.any():
        finding("LOW", "Methodology", "Market beta depends on the return frequency for some stocks",
                f"For {int(big.sum())} stock{'s' if big.sum() > 1 else ''} the 3-year market beta from daily returns differs from the weekly one by more than 0.25: "
                + ", ".join(f"{sid} daily {r.daily:.2f} vs weekly {r.weekly:.2f}" for sid, r in FREQ_BETA[big].iterrows())
                + ". Prices that react to the market with a delay, or that trade on a different close, pull daily betas away from weekly ones. "
                "Quote the default weekly figure for these names.", stocks=list(FREQ_BETA.index[big]))
    tvals = []
    for j, fit in enumerate(dv_fits("full")):
        if fit["ok"]:
            for f in fit["factors"]:
                if f != "MKT" and is_num(fit["t"][f]):
                    tvals.append((STOCK_IDS[j], f, fit["t"][f]))
    TT = pd.DataFrame(tvals, columns=["stock", "factor", "t"])
    TT["p"] = 2 * (1 - sps.norm.cdf(TT.t.abs()))
    m_ = len(TT)
    ranks_ = TT.p.rank(method="first")
    thresh = TT.p.sort_values().to_numpy() <= (np.arange(1, m_ + 1) / m_) * 0.05
    k_max = np.flatnonzero(thresh).max() + 1 if thresh.any() else 0
    TT["significant as shown (|t| >= 1.96)"] = TT.t.abs() >= 1.96
    TT["survives FDR 5%"] = ranks_ <= k_max
    n_shown, n_fdr = int(TT["significant as shown (|t| >= 1.96)"].sum()), int(TT["survives FDR 5%"].sum())
    display(Markdown(f"**Multiple testing:** {n_shown} of {m_} non-market coefficients in the default model are shown as significant (|t| ≥ 1.96); "
                     f"about {0.05 * m_:.0f} would be expected by chance alone, and **{n_fdr}** survive a Benjamini-Hochberg false-discovery-rate correction at 5%."))
    display(TT.groupby("factor")[["significant as shown (|t| >= 1.96)", "survives FDR 5%"]].sum())
    if n_shown > n_fdr:
        share_ = n_fdr / n_shown
        finding("LOW" if share_ >= 0.8 else "MEDIUM", "Methodology", "Some 'significant' labels would not survive a multiple-testing correction",
                f"{n_shown} of {m_} non-market exposures in the default model are coloured as significant; {n_fdr} ({share_:.0%}) survive a false-discovery-rate correction at 5% "
                f"(chance alone would give about {0.05 * m_:.0f}). The style takeaways pick the largest |t| per stock, which compounds this for style factors.")

### 8.7 Out-of-sample explanatory power of the factor model
The R² on screen is in-sample. Here the default macro model is estimated at each quarter-end since 2022 and used to explain the **next** 13 weeks of returns with the factor moves that actually happened. A large gap means the in-sample split into systematic and stock-specific risk overstates what the model can explain going forward.

In [ ]:
if RUN_BACKTESTS:
    rows = []
    q_ends = [t for t in np.flatnonzero(REF.month_end) if REF.iso[t][5:7] in ("03", "06", "09", "12") and REF.iso[t] >= "2022-01-01"]
    for st in REF.stocks:
        for T in q_ends:
            fit = REF.fit(st, MACRO_IDS, "W", 3, DV["ccy"], end=T, hac=False, fast=True)
            if not fit["ok"]:
                continue
            nxt = REF.panel(st, fit["factors"], "W", DV["ccy"], start=T, end=min(T + 70, REF.asof), orth=False)
            if len(nxt.y) < 8 or nxt.X.shape[1] != len(fit["factors"]):
                continue
            pred = fit["alpha"] + nxt.X @ np.array([fit["betas"][f] for f in fit["factors"]])
            rows.append({"stock": st.id, "origin": REF.iso[T], "in_sample_r2": fit["r2"], "sse": float(np.sum((nxt.y - pred) ** 2)),
                         "sst": float(np.sum((nxt.y - nxt.y.mean()) ** 2)), "n": len(nxt.y)})
    OOS = pd.DataFrame(rows)
    by_stock = OOS.groupby("stock").agg(in_sample_r2=("in_sample_r2", "mean"), sse=("sse", "sum"), sst=("sst", "sum"))
    by_stock["out_of_sample_r2"] = 1 - by_stock.sse / by_stock.sst
    pooled_oos = 1 - OOS.sse.sum() / OOS.sst.sum()
    display(Markdown(f"Pooled out-of-sample R² **{pooled_oos:.2f}** vs average in-sample R² **{OOS.in_sample_r2.mean():.2f}** ({len(OOS):,} stock-quarters)."))
    bt_verdict("macro model: out-of-sample R² close to in-sample R²", pooled_oos, float(OOS.in_sample_r2.mean()), 0.08, 0.15,
               "Pooled over stocks and quarters since 2022; 13-week horizons.", fmt=lambda v: f"{v:.2f}")
    fig, ax = plt.subplots(figsize=(6.2, 5))
    ax.plot([0, 1], [0, 1], color=C.axis, lw=1)
    ax.scatter(by_stock.in_sample_r2, by_stock.out_of_sample_r2, s=30, color=C.s1, edgecolor=C.surface, linewidth=1)
    worst = by_stock.assign(gap=by_stock.in_sample_r2 - by_stock.out_of_sample_r2).nlargest(5, "gap")
    for sid, r in worst.iterrows():
        ax.annotate(sid, (r.in_sample_r2, r.out_of_sample_r2), xytext=(4, -3), textcoords="offset points", fontsize=8, color=C.ink2)
    ax.set_xlim(0, 1)
    ax.set_ylim(min(-0.2, by_stock.out_of_sample_r2.min() - 0.05), 1)
    ax.set_xlabel("in-sample R² (what the dashboard shows)")
    ax.set_ylabel("out-of-sample R², next 13 weeks")
    ax.set_title("Macro model: explanatory power in and out of sample")
    show(fig, "Out-of-sample R2")

In [ ]:
if RUN_BACKTESTS:
    BACKTESTS = pd.DataFrame(BT_ROWS)
    display(style_status(BACKTESTS))

## 9. Methodology review: issues that do not show up as mismatches
The reconciliation proves the dashboard computes what its code says. This section asks whether what the code says is right, and measures each issue's effect on the numbers on screen, so it can be judged by materiality rather than in the abstract.

### 9.1 Month arithmetic on the 29th, 30th and 31st (latent)
`shiftYears()` in `model.js` subtracts months with JavaScript's `setUTCMonth`, which does not clamp the day: 31 March minus one month is "31 February", which JavaScript rolls forward to **3 March** (2 March in a leap year). The 1M, 3M and 6M returns, momentum and every window start then begin a few days late. It does not matter for today's as-of date if that is the 1st to 28th, but it will on later month-end refreshes.

In [ ]:
periods = {"1M": 1, "3M": 3, "6M": 6, "1Y": 12, "3Y": 36, "5Y": 60}
rows = []
for t in range(REF.n):
    iso = REF.iso[t]
    if int(iso[8:]) < 29:
        continue
    for lab, mths in periods.items():
        a_js, a_cal = REF.on_or_before(REF.shift_months(iso, mths)), REF.on_or_before(REF_CAL.shift_months(iso, mths))
        if a_js != a_cal and a_cal >= 0:
            gaps = []
            for st in REF.stocks:
                if a_cal >= st.first and st.obs[t]:
                    gaps.append(abs(st.tri[t] / st.tri[a_js] - st.tri[t] / st.tri[a_cal]) * 100)
            rows.append({"as_of": iso, "period": lab, "dashboard start": REF.iso[a_js], "calendar start": REF.iso[a_cal], "trading days lost": a_js - a_cal,
                         "largest return gap across stocks (pp)": max(gaps) if gaps else np.nan, "median gap (pp)": float(np.median(gaps)) if gaps else np.nan})
MONTH_BUG = pd.DataFrame(rows)
today_hit = MONTH_BUG[MONTH_BUG.as_of == AS_OF]
if len(MONTH_BUG):
    summary = MONTH_BUG.groupby("period").agg(as_of_dates=("as_of", "nunique"), max_days_lost=("trading days lost", "max"),
                                              worst_gap_pp=("largest return gap across stocks (pp)", "max"), typical_gap_pp=("median gap (pp)", "median"))
    display(summary)
    display(MONTH_BUG.sort_values("largest return gap across stocks (pp)", ascending=False).head(10))
    finding("MEDIUM" if len(today_hit) == 0 else "HIGH", "Latent bug", "Returns and windows start a few days late when the as-of date is the 29th-31st",
            f"On {MONTH_BUG.as_of.nunique()} of the {REF.n} historical as-of dates the dashboard's 1M/3M/6M (and Feb-29 year) start dates differ from calendar arithmetic, "
            f"losing up to {int(MONTH_BUG['trading days lost'].max())} trading days; the largest resulting gap in a single stock's return is "
            f"{MONTH_BUG['largest return gap across stocks (pp)'].max():.1f}pp. {'Today IS affected.' if len(today_hit) else 'Today (' + AS_OF + ') is not affected.'} "
            "Fix: clamp the day in shiftYears() (set the date to 1, subtract months, then take min(day, days in month)).", evidence="model.js shiftYears()")
add_checks([{"item": f"today's {k} return unchanged under calendar month arithmetic", "stock": st.id, "dashboard": REF_STATS["local"][j]["ret"][hk],
             "reference": REF_CAL.stock_stats(st, "local")["ret"][hk], "fmt": pct1s}
            for j, st in enumerate(REF.stocks) for k, hk in (("1M", "m1"), ("3M", "m3"), ("6M", "m6"))], "Month arithmetic (today)", tol_abs=1e-12, layer="3. Presentation")

### 9.2 EUR returns on local holidays (FX move dropped)
In the EUR view, and in the EUR-based sector index behind the SECTOR factor and the peer comparisons, a stock's daily return on the day after its own market holiday is measured from the holiday row, whose EUR level already carries that day's FX move. The FX move on the holiday itself is never counted. The effect is small; the question is whether it changes anything on screen.

In [ ]:
stats_fx = [REF_FX.stock_stats(st, "EUR") for st in REF_FX.stocks]
rel_fx = [REF_FX.fit(st, REF_FX.resolve(PRESETS["relative"]), DV["freq"], DV["years"], DV["ccy"]) for st in REF_FX.stocks]
rel_dash = ref_fits("relative")
rows = []
for j, st in enumerate(REF.stocks):
    a, b = REF_STATS["EUR"][j], stats_fx[j]
    rows += [{"item": "volatility 1Y, EUR view", "stock": st.id, "dashboard": a["vol1y"], "reference": b["vol1y"], "fmt": pct0},
             {"item": "1-day VaR, EUR view", "stock": st.id, "dashboard": a["var95"], "reference": b["var95"], "fmt": pct1},
             {"item": "1Y relative to sub-sector", "stock": st.id, "dashboard": a["rel"]["y1"]["subsector"], "reference": b["rel"]["y1"]["subsector"], "fmt": ppf},
             {"item": "1Y relative to coverage", "stock": st.id, "dashboard": a["rel"]["y1"]["coverage"], "reference": b["rel"]["y1"]["coverage"], "fmt": ppf}]
    if rel_dash[j]["ok"] and rel_fx[j]["ok"] and "SECTOR" in rel_dash[j]["betas"]:
        rows.append({"item": "SECTOR beta (sector-relative model)", "stock": st.id, "dashboard": rel_dash[j]["betas"]["SECTOR"], "reference": rel_fx[j]["betas"]["SECTOR"], "fmt": num2})
FX_EFFECT = add_checks(rows, "FX on local holidays", tol_rel=1e-9, layer="3. Presentation")
changed = FX_EFFECT[FX_EFFECT.status == "FAIL"]
unseen = FX_EFFECT[FX_EFFECT.status == "WARN"]
_fx_units = {"volatility 1Y, EUR view": lambda v: f"{v * 100:.2f}pp", "1-day VaR, EUR view": lambda v: f"{v * 100:.2f}pp",
             "1Y relative to sub-sector": lambda v: f"{v * 100:.2f}pp", "1Y relative to coverage": lambda v: f"{v * 100:.2f}pp",
             "SECTOR beta (sector-relative model)": lambda v: f"{v:.3f}"}
FX_LARGEST = "; ".join(f"{item} {_fx_units[item](v)}" for item, v in FX_EFFECT.groupby("item", sort=False).abs_diff.max().items())
display(Markdown(f"Fixing the convention changes **{len(changed)}** displayed values and {len(unseen)} more below display precision. Largest change by item: {FX_LARGEST}."))
display(changed.head(20)[["item", "stock", "shown_dashboard", "shown_reference"]] if len(changed) else FX_EFFECT.nlargest(8, "rel_diff")[["item", "stock", "dashboard", "reference", "rel_diff"]])
finding("LOW" if len(changed) < 10 else "MEDIUM", "Methodology", "FX move on a stock's own market holiday is dropped from EUR daily returns",
        f"Affects the EUR view (volatility, VaR, relative returns) and the EUR sector index. Measured effect: {len(changed)} displayed values change; "
        f"largest change by item: {FX_LARGEST}. Fix: in model.js prepare() and stockStats(), divide by the level on the stock's previous trading day, not the previous row.",
        stocks=sorted(set(changed.stock)) if len(changed) else None)

### 9.3 Other conventions worth knowing before quoting numbers

In [ ]:
# (a) VaR/ES when the last year contains a gap longer than the forward-fill: the dashboard sorts an array with NaN in it.
nan_var = [st.id for j, st in enumerate(REF.stocks) if REF_STATS["local"][j]["daily_has_nan"]]
if nan_var:
    finding("MEDIUM", "Latent bug", "VaR and expected shortfall are unreliable after a long trading gap",
            "stockStats() pushes NaN daily returns into the list it sorts for the historical VaR (JavaScript sorts NaN unpredictably) and counts them in the sample size. "
            "Affected now: " + ", ".join(nan_var), stocks=nan_var)
# (b) LTV headroom semantics when the LTV proxy is already above 50% / 60%
bso = {s: balance_sheet_outlook(FUND[s]) for s in STOCK_IDS}
over50 = [s for s, b in bso.items() if b and b["headroom50"] > 0]
over60 = [s for s, b in bso.items() if b and b["headroom60"] > 0]
finding("HIGH" if over50 else "LOW", "Presentation", "'Value fall to 50% LTV' reads wrongly once the LTV proxy is above 50%",
        "For a stock whose LTV proxy is already above 50%, the headroom is the RISE in property values needed to get back to 50%, but it is presented as a fall "
        "in three places: the 'Value fall to 50% LTV' tile (a positive number under a 'fall' label), the balance-sheet card ('Values can fall −x% before the LTV proxy "
        "reaches 50%') and the valuation takeaway ('would reach 50% after a −x% fall'). "
        + (f"Affected today: 50% — {', '.join(over50)}; 60% — {', '.join(over60) or 'none'}." if over50 else "No stock is above 50% today, so nothing is wrong on screen yet."),
        stocks=over50 or None)
# (c) The stock-specific return index on the Overview is an in-sample OLS residual: it ends near 100 by construction.
ends_ = []
for j, (st, fit) in enumerate(zip(REF.stocks, ref_fits("attrib", hac=False))):
    if fit["ok"]:
        lvl = 100 * np.prod(1 + fit["resid"][1:])
        a3 = REF_ATTR.get(("sector", "3Y", st.id)) or {}
        ends_.append({"stock": st.id, "index level at the end of the window": lvl, "3Y stock-specific (Attribution tab)": a3.get("specific")})
RESID_INDEX = pd.DataFrame(ends_).set_index("stock")
finding("LOW", "Presentation", "The 'Stock-specific return' chart always ends near 100",
        f"It compounds in-sample regression residuals, which sum to zero by construction, so its end level is {RESID_INDEX.iloc[:, 0].min():.0f}-{RESID_INDEX.iloc[:, 0].max():.0f} "
        "for every stock whatever the stock did, and it excludes alpha. The 1M/3M/6M figures above it use out-of-sample exposures and include alpha, so the two can disagree. "
        "Label the chart 'path of residuals (in-sample)' or plot the attribution's specific return instead.")
# (d) Where the regression windows actually end, and why
end_rows = []
for model in ("full", "macro", "attrib"):
    fits_ = dv_fits(model)
    ends = pd.Series([f["end"] for f in fits_ if f["ok"]]).value_counts()
    end_rows.append({"model": model, "factors": ", ".join(REF.resolve(MODEL_SPECS[model])), "most common window end": ends.index[0] if len(ends) else None, "stocks": int(ends.iloc[0]) if len(ends) else 0})
WINDOW_ENDS = pd.DataFrame(end_rows).set_index("model")
display(WINDOW_ENDS)
for model, r in WINDOW_ENDS.iterrows():
    if r["most common window end"] and r["most common window end"] < AS_OF:
        finding("INFO", "Data", f"'{model}' regressions end on {r['most common window end']}, not {AS_OF}",
                "Rows are dropped once any factor in the model has no data (rates publish with a lag of a day or two; the Fama-French factors arrive one to two months late).")
# (e) Correlated-move scenarios depend on the window's equity-rates correlation
corr_rows = []
for yrs in (1, 3, 5):
    moves = []
    for st in REF.stocks:
        fit = REF.fit(st, MACRO_IDS, "W", yrs, DV["ccy"], hac=False, fast=True)
        if fit["ok"] and "RATES" in fit["factors"] and "MKT" in fit["factors"]:
            x = conditional_shocks(fit["factorCov"], {fit["factors"].index("RATES"): 50.0})
            moves.append(x[fit["factors"].index("MKT")])
    corr_rows.append({"window": f"{yrs}Y weekly", "implied equity move for +50bp (median across stocks)": float(np.median(moves)) if moves else np.nan})
SCEN_WINDOW = pd.DataFrame(corr_rows).set_index("window")
display(SCEN_WINDOW.style.format("{:+.2%}"))
moves_ = SCEN_WINDOW.iloc[:, 0].dropna()
finding("INFO", "Methodology", "'Correlated moves' scenarios inherit the window's equity-rates correlation",
        "With correlated moves on, a +50bp rate shock also moves equities by " + ", ".join(f"{v:+.1%} ({k})" for k, v in moves_.items())
        + (" (median stock). The sign differs between windows" if len(set(np.sign(moves_))) > 1 else " (median stock). The size differs between windows")
        + ", so the companion moves in these scenarios depend on the window chosen.")
# (f) Standing caveats from the documentation, recorded so they travel with the report
finding("INFO", "Methodology", "Sector index weights use today's share counts",
        "The SECTOR factor and peer indices weight stocks by current shares x historical price; issuance and buybacks since then are ignored (mild look-ahead in the weights).")
finding("INFO", "Methodology", "P/B and LTV are IFRS proxies, not EPRA NTA / EPRA LTV",
        "P/B uses IFRS book value, which unlike EPRA NTA includes deferred tax on revaluation gains, so it is not P/NAV; the LTV proxy is net debt over total assets less cash, "
        "not EPRA LTV. The methodology note says so; expect differences from company-reported EPRA figures.")
finding("INFO", "Methodology", "Consensus target is scaled by Yahoo's current price",
        "target_mean = last close x (Yahoo target / Yahoo currentPrice). When currentPrice differs from the close (intraday fetch, stale quote) the target shown differs from Yahoo's by the same ratio. "
        "Switch on RUN_ONLINE_CHECKS to measure it.")
if ENGINE:
    edge = GARCH_OPT[GARCH_OPT.at_search_edge | (GARCH_OPT.loglik_gap > 1.0)]
    if len(edge):
        finding("LOW", "Methodology", "GARCH grid search stops short of the likelihood maximum for some stocks",
                f"For {len(edge)} stocks the dashboard's grid ends at the edge of its search region or at least 1 log-likelihood point below the continuous optimum; "
                f"the largest effect on next-month volatility is {(GARCH_OPT.vol21_dashboard - GARCH_OPT.vol21_mle).abs().max() * 100:.1f} percentage points (annualised).", stocks=list(edge.index))
display(pd.DataFrame(FINDINGS).assign(order=lambda d: d.severity.map(SEVERITY_ORDER)).sort_values("order").drop(columns="order")[["severity", "area", "title", "stocks"]])

## 10. Layer 3 — screen audit: read the rendered dashboard (optional)
Everything so far checks numbers before `app.js` puts them on the page. This layer opens `docs/index.html` in headless Chromium with the default settings, selects every stock, visits every tab and reads each tile, table cell, figure and generated sentence. Each is compared with the text expected from the verified values, built here from the *meaning of the label* (for example: "Rate sensitivity" = macro-model rate beta × 10bp, as a signed % to two decimals) with the ported formatters. A mismatch means the page shows a different number, unit or wording than its label promises. It also scans every view for `NaN`, `undefined`, `Infinity` and JavaScript errors.

Needs `pip install playwright` and `playwright install chromium`. Takes a few minutes for all stocks.

In [ ]:
LABELS = {
    "MKT": ("European equity market", "Market", "+1%"), "SECTOR": ("Real estate sector (ex-market)", "RE sector", "+1%"),
    "RATES": ("Home 10Y government yield", "Home 10Y", "+10bp"), "EUR10Y": ("EUR 10Y yield", "EUR 10Y", "+10bp"), "EUR2Y": ("EUR 2Y yield", "EUR 2Y", "+10bp"),
    "CURVE": ("EUR 2s10s curve", "2s10s", "+10bp steeper"), "CREDIT": ("EUR high-yield spread", "HY spread", "+25bp"),
    "CREDIT_ETF": ("EUR high-yield bonds vs governments", "HY vs govts", "−1% HY excess return"), "FXL": ("Home currency vs EUR", "FX vs EUR", "+1% home ccy"),
    "EURUSD": ("EUR/USD", "EUR/USD", "+1% EUR"), "OIL": ("Brent crude oil", "Oil", "+10%"), "VOL": ("Equity volatility index", "Volatility", "+5 pts"),
    "SMB": ("Size (small minus big)", "Size", "+1%"), "HML": ("Value (high minus low B/M)", "Value", "+1%"), "RMW": ("Profitability (robust minus weak)", "Profitability", "+1%"),
    "CMA": ("Investment (conservative minus aggressive)", "Investment", "+1%"), "WML": ("Momentum (winners minus losers)", "Momentum", "+1%"),
    "E_VAL": ("Value ETF vs market", "Value (ETF)", "+1%"), "E_MOM": ("Momentum ETF vs market", "Momentum (ETF)", "+1%"), "E_QUAL": ("Quality ETF vs market", "Quality (ETF)", "+1%"),
    "E_MINV": ("Min volatility ETF vs market", "Min vol (ETF)", "+1%"), "E_SIZE": ("Mid-cap ETF vs market", "Size (ETF)", "+1%"),
}
SIGNAL_LABELS = {"mom": "Momentum (12-1M return)", "momrel": "Momentum vs sub-sector", "rev": "Last month's return", "lowvol": "Low volatility", "high": "Nearness to 52-week high"}
CHAR_LABELS = [("value", "Value (book-to-price)", "x", "pb"), ("yield", "Dividend yield", "pct", None), ("momentum", "Momentum (12-1M return)", "pct", None),
               ("size", "Size (log market cap)", "eur", "market_cap_eur"), ("volatility", "Volatility (1Y daily)", "pct", None), ("beta", "Market beta", "num", None),
               ("leverage", "Leverage (LTV proxy)", "pct", None), ("liquidity", "Liquidity (log traded value)", "eur", "adv_3m_eur")]
FREQ_WORD = {"D": "daily", "W": "weekly", "M": "monthly"}
WINDOW_WORD = f"{DV['years']}Y {FREQ_WORD[DV['freq']]}"
per_shock = lambda fid: FACTORS[fid].unit in ("bp", "pts")
exp_units = lambda fid, v: (v * FACTORS[fid].shock * 100 if per_shock(fid) else v) if is_num(v) else np.nan
exp_fmt = lambda fid, v, dp=2: DASH if not is_num(v) else (f"{js_num(v, dp, True)}%" if per_shock(fid) else js_num(v, dp))
past_exposure = lambda fid, x: f"{exp_fmt(fid, exp_units(fid, x['long']))} over 2 years" if is_num(x["long"]) else f"{exp_fmt(fid, exp_units(fid, x['short']))} over the last year"


def exposure_text(fid, beta):
    if fid not in FACTORS or not is_num(beta):
        return DASH
    f = FACTORS[fid]
    return f"{js_pct(beta * f.shock, 2, True)} per {LABELS[fid][2]}" if f.unit in ("bp", "pts") else js_num(beta, 2)


def med(vals):
    v = [float(x) for x in vals if is_num(x)]
    return float(np.median(v)) if v else np.nan


def rank_of(values, value, descending=True):
    v = [x for x in values if is_num(x)]
    if not is_num(value):
        return None, len(v)
    return 1 + sum(1 for x in v if (x > value if descending else x < value)), len(v)


def rate_label(st):
    return "EUR 10Y yield" if st.rate_series == "EUR10Y" else DATA["series"].get(st.rate_series, {}).get("label", "home 10Y yield")


def norm_cdf_as(x):
    """The dashboard's normal CDF (Abramowitz-Stegun 7.1.26), so probabilities round the same way."""
    sign = -1 if x < 0 else 1
    z = abs(x) / math.sqrt(2)
    t = 1 / (1 + 0.3275911 * z)
    y = 1 - ((((1.061405429 * t - 1.453152027) * t + 1.421413741) * t - 0.284496736) * t + 0.254829592) * t * math.exp(-z * z)
    return 0.5 * (1 + sign * y)


def expected_page(j, today):
    """Text the dashboard should show for stock j at the default settings, keyed by (view, card, row, column)."""
    st, sid, f = REF.stocks[j], STOCK_IDS[j], FUND[STOCK_IDS[j]]
    ccy, ss = st.currency, st.subsector
    peers = [p for p in REF.stocks if p.subsector == ss]
    others = [p for p in peers if p is not st]
    s = E["stats"][DV["ccy"]][j]
    capm, macro, full, style = (E["fits"][f"{DV_KEY}|{m}"] for m in ("capm", "macro", "full", "style"))
    X = {}
    # ---- header
    caps = [FUND[x].get("market_cap_eur") for x in STOCK_IDS]
    rank_cap, _ = rank_of(caps, f.get("market_cap_eur"))
    X[("header", "tile", "Market cap", "value")] = js_money_compact(f.get("market_cap_eur"), "EUR")
    X[("header", "tile", "Market cap", "sub")] = js_money_compact(f.get("market_cap"), ccy) if (f.get("market_cap") and ccy != "EUR") else f"{js_ordinal(rank_cap) if rank_cap else 'NaNth'} largest of {len(STOCK_IDS)}"
    X[("header", "tile", "P/B (P/NAV proxy)", "value")] = js_mult(f.get("pb"))
    X[("header", "tile", "P/B (P/NAV proxy)", "sub")] = f"sub-sector median {js_mult(med([FUND[p.id].get('pb') for p in others]))}"
    X[("header", "tile", "Dividend yield", "value")] = js_pct(f.get("dividend_yield"), 1)
    X[("header", "tile", "Dividend yield", "sub")] = f"sub-sector median {js_pct(med([FUND[p.id].get('dividend_yield') for p in others]), 1)}"
    X[("header", "tile", "1Y total return", "value")] = js_pct(s["ret"]["y1"], 1, True)
    X[("header", "tile", "1Y total return", "sub")] = f"{js_pp(s['rel']['y1']['subsector'])} vs sub-sector"
    X[("header", "tile", "Beta vs STOXX 600", "value")] = js_num(capm[j]["betas"]["MKT"], 2) if capm[j]["ok"] else DASH
    X[("header", "tile", "Beta vs STOXX 600", "sub")] = f"{WINDOW_WORD}, market only"
    rb = macro[j]["betas"].get("RATES") if macro[j]["ok"] else None
    X[("header", "tile", "Rate sensitivity", "value")] = js_pct(rb * 10, 2, True) if is_num(rb) else DASH
    X[("header", "tile", "Rate sensitivity", "sub")] = f"per +10bp {rate_label(st).replace(' yield', '')}, market held constant"
    X[("header", "tile", "Volatility (1Y)", "value")] = js_pct(s["vol1y"], 0)
    X[("header", "tile", "Volatility (1Y)", "sub")] = f"{js_pct(macro[j]['specificVolAnn'], 0)} stock-specific" if macro[j]["ok"] else ""
    cns = f.get("consensus") or {}
    X[("header", "tile", "Consensus upside", "value")] = js_pct(cns.get("upside"), 0, True)
    X[("header", "tile", "Consensus upside", "sub")] = (f"{cns['analysts']} analysts" + (f" · {cns['rating']}" if cns.get("rating") else "")) if cns.get("analysts") else "Yahoo Finance consensus"
    X[("header", "price", "", "")] = js_money(f["price"] if f.get("price") is not None else s["price"], ccy)
    X[("header", "priceSub", "", "")] = f"{js_pct(s['ret']['d1'], 1, True)} on the day · close {js_date(f.get('price_date') or s['date'])}"
    # ---- overview: returns table and technicals
    for lab, k in (("1W", "w1"), ("1M", "m1"), ("3M", "m3"), ("6M", "m6"), ("YTD", "ytd"), ("1Y", "y1"), ("3Y", "y3"), ("5Y", "y5")):
        for col, val in (("Return", js_pct(s["ret"][k], 1, True)), ("vs sub-sector", js_pp(s["rel"][k]["subsector"])), ("vs coverage", js_pp(s["rel"][k]["coverage"])), ("vs STOXX 600", js_pp(s["rel"][k]["market"]))):
            X[("overview", "Returns", lab, col)] = val
    X[("overview", "Returns", "52-week range", "kv")] = f"{js_money(f.get('low_52w'), ccy)} – {js_money(f.get('high_52w'), ccy)}"
    X[("overview", "Returns", "vs 50-day average", "kv")] = js_pct(s["price"] / s["sma50"] - 1, 1, True) if is_num(s["sma50"]) else DASH
    X[("overview", "Returns", "vs 200-day average", "kv")] = js_pct(s["price"] / s["sma200"] - 1, 1, True) if is_num(s["sma200"]) else DASH
    X[("overview", "Returns", "RSI (14 days)", "kv")] = js_num(s["rsi14"], 0)
    X[("overview", "Returns", "Momentum (12-1 months)", "kv")] = js_pct(s["mom12_1"], 1, True)
    # ---- overview: fundamentals
    card = "Valuation, balance sheet & consensus"
    X[("overview", card, "Market cap", "kv")] = js_money_compact(f.get("market_cap_eur"), "EUR") + (f" ({js_money_compact(f['market_cap'], ccy)})" if ccy != "EUR" and is_num(f.get("market_cap")) else "")
    X[("overview", card, "Book value per share", "kv")] = js_money(f.get("book_value_per_share"), ccy)
    X[("overview", card, "P/B", "kv")] = js_mult(f.get("pb"))
    X[("overview", card, "Dividend yield (trailing 12M)", "kv")] = js_pct(f.get("dividend_yield"), 1) + (f" · DPS {js_money(f['dps_ttm'], ccy)}" if is_num(f.get("dps_ttm")) else "")
    X[("overview", card, "Forward P/E (consensus EPS)", "kv")] = js_mult(f.get("pe_forward"), 1)
    X[("overview", card, "LTV proxy", "kv")] = js_pct(f.get("ltv"), 1)
    X[("overview", card, "Net debt / EBITDA (indicative)", "kv")] = js_mult(f.get("nd_ebitda"), 1)
    X[("overview", card, "Interest cover (indicative)", "kv")] = js_mult(f.get("interest_cover"), 1)
    X[("overview", card, "EV / EBITDA (indicative)", "kv")] = js_mult(f.get("ev_ebitda"), 1)
    X[("overview", card, "Revenue growth (last FY)", "kv")] = js_pct(f.get("revenue_growth"), 1, True)
    X[("overview", card, "Avg daily traded value (3M)", "kv")] = js_money_compact(f.get("adv_3m_eur"), "EUR")
    X[("overview", card, "Consensus target", "kv")] = f"{js_money(cns['target_mean'], ccy)} ({js_pct(cns.get('upside'), 0, True)})" if is_num(cns.get("target_mean")) else DASH
    X[("overview", card, "Target range", "kv")] = f"{js_money(cns['target_low'], ccy)} – {js_money(cns['target_high'], ccy)}" if is_num(cns.get("target_low")) and is_num(cns.get("target_high")) else DASH
    cnt = cns.get("counts")
    X[("overview", card, "Ratings", "kv")] = (f"{cnt['strongBuy'] + cnt['buy']} buy · {cnt['hold']} hold · {cnt['sell'] + cnt['strongSell']} sell" if cnt
                                            else f"{cns['rating']} ({cns.get('analysts') or '?'} analysts)" if cns.get("rating") else DASH)
    X[("overview", card, "Next results", "kv")] = js_date(f["next_results"]) if f.get("next_results") else f"last {js_date(f['last_results'])}" if f.get("last_results") else DASH
    if f.get("ex_dividend"):
        X[("overview", card, "Next ex-dividend date" if f["ex_dividend"] >= AS_OF else "Last ex-dividend date", "kv")] = js_date(f["ex_dividend"])
    # ---- overview: factor profile and specific-return figures
    fu = full[j]
    if fu["ok"]:
        X[("overview", "Factor profile", "stats", "R²")] = f"R² {js_num(fu['r2'], 2)}"
        X[("overview", "Factor profile", "stats", "Observations")] = f"Observations {fu['n']}"
        X[("overview", "Factor profile", "stats", "Specific volatility")] = f"Specific volatility {js_pct(fu['specificVolAnn'], 1)}"
        X[("overview", "Factor profile", "stats", "Window")] = f"Window {js_month_year(fu['start'])} – {js_month_year(fu['end'])}"
    for k in ("1M", "3M", "6M"):
        a = E["attribution"][f"sector|{k}"][j]
        if a["ok"] and a["lastFitOk"]:
            z = a["specific"] / (a["lastFitSpecificVolAnn"] * math.sqrt(a["days"] / 252))
            X[("overview", "Stock-specific return", k, "figure")] = f"{js_pct(a['specific'], 1, True)}|{k} specific · {js_num(z, 1) + ' σ' if is_num(z) else 'n/a'}"
    # ---- overview: key takeaways
    tk = []
    mf = macro
    if mf[j]["ok"] and is_num(mf[j]["betas"].get("RATES")):
        v = mf[j]["betas"]["RATES"] * 10
        oth = [mf[p.i]["betas"]["RATES"] * 10 for p in others if mf[p.i]["ok"] and is_num(mf[p.i]["betas"].get("RATES"))]
        less = sum(1 for x in oth if x > v)
        sig_txt = "" if abs(mf[j]["t"]["RATES"]) >= 1.96 else " The estimate is not statistically significant."
        if v < 0:
            tot = len(oth) + 1
            rel = ("" if not oth else f" That makes it the least rate-sensitive of the {tot} {ss} stocks." if less == 0
                   else f" That makes it the most rate-sensitive of the {tot} {ss} stocks." if less == len(oth) else f" That is more rate-sensitive than {less} of its {len(oth)} {ss} peers.")
            tk.append(("Rates", f"A +10bp move in the {rate_label(st)} has come with a {js_pct(v, 2, True)} move in the shares, holding the market constant (t {js_num(mf[j]['t']['RATES'], 1)}).{rel}{sig_txt}"))
        else:
            tk.append(("Rates", f"The shares have tended to rise with the {rate_label(st)} ({js_pct(v, 2, True)} per +10bp, t {js_num(mf[j]['t']['RATES'], 1)}), unusual for real estate.{sig_txt}"))
    if capm[j]["ok"]:
        m_ = med([capm[p.i]["betas"]["MKT"] if capm[p.i]["ok"] else np.nan for p in others])
        b = capm[j]["betas"]["MKT"]
        if is_num(m_) and abs(b - m_) >= 0.15:
            tk.append(("Beta", f"Market beta of {js_num(b, 2)} against a {ss} median of {js_num(m_, 2)}: {'a higher-beta way to play European equities' if b > m_ else 'more defensive than its peers'}."))
        else:
            tk.append(("Beta", f"Market beta of {js_num(b, 2)}, in line with the {ss} median of {js_num(m_, 2)}."))
    if mf[j]["ok"]:
        if "CREDIT" in mf[j]["factors"] and abs(mf[j]["t"]["CREDIT"]) >= 1.96:
            tk.append(("Credit", f"Credit matters: {js_pct(mf[j]['betas']['CREDIT'] * 25, 2, True)} per +25bp in the EUR high-yield spread (t {js_num(mf[j]['t']['CREDIT'], 1)})."))
        if "CREDIT_ETF" in mf[j]["factors"] and abs(mf[j]["t"]["CREDIT_ETF"]) >= 1.96:
            tk.append(("Credit", "Credit matters: when EUR high-yield bonds lag governments by 1% (spreads widening), the shares have moved "
                                 f"{js_pct(-mf[j]['betas']['CREDIT_ETF'] * 0.01, 2, True)}, market held constant (t {js_num(mf[j]['t']['CREDIT_ETF'], 1)})."))
    if style[j]["ok"]:
        stl = sorted([(fid, style[j]["t"][fid], style[j]["betas"][fid]) for fid in style[j]["factors"] if fid not in ("MKT", "SECTOR") and abs(style[j]["t"][fid]) >= 1.96], key=lambda x: -abs(x[1]))
        words = {"SMB": ("behaves like a small cap", "behaves like a large cap"), "HML": ("trades with value stocks", "trades against value, like a growth stock"),
                 "RMW": ("moves with profitable companies", "moves with less profitable companies"), "CMA": ("moves with conservative investors (low asset growth)", "moves with high-investment companies"),
                 "WML": ("rides momentum: it tends to rise when recent winners rally", "is a momentum laggard: it tends to fall when recent winners rally")}
        if stl:
            fid, tv, bv = stl[0]
            w = words.get(fid, ("has a positive loading", "has a negative loading"))
            tk.append(("Style", f"Beyond market and sector, the stock {w[0] if bv > 0 else w[1]} ({LABELS[fid][1].lower()} factor, t {js_num(tv, 1)})."))
        else:
            tk.append(("Style", "No style factor is significant once market and sector are accounted for."))
    a3 = E["attribution"]["sector|3M"][j]
    if a3["ok"] and a3["lastFitOk"]:
        z = a3["specific"] / (a3["lastFitSpecificVolAnn"] * math.sqrt(a3["days"] / 252))
        tk.append(("Specific", f"Over 3 months the shares returned {js_pct(a3['total'], 1, True)}; market, sector, rates and credit explain {js_pct(a3['total'] - a3['specific'], 1, True)} "
                               f"and the stock-specific part is {js_pct(a3['specific'], 1, True)} ({js_num(z, 1)} standard deviations)"
                               f"{', a large idiosyncratic move worth explaining to clients' if abs(z) >= 1.5 else ''}."))
    if is_num(f.get("pb")):
        pbs = [FUND[x].get("pb") for x in STOCK_IDS]
        cheaper = sum(1 for v in pbs if is_num(v) and v < f["pb"])
        n_pb = sum(1 for v in pbs if is_num(v))
        tk.append(("Valuation", f"P/B of {js_mult(f['pb'])} versus a {ss} median of {js_mult(med([FUND[p.id].get('pb') for p in others]))}; {cheaper} of {n_pb - 1} coverage stocks trade on a lower multiple."
                                + (f" Trailing dividend yield {js_pct(f['dividend_yield'], 1)}." if is_num(f.get("dividend_yield")) else "")))
    if is_num(f.get("ltv")):
        ltvs = [FUND[x].get("ltv") for x in STOCK_IDS]
        higher = sum(1 for v in ltvs if is_num(v) and v > f["ltv"])
        tk.append(("Leverage", f"LTV proxy of {js_pct(f['ltv'], 0)} (net debt over total assets less cash): {js_ordinal(higher + 1)} highest of {sum(1 for v in ltvs if is_num(v))} in coverage."))
    if is_num(cns.get("upside")):
        tk.append(("Consensus", f"{cns.get('analysts') or 'Several'} analysts on Yahoo Finance; the mean target of {js_money(cns.get('target_mean'), ccy)} implies {js_pct(cns['upside'], 0, True)}"
                                + (f", average rating {cns['rating'].lower()}" if cns.get("rating") else "") + "."))
    if is_num(s["price"]) and is_num(s["sma200"]):
        gap = s["price"] / s["sma200"] - 1
        rsi = s["rsi14"]
        rsi_txt = f" RSI(14) {js_num(rsi, 0)}{', overbought territory' if rsi >= 70 else ', oversold territory' if rsi <= 30 else ''}." if is_num(rsi) else ""
        tk.append(("Technicals", f"Trading {js_pct(abs(gap), 1)} {'above' if gap >= 0 else 'below'} its 200-day average.{rsi_txt}"))
    X[("overview", "Key takeaways", "items", "")] = tk
    # ---- exposures tab
    if fu["ok"]:
        X[("exposures", "Regression results", "stats", "Window")] = f"Window {js_date(fu['start'])} – {js_date(fu['end'])}"
        X[("exposures", "Regression results", "stats", "Observations")] = f"Observations {fu['n']}"
        X[("exposures", "Regression results", "stats", "R²")] = f"R² {js_num(fu['r2'], 2)}"
        X[("exposures", "Regression results", "stats", "Adjusted R²")] = f"Adjusted R² {js_num(fu['adjR2'], 2)}"
        X[("exposures", "Regression results", "stats", "Alpha (annualised)")] = f"Alpha (annualised) {js_pct(fu['alphaAnn'], 1, True)} (t {js_num(fu['alphaT'], 1)})"
        X[("exposures", "Regression results", "stats", "Specific volatility")] = f"Specific volatility {js_pct(fu['specificVolAnn'], 1)}"
        X[("exposures", "Regression results", "stats", "Newey-West lags")] = f"Newey-West lags {fu['lags']}"
        for k, fid in enumerate(fu["factors"]):
            vals = [full[p.i]["betas"].get(fid, np.nan) if full[p.i]["ok"] else np.nan for p in peers]
            r_, of = rank_of(vals, fu["betas"][fid])
            row = LABELS[fid][0]
            for col, val in (("Exposure", exposure_text(fid, fu["betas"][fid])), ("t-stat", js_num(fu["t"][fid], 2)), ("Sig.", js_sig(fu["t"][fid])),
                             ("1σ move", js_pct(fu["betas"][fid] * fu["factorStd"][fid], 2, True)),
                             (f"{ss} median", exposure_text(fid, med([full[p.i]["betas"].get(fid, np.nan) if full[p.i]["ok"] else np.nan for p in others]))),
                             ("Rank in sub-sector", f"{r_} of {of}" if r_ else DASH), ("Share of variance", js_pct(fu["risk"]["share"][k], 0))):
                X[("exposures", "Regression results", row, col)] = val
    # ---- macro tab
    sens = E["macroSens"][DV_KEY]
    for r in sens[j]:
        peer_vals = []
        for p in peers:
            x = next((q for q in sens[p.i] if q["id"] == r["id"]), None)
            peer_vals.append((x["partialImpact"] if x["partialImpact"] is not None else x["totalImpact"]) if x else np.nan)
        own = r["partialImpact"] if r["partialImpact"] is not None else r["totalImpact"]
        rk, of = rank_of(peer_vals, own, descending=False)
        row = LABELS[r["id"]][0]
        for col, val in (("Shock", "+1%" if r["id"] in ("MKT", "SECTOR") else LABELS[r["id"]][2]), ("Total effect", js_pct(r["totalImpact"], 2, True)), ("t", js_num(r["totalT"], 1)),
                         ("Market held constant", "·" if r["partialImpact"] is None else js_pct(r["partialImpact"], 2, True)),
                         ("t#2", "·" if r["partialT"] is None else js_num(r["partialT"], 1)), ("R² alone", js_num(r["r2"], 2)),
                         (f"{ss} median", js_pct(med([v for v, p in zip(peer_vals, peers) if p is not st]), 2, True)), ("Rank", f"{rk} of {of}" if rk else DASH)):
            X[("macro", "Macro sensitivities", row, col)] = val
    fit = macro[j]
    if fit["ok"]:
        shocks = shocks_for(fit, {"MKT": 0, "RATES": 50, "CREDIT": 0, "FXL": 0, "OIL": 0})
        res = Ref.scenario({**fit, "factorCov": np.array(fit["factorCov"])}, shocks, True)
        X[("macro", "Scenario builder", "figure", "1")] = f"{js_pct(res['total'], 1, True)}|{NAME[sid]}, model-implied"
        X[("macro", "Scenario builder", "figure", "2")] = f"±{js_pct(fit['specificVolAnn'] / math.sqrt(fit['ppy']), 1)}|typical {FREQ_WORD[DV['freq']]} stock-specific noise"
        for e_i, ep in enumerate(EPISODES):
            imp = E["episodes"][j][e_i]
            if imp is None:
                continue
            cr = imp["moves"].get("CREDIT") if imp["moves"].get("CREDIT") is not None else imp["moves"].get("CREDIT_ETF")
            for col, val in (("Dates", f"{js_date(imp['startDate'])} – {js_date(imp['endDate'])}"), ("Market", js_pct(imp["moves"].get("MKT"), 1, True)),
                             ("Home 10Y", js_bp(imp["moves"].get("RATES"), 0)), ("Credit", js_bp(cr, 0) if imp["moves"].get("CREDIT") is not None else js_pct(cr, 1, True)),
                             ("Implied today", js_pct(imp["implied"], 1, True)), ("Actual then", js_pct(imp["actual"], 1, True)), ("Peer median then", js_pct(imp["peerActual"], 1, True))):
                X[("macro", "Historical stress episodes", ep["label"], col)] = val
    # ---- outlook tab
    fcl = E["forecast"][DV["ccy"]]
    mine = fcl["stocks"][j] if fcl["ok"] else {}
    g = E["garch"][j]
    bs = E["balance"][j]
    price = float(st.px[st.last]) if np.isfinite(st.px[st.last]) else f.get("price")
    m, r = mine.get("MKT"), mine.get("RATES")
    X[("outlook", "tile", "Market beta, next 12M", "value")] = js_num(m["forecast"], 2) if m else DASH
    X[("outlook", "tile", "Market beta, next 12M", "sub")] = f"{past_exposure('MKT', m)}; peers {js_num(m['peers'], 2)}" if m else "not enough history"
    X[("outlook", "tile", "Rate sensitivity, next 12M", "value")] = exp_fmt("RATES", exp_units("RATES", r["forecast"])) if r else DASH
    X[("outlook", "tile", "Rate sensitivity, next 12M", "sub")] = f"per +10bp; {past_exposure('RATES', r)}" if r else "not enough history"
    vol1y_local = E["stats"]["local"][j]["vol1y"]
    X[("outlook", "tile", "Volatility, next month", "value")] = js_pct(g["vol21"], 0) if g else DASH
    X[("outlook", "tile", "Volatility, next month", "sub")] = f"long-run {js_pct(g['lrVol'], 0)}; last year {js_pct(vol1y_local, 0)}" if g else "not enough history"
    dp = 0 if price >= 100 else 1 if price >= 5 else 2
    yr = g["range"]["252"] if g else None
    X[("outlook", "tile", "Price in 12 months", "value")] = f"{js_num(yr['lo'], dp)}–{js_num(yr['hi'], dp)}" if yr else DASH
    X[("outlook", "tile", "Price in 12 months", "sub")] = f"{ccy}, two in three outcomes" if yr else ""
    X[("outlook", "tile", "Consensus target", "value")] = f"{js_num(g['zTarget'], 1, True)}σ" if g and is_num(g["zTarget"]) else DASH
    X[("outlook", "tile", "Consensus target", "sub")] = f"{js_money(cns['target_mean'], ccy)} mean, {js_pct(cns.get('upside'), 0, True)}" if is_num(cns.get("target_mean")) else "no consensus on Yahoo Finance"
    X[("outlook", "tile", "Implied property values", "value")] = js_pct(bs["implied"], 0, True) if bs else DASH
    X[("outlook", "tile", "Implied property values", "sub")] = f"vs book, at P/B {js_mult(f.get('pb'))}" if bs else "accounts incomplete"
    X[("outlook", "tile", "Value fall to 50% LTV", "value")] = (js_pct(bs["headroom50"], 0) if is_num(bs["headroom50"]) else "Net cash") if bs else DASH
    X[("outlook", "tile", "Value fall to 50% LTV", "sub")] = f"LTV proxy {js_pct(bs['ltvNow'], 0)} today" if bs else ""
    days_between = lambda a, b: round((pd.Timestamp(b) - pd.Timestamp(a)).days)
    if f.get("next_results") and f["next_results"] >= today:
        X[("outlook", "tile", "Next results", "value")] = js_date(f["next_results"], False)
        X[("outlook", "tile", "Next results", "sub")] = f"{f['next_results'][:4]}, in {days_between(today, f['next_results'])} days"
    else:
        X[("outlook", "tile", "Last results", "value")] = js_date(f["last_results"], False) if f.get("last_results") else DASH
        X[("outlook", "tile", "Last results", "sub")] = f"{f['last_results'][:4]}; next date not published" if f.get("last_results") else "no dates on Yahoo Finance"
    for fid, x in mine.items():
        row = f"{LABELS[fid][1]} ({LABELS[fid][2]})" if per_shock(fid) else LABELS[fid][1]
        for col, val in (("2 years", exp_fmt(fid, exp_units(fid, x["long"]))), ("Last year", exp_fmt(fid, exp_units(fid, x["short"]))), ("Peers", exp_fmt(fid, exp_units(fid, x["peers"]))),
                         ("Next 12M", exp_fmt(fid, exp_units(fid, x["forecast"])) + ("*" if x["basis"] == "short" else "")),
                         ("Likely range", f"{exp_fmt(fid, exp_units(fid, x['lo']))} to {exp_fmt(fid, exp_units(fid, x['hi']))}")):
            X[("outlook", "Where exposures are heading", row, col)] = val
    if g:
        rng_ = lambda h: f"{js_num(g['range'][h]['lo'], dp)}–{js_num(g['range'][h]['hi'], dp)}"
        X[("outlook", "Price range ahead", "stats", "1 month")] = f"1 month {rng_('21')}"
        X[("outlook", "Price range ahead", "stats", "3 months")] = f"3 months {rng_('63')}"
        X[("outlook", "Price range ahead", "stats", "12 months")] = f"12 months {rng_('252')}"
        if is_num(cns.get("target_low")) and is_num(cns.get("target_high")):
            X[("outlook", "Price range ahead", "stats", "Target range")] = f"Target range {js_num(cns['target_low'], dp)}–{js_num(cns['target_high'], dp)}"
        for lab_, v in (("Next month", js_pct(g["vol21"], 0)), ("Next 3 months", js_pct(g["vol63"], 0)), ("Next year", js_pct(g["vol252"], 0)), ("Long-run", js_pct(g["lrVol"], 0)),
                        ("Persistence", js_num(g["persistence"], 3))):
            X[("outlook", "Volatility ahead", "stats", lab_)] = f"{lab_} {v}"
    if bs:
        for row in sorted(bs["rows"] + [{**bs["impliedRow"], "implied": True}], key=lambda r_: r_["change"]):
            lab_ = f"{js_pct(row['change'], 1, True)} priced in" if row.get("implied") else "Book" if row["change"] == 0 else js_pct(row["change"], 0, True)
            for col, val in (("NAV/share", js_money(row["nav"], ccy)), ("NAV change", js_pct(row["navChange"], 0, True)), ("P/NAV", js_mult(row["pnav"])), ("LTV", js_pct(row["ltv"], 0))):
                X[("outlook", "What the share price implies", lab_, col)] = val
    sigs = {s_["id"]: s_ for s_ in E["signals"]["signals"]}
    for s_id, sg in sigs.items():
        p = sg["percentile"][j]
        good = (p if sg["ic"] > 0 else 1 - p) if is_num(p) else np.nan
        reliable = is_num(sg["t"]) and abs(sg["t"]) >= 2
        reading = DASH if not is_num(p) else "No reliable record" if not reliable else "Tailwind" if good >= 2 / 3 else "Headwind" if good <= 1 / 3 else "Neutral"
        for col, val in (("Today", f"{js_ordinal(js_round(p * 100))} pct." if is_num(p) else DASH), ("Rank IC", js_num(sg["ic"], 2, True)), ("t", js_num(sg["t"], 1)), ("Reading", reading)):
            X[("outlook", "Price signals and their record", SIGNAL_LABELS[s_id], col)] = val
    wi = []
    if r:
        a_, b_, p_ = (exp_units("RATES", r[k]) for k in ("long", "short", "forecast"))
        err = exp_units("RATES", fcl["weights"]["RATES"]["rmse"])
        trend = (" The last year has been more rate-sensitive than the two-year average, so the exposure is building." if abs(b_) > abs(a_) + err / 2
                 else " The last year has been less rate-sensitive than the two-year average, so the exposure is fading." if abs(b_) < abs(a_) - err / 2 else "")
        wi.append(("Rates", f"Expect about {exp_fmt('RATES', p_)} per +10bp in the {rate_label(st)} over the next 12 months (likely {exp_fmt('RATES', exp_units('RATES', r['lo']))} "
                            f"to {exp_fmt('RATES', exp_units('RATES', r['hi']))}), against {past_exposure('RATES', r)}.{trend}"))
    if m:
        how = "move more than" if m["forecast"] > 1.1 else "move less than" if m["forecast"] < 0.9 else "move roughly in line with"
        past = ", ".join(x for x in [f"two years {js_num(m['long'], 2)}" if is_num(m["long"]) else None, f"last year {js_num(m['short'], 2)}" if is_num(m["short"]) else None] if x)
        wi.append(("Market", f"Forecast market beta {js_num(m['forecast'], 2)} ({past}): the shares should {how} European equities; a 10% market fall implies about {js_pct(-0.1 * m['forecast'], 1)} from the market alone."))
    if g:
        ratio = g["vol21"] / g["lrVol"]
        lr_t = js_pct(g["lrVol"], 0)
        regime = (f"above its long-run level of {lr_t}, and the model expects it to ease" if ratio > 1.15 else f"below its long-run level of {lr_t}, and the model expects it to drift back up" if ratio < 0.87
                  else f"close to its long-run level of {lr_t}")
        fade = f" Shocks fade with a half-life of about {js_num(g['halfLife'], 0)} trading days." if is_num(g["halfLife"]) else ""
        wi.append(("Volatility", f"Expected volatility is {js_pct(g['vol21'], 0)} for the next month, {regime}.{fade}"))
    if g and is_num(cns.get("target_mean")) and is_num(price):
        z = g["zTarget"]
        wi.append(("Consensus", f"The mean target of {js_money(cns['target_mean'], ccy)} is {js_num(abs(z), 1)} standard deviations of a year's move {'above' if z >= 0 else 'below'} the price. "
                                f"With the forecast volatility and no drift, the shares would end a year above it about {js_pct(1 - norm_cdf_as(z), 0)} of the time."))
    if bs:
        d10 = next((x for x in bs["rows"] if abs(x["change"] + 0.1) < 1e-9), None)
        reach = f" and would reach 50% after a {js_pct(-bs['headroom50'], 0)} fall" if is_num(bs["headroom50"]) else ""
        wi.append(("Valuation", f"The share price implies property values {js_pct(abs(bs['implied']), 0)} {'above' if bs['implied'] >= 0 else 'below'} book. If values fell 10% from book, "
                                f"the LTV proxy would rise from {js_pct(bs['ltvNow'], 0)} to {js_pct(d10['ltv'], 0) if d10 else DASH}{reach}."))
    strong = [sg for sg in sigs.values() if is_num(sg["t"]) and abs(sg["t"]) >= 2 and is_num(sg["percentile"][j])]
    if not strong:
        wi.append(("Signals", f"No price signal has a reliable record in this coverage since {E['signals']['first'][:4]} (all |t| below 2), so recent price action says little about next month."))
    for sg in strong[:2]:
        p = sg["percentile"][j]
        good = p if sg["ic"] > 0 else 1 - p
        reading = "a tailwind" if good >= 2 / 3 else "a headwind" if good <= 1 / 3 else "neutral"
        wi.append(("Signals", f"{SIGNAL_LABELS[sg['id']]} has {'pointed the right way' if sg['ic'] > 0 else 'worked in reverse'} for next-month returns in this coverage "
                              f"(rank IC {js_num(sg['ic'], 2, True)}, t {js_num(sg['t'], 1)}). {NAME[sid]} is in the {js_ordinal(js_round(p * 100))} percentile, which reads as {reading}."))
    if f.get("next_results") and f["next_results"] >= today:
        wi.append(("Calendar", f"Results due {js_date(f['next_results'])}, in {days_between(today, f['next_results'])} days."))
    elif f.get("last_results"):
        wi.append(("Calendar", f"Last results {js_date(f['last_results'])}; the next date is not on Yahoo Finance yet."))
    X[("outlook", "What to watch", "items", "")] = wi
    # ---- attribution tab (default 3M, market/sector/macro factors)
    a3 = E["attribution"]["sector|3M"][j]
    if a3["ok"]:
        X[("attribution", "tile", "Total return", "value")] = js_pct(a3["total"], 1, True)
        X[("attribution", "tile", "Total return", "sub")] = f"{js_date(a3['start'])} – {js_date(a3['end'])}"
        X[("attribution", "tile", "Explained by factors", "value")] = js_pct(a3["total"] - a3["specific"], 1, True)
        X[("attribution", "tile", "Explained by factors", "sub")] = f"{len(a3['factors'])} factors"
        X[("attribution", "tile", "Stock-specific", "value")] = js_pct(a3["specific"], 1, True)
        X[("attribution", "tile", "Stock-specific", "sub")] = (f"{js_num(a3['specific'] / (a3['lastFitSpecificVolAnn'] * math.sqrt(a3['days'] / 252)), 1)} σ" if a3["lastFitOk"] else "")
        for fid in a3["factors"]:
            v, to = a3["moves"][fid], a3["movesTo"][fid]
            txt = js_bp(v, 0) if FACTORS[fid].unit == "bp" else js_num(v, 1, True) + " pts" if FACTORS[fid].unit == "pts" else js_pct(v, 1, True)
            for col, val in (("Factor move", f"{txt} (to {js_date(to, False)})" if to and to < a3["end"] else txt), ("Avg exposure", exposure_text(fid, a3["avgBeta"][fid])),
                             ("Contribution", js_pct(a3["contributions"][fid], 2, True))):
                X[("attribution", "Contribution by factor", LABELS[fid][1], col)] = val
    # ---- risk tab
    if fu["ok"]:
        for lab_, v, sub in (("Volatility (window)", js_pct(fu["volAnn"], 1), f"{WINDOW_WORD}, annualised"), ("Systematic", js_pct(fu["systematicVolAnn"], 1), "from factor exposures"),
                             ("Stock-specific", js_pct(fu["specificVolAnn"], 1), f"{js_pct(fu['risk']['specificShare'], 0)} of variance"), ("R²", js_num(fu["r2"], 2), "share explained by factors")):
            X[("risk", "tile", lab_, "value")], X[("risk", "tile", lab_, "sub")] = v, sub
    for lab_, v, sub in (("Max drawdown (1Y)", js_pct(s["maxDD1y"], 1), "from the 1-year peak"), ("1-day VaR 95%", js_pct(s["var95"], 1), "historical, last year"),
                         ("1-day expected shortfall", js_pct(s["es95"], 1), "average of the worst 5%"),
                         ("Beta vs STOXX 600", js_num(capm[j]["betas"]["MKT"], 2) if capm[j]["ok"] else DASH, f"{WINDOW_WORD}, market only")):
        X[("risk", "tile", lab_, "value")], X[("risk", "tile", lab_, "sub")] = v, sub
    crow = [r_ for r_ in E["corr"][j] if r_["id"] != sid and is_num(r_["ret"])]
    crow = sorted(crow, key=lambda r_: -r_["ret"])[:12]
    for r_ in crow:
        k = STOCK_IDS.index(r_["id"])
        pr_ = E["pairs"][j][k]
        for col, val in (("Sub-sector", REF.stocks[k].subsector), ("Return corr.", js_num(r_["ret"], 2)), ("Specific corr.", js_num(r_["resid"], 2)),
                         ("Spread z (1Y)", js_num(pr_["z"] if pr_ else np.nan, 1, True)), ("Relative 3M", js_pct(pr_["change3m"] if pr_ else np.nan, 1, True))):
            X[("risk", "Closest peers and pair spreads", NAME[r_["id"]], col)] = val
    # ---- peers tab
    chars = {d["id"]: d for d in E["chars"]}
    for cid, lab_, fm, show_key in CHAR_LABELS:
        d = chars[cid]
        src = FUND[sid].get(show_key) if show_key else d["raw"][j]
        txt = DASH if not is_num(src) else js_pct(src, 1) if fm == "pct" else js_mult(src) if fm == "x" else js_money_compact(src, "EUR") if fm == "eur" else js_num(src, 2)
        rk, of = rank_of([d["raw"][p.i] for p in peers], d["raw"][j])
        for col, val in (("Value", txt), ("z vs coverage", js_num(d["z"][j], 2, True)), ("Rank in sub-sector (1 = highest)", f"{rk} of {of}" if rk else DASH)):
            X[("peers", "Style characteristics", lab_, col)] = val
    s_all = E["stats"][DV["ccy"]]
    for p in sorted(peers, key=lambda p: -(FUND[p.id].get("market_cap_eur") or -np.inf)):
        fp = FUND[p.id]
        c_ = fp.get("consensus") or {}
        rate = macro[p.i]["betas"]["RATES"] * 10 if macro[p.i]["ok"] and is_num(macro[p.i]["betas"].get("RATES")) else np.nan
        for col, val in (("Mkt cap", js_money_compact(fp.get("market_cap_eur"), "EUR")), ("P/B", js_mult(fp.get("pb"))), ("Div. yield", js_pct(fp.get("dividend_yield"), 1)),
                         ("LTV proxy", js_pct(fp.get("ltv"), 0)), ("Fwd P/E", js_mult(fp.get("pe_forward"), 1)), ("Cons. upside", js_pct(c_.get("upside"), 0, True)),
                         ("3M", js_pct(s_all[p.i]["ret"]["m3"], 1, True)), ("1Y", js_pct(s_all[p.i]["ret"]["y1"], 1, True)),
                         ("Beta", js_num(capm[p.i]["betas"]["MKT"], 2) if capm[p.i]["ok"] else DASH), ("Per +10bp", js_pct(rate, 2, True))):
            X[("peers", f"{ss} peer table", NAME[p.id], col)] = val
    return X


def expected_coverage():
    X = {}
    s_all = E["stats"][DV["ccy"]]
    capm, macro = E["fits"][f"{DV_KEY}|capm"], E["fits"][f"{DV_KEY}|macro"]
    rows = []
    for ss in REF.subsectors:
        for p in REF.stocks:
            if p.subsector != ss:
                continue
            fp, s = FUND[p.id], s_all[p.i]
            c_ = fp.get("consensus") or {}
            rate = macro[p.i]["betas"]["RATES"] * 10 if macro[p.i]["ok"] and is_num(macro[p.i]["betas"].get("RATES")) else np.nan
            r2 = macro[p.i]["r2"] if macro[p.i]["ok"] else np.nan
            beta = capm[p.i]["betas"]["MKT"] if capm[p.i]["ok"] else np.nan
            rows.append({"subsector": ss, "pb": fp.get("pb"), "dy": fp.get("dividend_yield"), "ltv": fp.get("ltv"), "r3m": s["ret"]["m3"], "r1y": s["ret"]["y1"], "beta": beta, "rate": rate, "vol": s["vol1y"]})
            for col, val in (("Ticker", p.rec["ticker"]), ("Mkt cap", js_money_compact(fp.get("market_cap_eur"), "EUR")), ("P/B", js_mult(fp.get("pb"))), ("Div. yield", js_pct(fp.get("dividend_yield"), 1)),
                             ("LTV", js_pct(fp.get("ltv"), 0)), ("Upside", js_pct(c_.get("upside"), 0, True)), ("1M", js_pct(s["ret"]["m1"], 1, True)), ("3M", js_pct(s["ret"]["m3"], 1, True)),
                             ("YTD", js_pct(s["ret"]["ytd"], 1, True)), ("1Y", js_pct(s["ret"]["y1"], 1, True)), ("Vol 1Y", js_pct(s["vol1y"], 0)), ("Beta", js_num(beta, 2)),
                             ("Per +10bp", js_pct(rate, 2, True)), ("R²", js_num(r2, 2))):
                X[("coverage", "Coverage table", NAME[p.id], col)] = val
            X[("rail", "rail", p.id, "1M")] = js_pct(s["ret"]["m1"], 1, True)
    df = pd.DataFrame(rows)
    for ss, g in df.groupby("subsector", sort=False):
        m_ = lambda k: med(g[k].tolist())
        for col, val in (("Stocks", str(len(g))), ("P/B", js_mult(m_("pb"))), ("Div. yield", js_pct(m_("dy"), 1)), ("LTV", js_pct(m_("ltv"), 0)), ("3M", js_pct(m_("r3m"), 1, True)),
                         ("1Y", js_pct(m_("r1y"), 1, True)), ("Beta", js_num(m_("beta"), 2)), ("Per +10bp", js_pct(m_("rate"), 2, True)), ("Vol 1Y", js_pct(m_("vol"), 0))):
            X[("coverage", "Sub-sector averages", ss, col)] = val
    return X

In [ ]:
SCRAPE_JS = r"""
() => {
  const txt = (el) => (el ? el.textContent.replace(/\s+/g, " ").trim() : null);
  const tiles = (root) => (root ? [...root.querySelectorAll(".tile")].map((t) => ({ label: txt(t.querySelector(".tile-label")), value: txt(t.querySelector(".tile-value")), sub: txt(t.querySelector(".tile-sub")) })) : []);
  const tables = (root) => [...root.querySelectorAll("table.table")].map((tb) => ({
    head: [...tb.querySelectorAll("thead th")].map((th) => txt(th).replace(/ [↑↓]$/, "")),
    rows: [...tb.querySelectorAll("tbody tr")].filter((tr) => !tr.classList.contains("group-row")).map((tr) => [...tr.querySelectorAll("td")].map(txt)),
  }));
  const cards = [...document.querySelectorAll("#panel section.card")].map((c) => ({
    title: txt(c.querySelector(".card-title")),
    tables: tables(c),
    kv: [...c.querySelectorAll("dl.kv")].flatMap((dl) => { const dt = [...dl.querySelectorAll("dt")]; const dd = [...dl.querySelectorAll("dd")]; return dt.map((d, i) => [txt(d), txt(dd[i])]); }),
    stats: [...c.querySelectorAll(".stats-line")].flatMap((s) => [...s.children].map(txt)),
    figures: [...c.querySelectorAll(".figure")].map((f) => ({ v: txt(f.querySelector(".v")), l: txt(f.querySelector(".l")) })),
    items: [...c.querySelectorAll("ul.takeaways li")].map((li) => ({ tag: txt(li.querySelector(".tag")), text: txt(li.children[1]) })),
  }));
  return {
    today: new Date().toISOString().slice(0, 10),
    name: txt(document.querySelector(".stock-name")), price: txt(document.querySelector(".price")), priceSub: txt(document.querySelector(".price-sub")),
    headTiles: tiles(document.querySelector(".stock-head")), panelTiles: tiles(document.getElementById("panel")), cards,
    text: document.getElementById("app").innerText,
    rail: [...document.querySelectorAll(".rail-item")].map((b) => [b.dataset.id, txt(b.querySelector(".rt"))]),
  };
}
"""
SCREEN_TABS = ["overview", "exposures", "macro", "outlook", "attribution", "risk", "peers"]


# The browser runs in its own Python process: Playwright's synchronous API cannot run inside Jupyter's event loop,
# and on Windows the kernel's event-loop policy cannot start the browser from a thread either.
SCRAPER_PY = r"""
import json, os, sys
from pathlib import Path
from playwright.sync_api import sync_playwright

job = json.loads(Path(sys.argv[1]).read_text(encoding="utf-8"))
views, errors = [], []
with sync_playwright() as pw:
    try:
        browser = pw.chromium.launch()
    except Exception:
        exe = Path(os.environ.get("PLAYWRIGHT_BROWSERS_PATH", "/opt/pw-browsers")) / "chromium"
        browser = pw.chromium.launch(executable_path=str(exe)) if exe.exists() else pw.chromium.launch(channel="chrome")
    page = browser.new_page(viewport={"width": 1480, "height": 1000})
    page.on("pageerror", lambda e: errors.append(["pageerror", str(e)]))
    page.on("console", lambda m: errors.append(["console", m.text]) if m.type == "error" else None)
    page.goto(job["url"])
    page.evaluate("() => localStorage.clear()")
    page.goto(job["url"] + "#" + job["stocks"][0])
    page.reload()
    page.wait_for_selector(".stock-name")
    for sid in job["stocks"]:
        page.evaluate("id => { location.hash = '#' + id; }", sid)
        page.wait_for_function("n => document.querySelector('.stock-name') && document.querySelector('.stock-name').textContent === n", arg=job["names"][sid])
        for tab in job["tabs"]:
            page.click(f"#tab-{tab}")
            page.wait_for_function("t => document.getElementById('tab-' + t).getAttribute('aria-selected') === 'true'", arg=tab)
            page.wait_for_timeout(60)  # let requestAnimationFrame fill the scenario figures
            views.append([sid, tab, page.evaluate(job["scrape_js"])])
    page.click("#tab-coverage")
    page.wait_for_timeout(60)
    views.append(["*", "coverage", page.evaluate(job["scrape_js"])])
    browser.close()
Path(job["out"]).write_text(json.dumps({"views": views, "errors": errors}), encoding="utf-8")
"""


def scrape_dashboard(stocks, timeout=1800):
    with tempfile.TemporaryDirectory() as tmp:
        tmp = Path(tmp)
        (tmp / "scraper.py").write_text(SCRAPER_PY, encoding="utf-8")
        job = {"url": (DOCS / "index.html").as_uri(), "stocks": list(stocks), "names": {s: NAME[s] for s in stocks}, "tabs": SCREEN_TABS,
               "scrape_js": SCRAPE_JS, "out": str(tmp / "screen.json")}
        (tmp / "job.json").write_text(json.dumps(job), encoding="utf-8")
        proc = subprocess.run([sys.executable, str(tmp / "scraper.py"), str(tmp / "job.json")], capture_output=True, text=True, timeout=timeout)
        if proc.returncode != 0:
            lines = (proc.stderr or proc.stdout or "").strip().splitlines()
            raise RuntimeError(next((ln for ln in reversed(lines) if "Error" in ln), lines[-1] if lines else f"exit code {proc.returncode}"))
        res = json.loads((tmp / "screen.json").read_text(encoding="utf-8"))
    return {(sid, tab): v for sid, tab, v in res["views"]}, [tuple(e) for e in res["errors"]]


def observed_page(shots, sid):
    """Flatten the scraped views of one stock into the same keys as expected_page()."""
    O = {}
    head = shots[(sid, "overview")]
    for t in head["headTiles"]:
        O[("header", "tile", t["label"], "value")], O[("header", "tile", t["label"], "sub")] = t["value"], t["sub"]
    O[("header", "price", "", "")], O[("header", "priceSub", "", "")] = head["price"], head["priceSub"]
    for tab in SCREEN_TABS:
        v = shots[(sid, tab)]
        if tab in ("outlook", "attribution", "risk"):
            for t in v["panelTiles"]:
                O[(tab, "tile", t["label"], "value")], O[(tab, "tile", t["label"], "sub")] = t["value"], t["sub"]
        for c in v["cards"]:
            title = c["title"]
            for tb in c["tables"]:
                seen = {}
                head_ = []
                for h in tb["head"]:
                    seen[h] = seen.get(h, 0) + 1
                    head_.append(h if seen[h] == 1 else f"{h}#{seen[h]}")
                for row in tb["rows"]:
                    for h, cell in zip(head_[1:], row[1:]):
                        O[(tab, title, row[0], h)] = cell
            for k, val in c["kv"]:
                O[(tab, title, k, "kv")] = val
            for sline in c["stats"]:
                for key in ("R²", "Observations", "Specific volatility", "Window", "Adjusted R²", "Alpha (annualised)", "Newey-West lags", "1 month", "3 months",
                            "12 months", "Target range", "Next month", "Next 3 months", "Next year", "Long-run", "Persistence"):
                    if sline.startswith(key + " "):
                        O[(tab, title, "stats", key)] = sline
            for n_, fg in enumerate(c["figures"], 1):
                if title == "Stock-specific return":
                    O[(tab, title, fg["l"].split(" ")[0], "figure")] = f"{fg['v']}|{fg['l']}"
                else:
                    O[(tab, title, "figure", str(n_))] = f"{fg['v']}|{fg['l']}"
            if c["items"]:
                O[(tab, title, "items", "")] = [(it["tag"], it["text"]) for it in c["items"]]
    return O

In [ ]:
import importlib.util

SCREEN = None
_want_screen = RUN_SCREEN_AUDIT is True or (RUN_SCREEN_AUDIT == "auto" and importlib.util.find_spec("playwright") is not None)
if _want_screen and not ENGINE:
    print("The screen audit compares the page with the engine's verified values; set RUN_ENGINE = True.")
elif _want_screen:
    t0 = time.time()
    screen_stocks = SCREEN_AUDIT_STOCKS or STOCK_IDS
    try:
        SCREEN, SCREEN_ERRORS = scrape_dashboard(screen_stocks)
    except Exception as exc:  # most often: the Chromium browser is not installed yet
        SCREEN = None
        print(f"Screen audit did not complete ({type(exc).__name__}: {str(exc)[:300]}). If the browser is missing, run `playwright install chromium` and re-run this cell.")
if SCREEN:
    today = next(iter(SCREEN.values()))["today"]
    rows = []
    for sid in screen_stocks:
        j = STOCK_IDS.index(sid)
        X, O = expected_page(j, today), observed_page(SCREEN, sid)
        for key, exp in X.items():
            got = O.get(key)
            if isinstance(exp, list):
                got = got or []
                for n_, (tag, text) in enumerate(exp):
                    obs = next((t for tg, t in got if tg == tag and t == text), None) or (got[n_][1] if n_ < len(got) and got[n_][0] == tag else None)
                    rows.append({"item": f"{key[0]} › {key[1]} › {tag}", "stock": sid, "passed": obs == text, "dashboard": obs, "reference": text})
                if len(got) != len(exp):
                    rows.append({"item": f"{key[0]} › {key[1]} › number of items", "stock": sid, "passed": False, "dashboard": [t for t, _ in got], "reference": [t for t, _ in exp]})
                continue
            rows.append({"item": " › ".join(k for k in key if k), "stock": sid, "passed": got == exp, "dashboard": got, "reference": exp})
        for tab in SCREEN_TABS:
            bad_tokens = re.findall(r"\b(NaN|undefined|Infinity|null)\b", SCREEN[(sid, tab)]["text"])
            rows.append({"item": f"{tab}: no NaN / undefined / Infinity on screen", "stock": sid, "passed": not bad_tokens, "dashboard": ", ".join(sorted(set(bad_tokens))) or None})
    cov = SCREEN[("*", "coverage")]
    O = {}
    for c in cov["cards"]:
        for tb in c["tables"]:
            for row in tb["rows"]:
                for h, cell in zip(tb["head"][1:], row[1:]):
                    O[("coverage", c["title"], row[0], h)] = cell
    for sid_, val in cov["rail"]:
        O[("rail", "rail", sid_, "1M")] = val
    for key, exp in expected_coverage().items():
        rows.append({"item": " › ".join(key[:2] + (key[3],)), "stock": key[2], "passed": O.get(key) == exp, "dashboard": O.get(key), "reference": exp})
    rows.append({"item": "no JavaScript errors while rendering", "passed": not SCREEN_ERRORS, "dashboard": "; ".join(e[1][:120] for e in SCREEN_ERRORS[:5]) or None})
    SCREEN_CHECKS = add_bool_checks(rows, "Screen audit", layer="3. Presentation")
    n_bad = int((SCREEN_CHECKS.status != "PASS").sum())
    print(f"Screen audit: {len(SCREEN_CHECKS):,} on-screen values and sentences read for {len(screen_stocks)} stocks in {time.time() - t0:.0f}s; {n_bad} differ from the expected text.")
    display(SCREEN_CHECKS[SCREEN_CHECKS.status != "PASS"].head(60)[["item", "stock", "shown_dashboard", "shown_reference"]])
elif not _want_screen:
    print("Screen audit skipped: install playwright (pip install playwright; playwright install chromium) or set RUN_SCREEN_AUDIT = True.")

## 11. Source verification against the original providers (optional, needs internet)
The data file is a processed copy of public sources. With `RUN_ONLINE_CHECKS = True` this section downloads the originals again and compares:
- **Yahoo Finance** (via `yfinance`): every stock's daily total-return series (as returns, since Yahoo rescales adjusted history whenever a new dividend is paid), the last close, and the consensus, market-cap and dividend fields.
- **ECB**: the euro AAA yield curve points (same source, exact to four decimals) and the ECB's official **euro reference rates**, an independent source for the FX the dashboard takes from Yahoo.
- **Kenneth French Data Library**: the European factor returns.

In [ ]:
def http_get(url, params=None, timeout=60):
    import requests
    r = requests.get(url, params=params, timeout=timeout, headers={"User-Agent": "Mozilla/5.0 (verification notebook)"})
    r.raise_for_status()
    return r


ONLINE = {}
if RUN_ONLINE_CHECKS:
    # ---- Yahoo Finance: total-return series, last close and snapshot fields
    try:
        import yfinance as yf
        status = META.get("stock_status", {})
        tickers = [STOCK_META.loc[s, "ticker"] for s in STOCK_IDS]
        raw = yf.download(tickers, start=ISO[0], auto_adjust=False, actions=False, group_by="ticker", progress=False, threads=True, repair=False)
        rows, brows = [], []
        for sid, tk in zip(STOCK_IDS, tickers):
            if tk not in raw.columns.get_level_values(0):
                brows.append({"item": "Yahoo returns data for the ticker", "stock": sid, "passed": False, "dashboard": tk})
                continue
            fr = raw[tk].dropna(subset=["Close"])
            fr.index = pd.DatetimeIndex(fr.index).tz_localize(None).normalize()
            scale = 0.01 if status.get(sid, {}).get("quote_currency") in ("GBp", "GBX") else 1.0
            ours = TRI_RAW[sid].dropna()
            theirs = (fr["Adj Close"] * scale).reindex(ours.index).dropna()
            common = ours.index.intersection(theirs.index)
            r_ours = np.log(ours[common]).diff().dropna()
            r_theirs = np.log(theirs[common]).diff().dropna()
            d = (r_ours - r_theirs).abs()
            bad_days = d[d > 5e-4]
            ONLINE.setdefault("yahoo_returns", []).append({"stock": sid, "ticker": tk, "common_days": len(d), "max_abs_diff": float(d.max()) if len(d) else np.nan,
                                                           "days_over_5bp": len(bad_days), "worst_day": bad_days.idxmax().date() if len(bad_days) else None})
            brows.append({"item": "daily total returns match Yahoo (within 5bp)", "stock": sid, "passed": len(bad_days) <= max(3, 0.002 * len(d)),
                          "dashboard": f"{len(bad_days)} of {len(d)} days differ", "note": "a handful of differences usually means Yahoo revised a dividend or split since the build"})
            last_close = fr["Close"].reindex([pd.Timestamp(AS_OF)]).iloc[0] * scale if pd.Timestamp(AS_OF) in fr.index else np.nan
            rows.append({"item": "close on the as-of date (Yahoo today)", "stock": sid, "dashboard": FUND[sid].get("price"), "reference": last_close, "fmt": num2})
        add_bool_checks(brows, "Online: Yahoo Finance", layer="1. Data")
        add_checks(rows, "Online: Yahoo Finance", tol_rel=1e-4, layer="1. Data")
        display(pd.DataFrame(ONLINE["yahoo_returns"]).set_index("stock").sort_values("days_over_5bp", ascending=False).head(15))
        info_rows = []
        for sid, tk in zip(STOCK_IDS, tickers):
            try:
                info = yf.Ticker(tk).get_info()
            except Exception as exc:
                info_rows.append({"stock": sid, "error": str(exc)[:80]})
                continue
            f = FUND[sid]
            scale = 0.01 if status.get(sid, {}).get("quote_currency") in ("GBp", "GBX") else 1.0
            cur, tgt = info.get("currentPrice"), info.get("targetMeanPrice")
            info_rows.append({"stock": sid, "yahoo_target_major": tgt * scale if tgt else np.nan, "dashboard_target": (f.get("consensus") or {}).get("target_mean"),
                              "yahoo_current_major": cur * scale if cur else np.nan, "dashboard_price": f.get("price"),
                              "yahoo_market_cap": info.get("marketCap"), "dashboard_market_cap": f.get("market_cap"), "yahoo_analysts": info.get("numberOfAnalystOpinions"),
                              "dashboard_analysts": (f.get("consensus") or {}).get("analysts")})
        INFO_CMP = pd.DataFrame(info_rows).set_index("stock")
        if "yahoo_target_major" in INFO_CMP:
            INFO_CMP["target gap"] = INFO_CMP.dashboard_target / INFO_CMP.yahoo_target_major - 1
            INFO_CMP["market-cap gap"] = INFO_CMP.dashboard_market_cap / INFO_CMP.yahoo_market_cap - 1
            add_checks([{"item": "consensus target vs Yahoo's current target", "stock": s, "dashboard": r.dashboard_target, "reference": r.yahoo_target_major, "fmt": num2,
                         "note": "consensus moves daily; gaps of a few % are revisions since the build"} for s, r in INFO_CMP.iterrows()],
                       "Online: Yahoo Finance", tol_rel=0.02, layer="1. Data")
            add_checks([{"item": "market cap vs Yahoo's market cap", "stock": s, "dashboard": r.dashboard_market_cap, "reference": r.yahoo_market_cap, "fmt": js_compact,
                         "note": "Yahoo counts some share classes differently; > 10% gaps deserve a look"} for s, r in INFO_CMP.iterrows()],
                       "Online: Yahoo Finance", tol_rel=0.10, layer="1. Data")
            display(INFO_CMP.sort_values("target gap", key=lambda s: s.abs(), ascending=False).head(15))
    except Exception as exc:
        finding("INFO", "Online checks", "Yahoo Finance could not be checked", f"{type(exc).__name__}: {exc}"[:300])

    # ---- ECB yields (same source as the dashboard) and ECB euro reference rates (independent FX source)
    ecb = "https://data-api.ecb.europa.eu/service/data/"
    for sid, key in (("EUR10Y", "YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y"), ("EUR5Y", "YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_5Y"), ("EUR2Y", "YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y")):
        try:
            df = pd.read_csv(io.StringIO(http_get(ecb + key, {"startPeriod": ISO[0], "format": "csvdata"}).text))
            s = pd.Series(df.OBS_VALUE.to_numpy(float), index=pd.to_datetime(df.TIME_PERIOD))
            ours = SERIES_RAW[sid].dropna()
            common = ours.index.intersection(s.index)
            d = (ours[common] - s[common].round(4)).abs()
            add_bool_checks([{"item": "yield matches the ECB to 4 decimals on every common date", "stock": sid, "passed": bool((d <= 5e-5).all()),
                              "dashboard": f"{int((d > 5e-5).sum())} of {len(d)} dates differ, max {d.max():.4f}pp"}], "Online: ECB", layer="1. Data")
        except Exception as exc:
            finding("INFO", "Online checks", f"ECB {sid} could not be checked", f"{type(exc).__name__}: {exc}"[:300])
    fx_rows = []
    for ccy, sid in (("USD", "FX_USD"), ("GBP", "FX_GBP"), ("SEK", "FX_SEK"), ("CHF", "FX_CHF"), ("NOK", "FX_NOK")):
        try:
            df = pd.read_csv(io.StringIO(http_get(ecb + f"EXR/D.{ccy}.EUR.SP00.A", {"startPeriod": ISO[0], "format": "csvdata"}).text))
            s = pd.Series(df.OBS_VALUE.to_numpy(float), index=pd.to_datetime(df.TIME_PERIOD))
            ours = SERIES_RAW[sid].dropna()
            common = ours.index.intersection(s.index)
            rel = (ours[common] / s[common] - 1).abs()
            fx_rows.append({"currency": ccy, "common_days": len(rel), "median_gap": float(rel.median()), "max_gap": float(rel.max()), "worst_day": rel.idxmax().date(),
                            "days_over_1.5pct": int((rel > 0.015).sum())})
        except Exception as exc:
            finding("INFO", "Online checks", f"ECB reference rate {ccy} could not be checked", f"{type(exc).__name__}: {exc}"[:300])
    if fx_rows:
        FX_CMP = pd.DataFrame(fx_rows).set_index("currency")
        display(Markdown("Yahoo FX closes vs the ECB 14:15 CET reference rates (a gap of a few tenths of a percent is timing; above 1.5% suggests a bad print):"))
        display(FX_CMP.style.format({"median_gap": "{:.3%}", "max_gap": "{:.2%}"}))
        add_bool_checks([{"item": "Yahoo FX within 1.5% of the ECB reference rate every day", "stock": c, "passed": r["days_over_1.5pct"] == 0,
                          "dashboard": f"{r['days_over_1.5pct']} days, worst {r.max_gap:.2%} on {r.worst_day}"} for c, r in FX_CMP.iterrows()], "Online: ECB", layer="1. Data")

    # ---- Kenneth French European factors
    try:
        import zipfile
        base = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
        frames = []
        for fname in ("Europe_5_Factors_Daily_CSV.zip", "Europe_Mom_Factor_Daily_CSV.zip"):
            with zipfile.ZipFile(io.BytesIO(http_get(base + fname, timeout=90).content)) as zf:
                text = zf.read(next(n for n in zf.namelist() if n.lower().endswith(".csv"))).decode("latin-1")
            header, recs = None, []
            for line in text.splitlines():
                cells = [c.strip() for c in line.split(",")]
                if header is None:
                    if len(cells) > 1 and cells[0] == "" and any(c in ("SMB", "WML", "Mom", "MOM") for c in cells[1:]):
                        header = [{"Mom": "WML", "MOM": "WML"}.get(c, c) for c in cells[1:]]
                    continue
                if not re.fullmatch(r"\d{8}", cells[0] if cells else ""):
                    if recs:
                        break
                    continue
                recs.append([cells[0]] + [float(c) for c in cells[1:len(header) + 1]])
            fr = pd.DataFrame(recs, columns=["date"] + header).set_index("date")
            fr.index = pd.to_datetime(fr.index, format="%Y%m%d")
            frames.append(fr.replace([-99.99, -999.0], np.nan) / 100)
        ff = pd.concat(frames, axis=1)
        rows = []
        for fid, col in (("FF_SMB", "SMB"), ("FF_HML", "HML"), ("FF_RMW", "RMW"), ("FF_CMA", "CMA"), ("FF_WML", "WML")):
            lvl = SERIES_RAW[fid].dropna()
            ours = (lvl / lvl.shift(1) - 1).dropna()
            common = ours.index.intersection(ff.index)
            d = (ours[common] - ff[col][common]).abs()
            rows.append({"item": "daily factor return matches the French library (1bp)", "stock": fid, "passed": bool((d <= 1e-4).mean() > 0.995),
                         "dashboard": f"{int((d > 1e-4).sum())} of {len(d)} days differ, max {d.max() * 1e4:.1f}bp",
                         "note": "only days where both calendars have a value; the dashboard's calendar drops days when few stocks trade"})
        add_bool_checks(rows, "Online: Kenneth French", layer="1. Data")
    except Exception as exc:
        finding("INFO", "Online checks", "Kenneth French factors could not be checked", f"{type(exc).__name__}: {exc}"[:300])
    display(summarise(checks_frame()[checks_frame().section.str.startswith("Online")]))
else:
    print("Online source checks skipped (set RUN_ONLINE_CHECKS = True; needs internet and yfinance).")

## 12. Scorecard

In [ ]:
CK = checks_frame()
FIND = pd.DataFrame(FINDINGS, columns=["severity", "area", "title", "detail", "stocks", "evidence"])
FIND = FIND.assign(order=FIND.severity.map(SEVERITY_ORDER)).sort_values(["order", "area"]).drop(columns="order").reset_index(drop=True)
BACKTESTS = pd.DataFrame(BT_ROWS) if "BT_ROWS" in globals() else pd.DataFrame(columns=["test", "measured", "target", "status", "detail"])
LAYER_SUMMARY = summarise(CK, by=("layer",))
SECTION_SUMMARY = summarise(CK, by=("layer", "section"))

eng = CK[CK.layer == "2. Engine"]
exact = eng[~eng.section.isin(["GARCH optimiser"])]  # the optimiser check compares two different estimators, not two implementations
eng_num = exact.dropna(subset=["dashboard", "reference"])
n_eng, n_eng_fail, n_eng_warn = len(exact), int((exact.status == "FAIL").sum()), int((exact.status == "WARN").sum())
impact = CK[CK.section == "Impact of data errors"]
n_impact = int((impact.status == "FAIL").sum())
pres = CK[CK.layer == "3. Presentation"]
pres_fail = int((pres.status == "FAIL").sum())
scr = CK[CK.section == "Screen audit"]
bt_pass = int((BACKTESTS.status == "PASS").sum()) if len(BACKTESTS) else 0
sev = FIND.severity.value_counts()
KPIS = [
    ("Engine numbers reconciled", f"{(n_eng - n_eng_fail) / max(n_eng, 1):.2%}" if ENGINE else DASH,
     f"{n_eng:,} comparisons, {n_eng_fail:,} visible differences" if ENGINE else "engine not run"),
    ("Largest engine difference", f"{eng_num.rel_diff.max():.0e}" if len(eng_num) else DASH, "relative, over every reconciled number"),
    ("Screen values checked", f"{len(scr):,}" if len(scr) else DASH,
     f"{int((scr.status != 'PASS').sum()):,} differ from the expected text" if len(scr) else "screen audit not run"),
    ("Numbers on screen hit by data errors", f"{n_impact:,}", f"{impact[impact.status == 'FAIL'].stock.nunique()} stocks; see section 6b" if len(impact) else "no data errors found"),
    ("Presentation checks failing", f"{pres_fail:,}", f"of {len(pres):,}: labels, cross-tab consistency, screen"),
    ("Backtests passed", f"{bt_pass} of {len(BACKTESTS)}", "forward-looking claims tested out of sample"),
    ("Critical or high findings", f"{int(sev.get('CRITICAL', 0) + sev.get('HIGH', 0))}",
     f"{int(sev.get('CRITICAL', 0))} critical; {int(sev.get('MEDIUM', 0))} medium, {int(sev.get('LOW', 0))} low, {int(sev.get('INFO', 0))} info"),
]
TILE = ("display:inline-block;vertical-align:top;width:190px;margin:0 10px 10px 0;padding:12px 14px;background:#fcfcfb;"
        "border:1px solid rgba(11,11,11,.10);border-radius:10px;font-family:system-ui,-apple-system,'Segoe UI',sans-serif")
KPI_HTML = "".join(
    f'<div style="{TILE}"><div style="font-size:12px;color:#52514e">{a}</div>'
    f'<div style="font-size:26px;font-weight:600;color:#0b0b0b;margin:4px 0 2px">{b}</div>'
    f'<div style="font-size:11.5px;color:#898781;line-height:1.35">{c}</div></div>' for a, b, c in KPIS)
display(HTML(f'<div style="font:600 15px system-ui,sans-serif;margin:6px 0 10px">Property Factor Lens verification, data as of {AS_OF}</div>{KPI_HTML}'))
display(style_status(LAYER_SUMMARY.reset_index()))
verdict = [
    f"**Engine fidelity.** {n_eng:,} numbers compared across every model, frequency, window and currency; "
    + (f"all agree with the independent implementation within tolerance (largest relative difference {eng_num.rel_diff.max():.1e})." if ENGINE and n_eng_fail == 0 and n_eng_warn == 0
       else f"{n_eng_fail:,} differ visibly on screen and {n_eng_warn:,} only below display precision (see the discrepancy table)." if ENGINE else "not run."),
    f"**Findings.** {int(sev.get('CRITICAL', 0))} critical, {int(sev.get('HIGH', 0))} high, {int(sev.get('MEDIUM', 0))} medium, {int(sev.get('LOW', 0))} low, {int(sev.get('INFO', 0))} informational (table below).",
    f"**Backtests.** {bt_pass} of {len(BACKTESTS)} forward-looking claims hold out of sample; the rest are listed with the measured value next to the promise.",
]
display(Markdown("\n\n".join(verdict)))

In [ ]:
display(Markdown("### Findings, most severe first"))
display(style_status(FIND[["severity", "area", "title", "stocks", "detail"]], col="severity"))
display(Markdown("### Backtest verdicts"))
display(style_status(BACKTESTS))
display(Markdown("### Every section"))
display(style_status(SECTION_SUMMARY.reset_index()))

## 13. Export every data point
One Excel workbook (a sheet per table), the full comparison log as compressed CSV, and a self-contained HTML report with the scorecard, findings and every chart, for circulation.

In [ ]:
def _clean_for_excel(df):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].map(lambda v: v if v is None or isinstance(v, (str, int, float, bool)) else str(v))
    return df


if EXPORT:
    stamp = AS_OF
    CK.to_csv(OUT / f"all_checks_{stamp}.csv.gz", index=False, compression="gzip")
    FIND.to_csv(OUT / f"findings_{stamp}.csv", index=False)
    tables = {
        "Scorecard": LAYER_SUMMARY.reset_index(), "Sections": SECTION_SUMMARY.reset_index(), "Findings": FIND, "Backtests": BACKTESTS,
        "Discrepancies": CK[CK.status != "PASS"], "Data - series": SERIES_AUDIT.reset_index(), "Data - stocks": STOCK_AUDIT.reset_index(),
        "Data - big moves": STOCK_MOVES, "Data - series jumps": SERIES_JUMPS, "Dividends": DIVIDENDS.reset_index(), "Fundamentals": FUND_TABLE.reset_index(),
        "Fundamental flags": FUND_FLAGS, "Rate sensitivity labels": RATE_LABELS.reset_index(), "Returns vs attribution": RET_VS_ATTR,
        "Month-end date probe": MONTH_BUG,
    }
    for name, var in (("GARCH optimiser", "GARCH_OPT"), ("GARCH backtest", "GARCH_SUMMARY"), ("VaR backtest", "VAR_BT"), ("Exposure fcst OOS", "FC_BT"),
                      ("Exposure fcst coverage", "FC_COV"), ("Signals robustness", "SIG_BT"), ("Episodes ex-ante", "EP_SUMMARY"), ("Exposure influence", "INFLUENCE"),
                      ("Beta by frequency", "FREQ_BETA"), ("Multiple testing", "TT")):
        if var in globals():
            tables[name] = globals()[var].reset_index() if not isinstance(globals()[var].index, pd.RangeIndex) else globals()[var]
    xlsx = OUT / f"dashboard_verification_{stamp}.xlsx"
    try:
        with pd.ExcelWriter(xlsx, engine="openpyxl") as xw:
            for name, df in tables.items():
                _clean_for_excel(df).head(1_000_000).to_excel(xw, sheet_name=name[:31], index=False)
        print(f"Excel workbook: {xlsx} (every individual comparison is in the CSV below)")
    except ImportError:
        for name, df in tables.items():
            df.to_csv(OUT / f"{re.sub(r'[^A-Za-z0-9]+', '_', name).strip('_').lower()}_{stamp}.csv", index=False)
        print("openpyxl is not installed: wrote one CSV per table instead of the Excel workbook.")

    def html_table(df, max_rows=400):
        return df.head(max_rows).to_html(index=False, border=0, classes="t", na_rep=DASH, escape=True)

    kpi_html = "".join(f'<div class="kpi"><div class="l">{a}</div><div class="v">{b}</div><div class="s">{c}</div></div>' for a, b, c in KPIS)  # noqa: E501
    verdict_html = "".join("<p>" + re.sub(r"[*][*](.+?)[*][*]", r"<b>\1</b>", v) + "</p>" for v in verdict)
    figs_html = "".join(f'<figure><figcaption>{t}</figcaption><img alt="{t}" src="data:image/png;base64,{base64.b64encode(p).decode()}"></figure>' for t, p in FIGURES)
    html = f"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Dashboard verification {stamp}</title><style>
:root {{ --surface:{C.surface}; --ink:{C.ink}; --ink2:{C.ink2}; --muted:{C.muted}; --grid:{C.grid}; }}
body {{ margin:0; background:#f9f9f7; color:var(--ink); font:14px/1.5 system-ui,-apple-system,"Segoe UI",sans-serif; }}
main {{ max-width:1180px; margin:0 auto; padding:24px 16px 64px; }}
h1 {{ font-size:22px; margin:0 0 4px; }} h2 {{ font-size:17px; margin:32px 0 8px; }} p.sub {{ color:var(--ink2); margin:0 0 18px; }}
.kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(170px,1fr)); gap:10px; }}
.kpi {{ background:var(--surface); border:1px solid rgba(11,11,11,.1); border-radius:10px; padding:12px 14px; }}
.kpi .l {{ font-size:12px; color:var(--ink2); }} .kpi .v {{ font-size:24px; font-weight:600; margin:2px 0; }} .kpi .s {{ font-size:11.5px; color:var(--muted); }}
.wrap {{ overflow-x:auto; background:var(--surface); border:1px solid rgba(11,11,11,.1); border-radius:10px; }}
table.t {{ border-collapse:collapse; width:100%; font-size:12.5px; font-variant-numeric:tabular-nums; }}
table.t th {{ text-align:left; background:#f0efec; padding:6px 8px; position:sticky; top:0; }} table.t td {{ padding:5px 8px; border-top:1px solid var(--grid); vertical-align:top; }}
figure {{ margin:18px 0; background:var(--surface); border:1px solid rgba(11,11,11,.1); border-radius:10px; padding:12px; }}
figcaption {{ font-weight:600; margin-bottom:6px; }} img {{ max-width:100%; height:auto; }}
</style></head><body><main>
<h1>Property Factor Lens: independent verification and backtest</h1>
<p class="sub">Data as of {AS_OF} (built {META.get('generated_at')}); report generated {time.strftime('%Y-%m-%d %H:%M')}; engine: {ENGINE_NAME if ENGINE else 'not run'}.</p>
<div class="kpis">{kpi_html}</div>
<h2>Verdict</h2>{verdict_html}
<h2>Findings</h2><div class="wrap">{html_table(FIND[["severity", "area", "title", "stocks", "detail"]])}</div>
<h2>Backtests</h2><div class="wrap">{html_table(BACKTESTS)}</div>
<h2>Checks by section</h2><div class="wrap">{html_table(SECTION_SUMMARY.reset_index())}</div>
<h2>Discrepancies (first 400; the workbook has all)</h2><div class="wrap">{html_table(CK[CK.status != 'PASS'][['layer', 'section', 'item', 'stock', 'setting', 'shown_dashboard', 'shown_reference', 'status', 'note']])}</div>
<h2>Charts</h2>{figs_html}
</main></body></html>"""
    report = OUT / f"verification_report_{stamp}.html"
    report.write_text(html, encoding="utf-8")
    print(f"HTML report:    {report}\nAll checks:     {OUT / f'all_checks_{stamp}.csv.gz'} ({len(CK):,} rows)")
print(f"Total run time {time.time() - T_START:.0f}s")

## 14. Drill down into one stock
`drilldown("SGRO")` lists every check for a stock (anything not passing first), the findings that name it, and plots its data with the flagged points.

In [ ]:
def drilldown(stock_id, show_all=False):
    c = checks_frame()
    d = c[c.stock == stock_id]
    display(Markdown(f"### {NAME.get(stock_id, stock_id)} ({stock_id}): {len(d):,} checks, {int((d.status == 'FAIL').sum())} FAIL, {int((d.status == 'WARN').sum())} WARN"))
    display(summarise(d, by=("layer", "section")))
    cols = ["layer", "section", "item", "setting", "dashboard", "reference", "rel_diff", "shown_dashboard", "shown_reference", "status", "note"]
    display(d[cols] if show_all else d[d.status != "PASS"][cols])
    f_ = FIND[FIND.stocks.fillna("").str.contains(rf"\b{re.escape(stock_id)}\b")]
    if len(f_):
        display(Markdown("**Findings that name this stock:**"))
        display(f_[["severity", "title", "detail"]])
    j = STOCK_IDS.index(stock_id)
    st = REF.stocks[j]
    fig, axes = plt.subplots(1, 2, figsize=(14, 3.8))
    ax = axes[0]
    L = pd.Series(st.tri, index=REF.dates).dropna()
    peer = pd.Series(REF.peer_index(st, "sub"), index=REF.dates).loc[L.index[0]:]
    ax.plot(L.index, L / L.iloc[0] * 100, color=C.s1, lw=2, label=f"{stock_id} total return ({st.currency})")
    ax.plot(peer.index, peer / peer.iloc[0] * 100, color=C.deemph, lw=1.6, label="sub-sector peers (EUR)")
    mv = STOCK_MOVES[STOCK_MOVES.stock == stock_id] if len(STOCK_MOVES) else pd.DataFrame()
    for _, r in mv.iterrows():
        dt = pd.Timestamp(r["date"])
        ax.scatter([dt], [L.loc[dt] / L.iloc[0] * 100], s=40, color=C.s2, zorder=4)
        ax.annotate(f"{r.log_return:+.0%}", (dt, L.loc[dt] / L.iloc[0] * 100), xytext=(4, 4), textcoords="offset points", fontsize=8, color=C.ink2)
    ax.set_title("Total return, rebased to 100 (orange = daily move above 20%)")
    ax.legend(loc="upper left", fontsize=8)
    ax = axes[1]
    if ENGINE:
        js = E["rolling"]["rates"][j]
        if js["ok"] and "RATES" in js["beta"]:
            py = REF.rolling_fit(st, REF.resolve(MODEL_SPECS["mktrates"]), DV["freq"], DV["ccy"], step=5 if DV["freq"] == "D" else 1)
            ax.plot(pd.to_datetime(js["dates"]), np.array(js["beta"]["RATES"]) * 1000, color=C.s1, lw=2, label="dashboard engine")
            ax.plot(pd.to_datetime(py["dates"]), np.array(py["beta"]["RATES"]) * 1000, color=C.s2, lw=1, ls="none", marker=".", ms=3, label="independent (Python)")
            ax.axhline(0, color=C.axis, lw=1)
            ax.set_title("Rolling rate sensitivity, % per +10bp (the two should overlap exactly)")
            ax.legend(loc="lower left", fontsize=8)
    fig.tight_layout()
    plt.show()
    plt.close(fig)


worst = CK[CK.status == "FAIL"].stock.value_counts()
example = worst.index[0] if len(worst) else STOCK_IDS[0]
drilldown(example if example in STOCK_IDS else STOCK_IDS[0])